# KAMP 전체 분석 통합 실행 노트북

이 노트북은 원본 데이터 하나에서 시작해 다음 계산을 순서대로 모두 실행합니다.

1. 자료 이해와 EDA
2. 24시간 예측용 전처리
3. LightGBM 수치 예측과 피크 경보
4. GRU 실험과 LightGBM 동일 조건 비교
5. LightGBM GRU 앙상블 선택
6. 영향요인과 오류 분석
7. 7월과 8월 자원 최적화 모의실험

별도의 파이썬 파일을 불러오지 않습니다. dataset/okm_augumented_2021.csv와 설치 패키지만 준비한 뒤 **Run All**을 실행합니다. 모든 결과는 results/와 submission/에 저장됩니다.

In [ ]:
from pathlib import Path
import os, sys, time, json, random, platform, warnings

_start = Path.cwd().resolve()
_candidates = [_start / 'final_code', _start, _start.parent / 'final_code', _start.parent]
PROJECT_ROOT = next((p for p in _candidates if (p / 'dataset/okm_augumented_2021.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('dataset/okm_augumented_2021.csv가 있는 final_code 폴더에서 실행하세요.')
PROJECT_ROOT = PROJECT_ROOT.resolve()
ROOT = PROJECT_ROOT
(ROOT / 'results').mkdir(parents=True, exist_ok=True)

RUN_ALL_STARTED = time.time()
print('프로젝트 폴더:', ROOT)
print('원본 데이터:', ROOT / 'dataset/okm_augumented_2021.csv')
print('실행 시작:', time.strftime('%Y-%m-%d %H:%M:%S'))

In [ ]:
# 자체 검증 기준값도 노트북 안에 포함해 다른 컴퓨터에서 별도 reference 파일이 필요하지 않습니다.
_REFERENCE = ROOT / 'reference'
_REFERENCE.mkdir(parents=True, exist_ok=True)
(_REFERENCE / 'expected_metrics.csv').write_text("﻿구간,모델,precision,recall,F2,MAE,FP\r\n7월(검증),baseline_w1@176,0.7002764090854464,0.6175940646528881,0.6325307744078504,10.290038108825684,2494\r\n8~9월(개발 검증),baseline_w1@176,0.3026026702720973,0.363000506842372,0.3490661675829532,33.157501220703125,8253\r\n7월(검증),q1_current@176,0.7433215869046266,0.6930577636459989,0.7025592539269828,13.868385314941406,2258\r\n8~9월(개발 검증),q1_current@176,0.3864727854855923,0.8809934110491637,0.7014754310066508,27.040611267089844,13797\r\n7월(검증),gru_tuned@176,0.6670104059248149,0.7541070482246953,0.7349143718883633,11.870945930480955,3552\r\n8~9월(개발 검증),gru_tuned@176,0.4681365958591019,0.5294475418144957,0.5159333820653139,17.127046585083008,5934\r\n7월(검증),ens_current@176,0.7099138357928098,0.7597244303126656,0.7492108618851517,11.167757987976074,2929\r\n8~9월(개발 검증),ens_current@176,0.4540939303191877,0.6302078053725292,0.5848431826306185,18.996700286865234,7474\r\n7월(검증),q1_new@176,0.8165657097708717,0.6572337042925278,0.6839237658269731,9.729681015014648,1393\r\n8~9월(개발 검증),q1_new@176,0.5146555656934306,0.9148504815002534,0.7917222261211313,20.941814422607425,8511\r\n7월(검증),q1_new@R85-최근4주,0.6875544640735166,0.9198728139904612,0.8616444612116038,9.729681015014648,3944\r\n8~9월(개발 검증),q1_new@R85-최근4주,0.5144831557731068,0.9056259503294476,0.7860976682798063,20.941814422607425,8431\r\n7월(검증),ens_new_w05@176,0.764934057408844,0.7315315315315315,0.7379766054359216,9.570734977722168,2121\r\n8~9월(개발 검증),ens_new_w05@176,0.4859509249941946,0.6363912823112012,0.5992859733862809,16.268465042114258,6641\r\n", encoding='utf-8')
(_REFERENCE / 'expected_extra.csv').write_text("﻿항목,구간,값,허용오차\r\ncurve_final MAE,7월(검증),7.9493,0.05\r\nrule3 MAE,7월(검증),10.19245719909668,0.05\r\ngru MAE,7월(검증),11.716739654541016,0.3\r\ncurve_final MAE,8~9월(개발 검증),7.8266,0.05\r\nrule3 MAE,8~9월(개발 검증),13.430357933044434,0.05\r\ngru MAE,8~9월(개발 검증),17.127046585083008,0.1\r\n경보 정밀도,7월(검증),0.6875544640735166,0.003\r\n경보 재현율,7월(검증),0.9198728139904612,0.003\r\n경보와주의 재현율,7월(검증),0.9542130365659778,0.003\r\n놓친 피크(경보와주의),7월(검증),432.0,15.0\r\nBrier,7월(검증),0.0509512347857434,0.002\r\nECE,7월(검증),0.0190517685621054,0.004\r\n경보 정밀도,8~9월(개발 검증),0.5144831557731068,0.003\r\n경보 재현율,8~9월(개발 검증),0.9056259503294476,0.003\r\n경보와주의 재현율,8~9월(개발 검증),0.970704510897111,0.003\r\n놓친 피크(경보와주의),8~9월(개발 검증),289.0,15.0\r\nBrier,8~9월(개발 검증),0.0608409356142076,0.002\r\nECE,8~9월(개발 검증),0.0435716328021572,0.004\r\n", encoding='utf-8')
print('자체 검증 기준값 준비 완료:', _REFERENCE)

# 1. 데이터 이해와 EDA

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

ROOT = PROJECT_ROOT
if not (ROOT / 'dataset' / 'okm_augumented_2021.csv').exists():
    ROOT = PROJECT_ROOT
DATA_PATH = ROOT / 'dataset' / 'okm_augumented_2021.csv'
CSV_DIR = ROOT / 'results' / 'eda' / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)
POWER_COLS = ['15분', '30분', '45분', '60분']
VARIABLES = POWER_COLS + ['평균', '생산량', '기온', '풍속', '습도', '강수량', '전기요금(계절)', '공장인원', '인건비']
sns.set_theme(style='whitegrid', font='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)

def save_table(table, name, index=True):
    table.to_csv(CSV_DIR / f'{name}.csv', index=index, encoding='utf-8-sig')

def date_axis(ax):
    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.set_xlabel('날짜')

raw = pd.read_csv(DATA_PATH)
display(raw.head())
print(f'원본: {raw.shape[0]:,}행 × {raw.shape[1]}열')
print('CSV 저장 위치:', CSV_DIR)

In [ ]:
quality = pd.DataFrame({'dtype': raw.dtypes.astype(str), '결측수': raw.isna().sum(), '고유값수': raw.nunique()})
display(quality)
print('전체 행 중복:', raw.duplicated().sum())
save_table(quality, 'column_quality')
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
raw.isna().sum().plot.bar(ax=axes[0], color='steelblue')
axes[0].set(title='변수별 결측 개수', ylabel='행 수')
sns.heatmap(raw[VARIABLES].isna().T, cbar=False, ax=axes[1])
axes[1].set(title='결측 위치 (원본 행 순서)', xlabel='행 순서', ylabel='변수')
plt.tight_layout()
plt.show()

df = raw.copy()
df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
df['원본시간'] = df['시간']
order_hour = df.groupby('날짜', sort=False).cumcount()
valid_hour = df['시간'].between(0, 23) & df['시간'].eq(df['시간'].round())
assert df.groupby('날짜').size().eq(24).all(), '날짜별 행 수를 먼저 검토하세요.'
assert df.loc[valid_hour, '시간'].eq(order_hour[valid_hour]).all(), '정상 시간의 정렬을 먼저 검토하세요.'
df['시간수정'] = ~valid_hour
df.loc[~valid_hour, '시간'] = order_hour[~valid_hour]
df['시간'] = df['시간'].astype(int)
df['날짜시간'] = df['날짜'] + pd.to_timedelta(df['시간'], unit='h')
df = df.sort_values('날짜시간').reset_index(drop=True)
assert not df['날짜시간'].duplicated().any()
assert df['날짜시간'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
df['월'] = df['날짜'].dt.month
df['요일번호'] = df['날짜'].dt.dayofweek
df['주말'] = df['요일번호'].ge(5)
df['시간최대전력'] = df[POWER_COLS].max(axis=1)
df['전력평균계산'] = df[POWER_COLS].mean(axis=1)
df['기간'] = np.select([df['월'].isin([1,2]), df['월'].isin([3,4,5]), df['월'].isin([6,7,8])], ['1~2월','3~5월','6~8월'], default='9월')
display(df.loc[df['시간수정'], ['날짜','원본시간','시간','인건비']])
print('수정 시간:', df['시간수정'].sum(), '행 / 기간:', df['날짜시간'].min(), '~', df['날짜시간'].max())
save_table(df.loc[df['시간수정']], 'time_corrections', index=False)

In [ ]:
denominator = df[POWER_COLS].sum(axis=1).replace(0, np.nan)
staff_calc = df['생산량'] / denominator
mask = df['공장인원'].notna() & staff_calc.notna()
checks = pd.DataFrame({
    '검사': ['평균과 4개 전력 평균의 최대 차이', '공장인원과 생산량/전력합의 최대 차이', '공장인원 비교 가능 행 수'],
    '값': [(df['평균']-df['전력평균계산']).abs().max(), (df.loc[mask,'공장인원']-staff_calc[mask]).abs().max(), mask.sum()]})
display(checks)
df['인건비_시간규칙'] = np.where(df['시간'].between(9,17), 1.0, 1.5)
print('인건비 원본과 시간 규칙 불일치:', df['인건비'].ne(df['인건비_시간규칙']).sum())
display(df.groupby('월')['전기요금(계절)'].agg(['min','max','nunique']))
calendar_check = pd.DataFrame({'day_요일1시작': df['day'].eq(df['요일번호']+1), 'd_일': df['d'].eq(df['날짜'].dt.day), 'm_월': df['m'].eq(df['월'])}).mean().rename('일치비율')
display(calendar_check)
profiles = df.pivot(index='날짜', columns='시간', values=POWER_COLS)
repeat_counts = pd.Series({'전력4개 조합 반복 행': df.duplicated(POWER_COLS, keep=False).sum(), '일별 전력패턴 반복 날짜': profiles.duplicated(keep=False).sum()})
display(repeat_counts)
fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].scatter(staff_calc[mask], df.loc[mask,'공장인원'], s=8, alpha=.4)
axes[0].set(xlabel='생산량 / 네 전력값 합', ylabel='공장인원', title='공장인원 파생관계')
(df['평균']-df['전력평균계산']).value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(title='평균 컬럼의 반올림 차이', xlabel='원본 평균 − 계산 평균', ylabel='행 수')
plt.tight_layout(); plt.show()
save_table(checks, 'derived_checks', index=False)
save_table(repeat_counts.to_frame('개수'), 'repeated_patterns')

In [ ]:
long_df = df.melt(id_vars=[c for c in df.columns if c not in POWER_COLS], value_vars=POWER_COLS, var_name='분구간', value_name='전력값')
long_df['분'] = long_df['분구간'].str.replace('분','',regex=False).astype(int)
long_df['관측시각'] = long_df['날짜시간'] + pd.to_timedelta(long_df['분'], unit='m')
long_df = long_df.sort_values('관측시각').reset_index(drop=True)
assert len(long_df) == len(df)*4
assert not long_df['관측시각'].duplicated().any()
assert long_df['관측시각'].diff().dropna().eq(pd.Timedelta(minutes=15)).all()
long_df['전력변화량'] = long_df['전력값'].diff()
print('15분 관측:', len(long_df), '개 / 연속성 검사 통과')
display(long_df[['날짜시간','관측시각','분구간','전력값','생산량']].head(8))
# 이후 시간·요일 집계는 원본 시간구간 기준입니다. 60분 값도 원래 구간에 포함됩니다.

In [ ]:
for col in VARIABLES:
    fig, ax = plt.subplots(figsize=(15,3))
    ax.plot(df['날짜시간'], df[col], linewidth=.65)
    ax.set(title=f'전체 기간 {col}', ylabel=col)
    date_axis(ax)
    plt.tight_layout(); plt.show()

In [ ]:
for col in ['평균','생산량','기온','공장인원','인건비']:
    fig, axes = plt.subplots(2,2, figsize=(16,6), sharey=True)
    for ax, period in zip(axes.flat, ['1~2월','3~5월','6~8월','9월']):
        part = df[df['기간'].eq(period)]
        ax.plot(part['날짜시간'], part[col], linewidth=.65)
        ax.set(title=f'{period} {col}', ylabel=col)
        date_axis(ax)
    plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.boxplot(data=long_df, x='월', y='전력값', showfliers=False, ax=axes[0])
axes[0].set_title('월별 15분 전력 분포 (이상점 표시는 생략)')
df.groupby('월')['생산량'].mean().plot.bar(ax=axes[1], color='steelblue')
axes[1].set(title='월별 시간당 평균 생산량', ylabel='생산량')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(15,4))
for weekend, part in df.groupby('주말'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0], marker='o', label='주말' if weekend else '평일')
axes[0].set(title='평일/주말 시간대별 평균전력', ylabel='전력값', xticks=range(0,24,2)); axes[0].legend()
heat = df.pivot_table(index='요일번호', columns='시간', values='전력평균계산', aggfunc='mean')
heat.index = ['월','화','수','목','금','토','일']
sns.heatmap(heat, ax=axes[1], cmap='YlOrRd')
axes[1].set_title('요일 × 시간 평균전력')
plt.tight_layout(); plt.show()
for col in VARIABLES:
    fig, ax = plt.subplots(figsize=(15,2.8))
    ax.plot(df['날짜시간'], df[col].where(df['주말']), linewidth=.8)
    ax.set(title=f'주말 {col}', ylabel=col)
    date_axis(ax)
    plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(13,8))
for ax, col in zip(axes.flat, POWER_COLS):
    ax.scatter(df['생산량'], df[col], s=8, alpha=.25)
    ax.set(xlabel='시간별 생산량', ylabel='전력값', title=f'생산량과 {col}')
plt.tight_layout(); plt.show()
df['생산여부'] = np.where(df['생산량'].gt(0), '생산량 > 0', '생산량 = 0')
df['생산량구간'] = pd.qcut(df['생산량'].where(df['생산량'].gt(0)), q=5, duplicates='drop')
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.boxplot(data=df, x='생산여부', y='전력평균계산', ax=axes[0])
sns.boxplot(data=df, x='생산량구간', y='전력평균계산', showfliers=False, ax=axes[1])
axes[0].set_title('생산 여부별 시간 평균전력')
axes[1].set_title('양의 생산량 분위 구간별 전력'); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(12,4))
for label, part in df.groupby('생산여부'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=ax, marker='o', label=label)
ax.set(title='생산 여부별 시간대 전력', ylabel='전력값'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(13,8))
for ax, col in zip(axes.flat, ['기온','습도','풍속','강수량']):
    sns.scatterplot(data=df, x=col, y='전력평균계산', hue='생산여부', s=12, alpha=.3, ax=ax)
    ax.set_title(f'{col}과 전력')
plt.tight_layout(); plt.show()
cols = ['전력평균계산','생산량','기온','습도','풍속','강수량']
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[cols].corr(method='spearman'), annot=True, fmt='.2f', vmin=-1, vmax=1, cmap='coolwarm', ax=ax)
ax.set_title('Spearman 상관계수'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(data=df, x='생산량구간', y='전력평균계산', hue='주말', showfliers=False, ax=ax)
ax.set_title('비슷한 생산량 구간에서 평일/주말 전력 비교')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
thresholds = long_df['전력값'].quantile([.90,.95,.99])
peak_summary = pd.DataFrame({'기준값': thresholds, '기준이상비율': [long_df['전력값'].ge(v).mean() for v in thresholds]})
display(peak_summary)
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.histplot(long_df['전력값'], bins=60, ax=axes[0])
for q, value in thresholds.items():
    axes[0].axvline(value, linestyle='--', label=f'{q:.0%}: {value:g}')
    long_df.assign(고전력=long_df['전력값'].ge(value)).groupby('시간')['고전력'].mean().mul(100).plot(ax=axes[1], marker='o', label=f'{q:.0%} 기준')
axes[0].legend(); axes[0].set_title('전력 분포와 후보 기준')
axes[1].set(title='시간대별 고전력 발생률', ylabel='발생률 (%)'); axes[1].legend()
plt.tight_layout(); plt.show()
long_df['피크후보95'] = long_df['전력값'].ge(thresholds.loc[.95])
fig, axes = plt.subplots(2,1, figsize=(15,7))
axes[0].plot(long_df['관측시각'], long_df['전력값'], linewidth=.5)
p = long_df[long_df['피크후보95']]
axes[0].scatter(p['관측시각'], p['전력값'], color='tomato', s=7, label='95% 기준 이상')
axes[0].set(title='고전력 후보 시점', ylabel='전력값'); axes[0].legend()
axes[1].plot(long_df['관측시각'], long_df['전력변화량'], linewidth=.5)
axes[1].set(title='직전 15분 대비 전력 변화', ylabel='전력값 차이')
for ax in axes: date_axis(ax)
plt.tight_layout(); plt.show()
display(long_df.nlargest(15, '전력변화량')[['관측시각','전력값','전력변화량','생산량','시간수정']])
peak_by_hour = long_df.groupby('시간').agg(관측수=('전력값','size'), 피크수=('피크후보95','sum'), 발생률=('피크후보95','mean'))
display(peak_by_hour)
save_table(peak_summary, 'peak_threshold_candidates')
save_table(peak_by_hour, 'peak_by_hour')

In [ ]:
monthly_summary = df.groupby('월').agg(관측수=('평균','size'), 평균전력=('전력평균계산','mean'), 최대전력=('시간최대전력','max'), 평균생산량=('생산량','mean'))
save_table(monthly_summary, 'monthly_summary')
save_table(df, 'hourly_eda', index=False)
save_table(long_df, 'quarter_hour_eda', index=False)
save_table(df[VARIABLES].describe().T, 'descriptive_statistics')
display(monthly_summary)
print('저장된 CSV:')
for path in sorted(CSV_DIR.glob('*.csv')):
    print(path.name)

In [ ]:
zero_mask = df['생산량'].eq(0)
zero_threshold = df.loc[zero_mask, '전력평균계산'].quantile(.9)
zero_high = zero_mask & df['전력평균계산'].ge(zero_threshold)
zero_analysis = df[['날짜시간','시간','월','요일번호','생산량','전력평균계산','시간수정']].copy()
zero_analysis['고전력_비생산'] = zero_high
zero_analysis['직전시간_생산중'] = df['생산량'].shift(1).gt(0)
zero_analysis['다음시간_생산중'] = df['생산량'].shift(-1).gt(0)
zero_run_id = zero_mask.ne(zero_mask.shift()).cumsum()
zero_runs = df.loc[zero_mask].groupby(zero_run_id[zero_mask]).agg(
    시작=('날짜시간','min'), 마지막관측=('날짜시간','max'),
    지속시간=('날짜시간','size'), 평균전력=('전력평균계산','mean'), 최대전력=('시간최대전력','max'))
zero_runs['종료'] = zero_runs['마지막관측'] + pd.Timedelta(hours=1)
print(f'비생산 시간 평균전력 90% 기준: {zero_threshold:.2f}; 해당 시간: {zero_high.sum()}개')
fig, axes = plt.subplots(2,2, figsize=(15,8))
sns.histplot(df.loc[zero_mask,'전력평균계산'], bins=40, ax=axes[0,0])
axes[0,0].axvline(zero_threshold, color='tomato', linestyle='--')
axes[0,0].set_title('생산량 0: 평균전력 분포와 상위 10% 경계')
axes[0,1].plot(df['날짜시간'], df['전력평균계산'].where(zero_mask), linewidth=.6)
axes[0,1].scatter(df.loc[zero_high,'날짜시간'], df.loc[zero_high,'전력평균계산'], color='tomato', s=12)
axes[0,1].set_title('비생산 전력과 높은 전력 시점'); date_axis(axes[0,1])
zero_hour = zero_analysis.loc[zero_mask].groupby('시간').agg(비생산관측수=('고전력_비생산','size'), 높은전력수=('고전력_비생산','sum'), 발생률=('고전력_비생산','mean'))
zero_hour['발생률'].mul(100).plot.bar(ax=axes[1,0])
axes[1,0].set(title='비생산 시간 중 높은 전력 발생률', ylabel='%')
sns.histplot(zero_runs['지속시간'], bins=25, ax=axes[1,1])
axes[1,1].set(title='연속 비생산 구간의 길이', xlabel='시간')
plt.tight_layout(); plt.show()
display(zero_hour)
display(zero_analysis.loc[zero_high].sort_values('전력평균계산', ascending=False).head(20))
save_table(zero_analysis.loc[zero_high], 'zero_production_high_power', index=False)
save_table(zero_runs, 'zero_production_runs')
save_table(zero_hour, 'zero_production_by_hour')


In [ ]:
def event_windows(frame, positions, offsets, columns):
    records = []
    for event_id, pos in enumerate(positions):
        if pos + min(offsets) < 0 or pos + max(offsets) >= len(frame):
            continue
        for offset in offsets:
            row = frame.iloc[pos + offset]
            records.append({'사건번호': event_id, '상대시간': offset, **{col: row[col] for col in columns}})
    return pd.DataFrame(records, columns=['사건번호','상대시간'] + columns)

def plot_event_profile(windows, column, ax, title, xlabel):
    if windows.empty:
        ax.set_title(title + ' (완전한 관측창 없음)')
        return
    grouped = windows.groupby('상대시간')[column]
    stats = grouped.agg(['mean','count'])
    lower, upper = grouped.quantile(.25), grouped.quantile(.75)
    ax.plot(stats.index, stats['mean'], marker='o', label='평균')
    ax.fill_between(stats.index, lower, upper, alpha=.2, label='25~75%')
    ax.axvline(0, color='tomato', linestyle='--')
    ax.set(title=f"{title} (사건 {windows['사건번호'].nunique()}개)", xlabel=xlabel, ylabel=column)
    ax.legend()

previous_production = df['생산량'].shift(1)
start_positions = np.flatnonzero(previous_production.eq(0) & df['생산량'].gt(0))
end_positions = np.flatnonzero(previous_production.gt(0) & df['생산량'].eq(0))
transition_windows = {}
fig, axes = plt.subplots(2,2, figsize=(14,8))
for row, (label, positions) in enumerate([('생산 시작',start_positions), ('생산 종료',end_positions)]):
    windows = event_windows(df, positions, range(-3,4), ['전력평균계산','생산량'])
    transition_windows[label] = windows
    for ax, col in zip(axes[row], ['전력평균계산','생산량']):
        plot_event_profile(windows, col, ax, label, '전환 기준 상대시간 (시간)')
    save_table(windows, 'production_start_windows' if row == 0 else 'production_end_windows', index=False)
plt.tight_layout(); plt.show()
transition_counts = pd.DataFrame({'시작':df.iloc[start_positions]['시간'].value_counts(), '종료':df.iloc[end_positions]['시간'].value_counts()}).reindex(range(24)).fillna(0).astype(int)
transition_counts.plot.bar(figsize=(13,4))
plt.title('시간대별 생산 시작·종료 횟수'); plt.ylabel('사건 수')
plt.tight_layout(); plt.show()
save_table(transition_counts, 'production_transition_counts')


In [ ]:
peak_mask = long_df['피크후보95']
peak_run_id = peak_mask.ne(peak_mask.shift()).cumsum()
peak_events = long_df.loc[peak_mask].groupby(peak_run_id[peak_mask]).agg(
    첫관측=('관측시각','min'), 마지막관측=('관측시각','max'),
    관측수=('전력값','size'), 최대전력=('전력값','max'))
peak_events['지속분'] = peak_events['관측수'] * 15
peak_events['경계사건'] = peak_events['첫관측'].eq(long_df['관측시각'].min()) | peak_events['마지막관측'].eq(long_df['관측시각'].max())
peak_starts = np.flatnonzero(peak_mask & ~peak_mask.shift(1, fill_value=False))
peak_windows = event_windows(long_df, peak_starts, range(-8,9), ['전력값','생산량'])
peak_windows['상대시간'] = peak_windows['상대시간'] * 15
fig, axes = plt.subplots(2,2, figsize=(14,8))
plot_event_profile(peak_windows, '전력값', axes[0,0], '고전력 구간 시작 전후', '첫 고전력 관측 기준 (분)')
plot_event_profile(peak_windows, '생산량', axes[0,1], '원본 시간별 생산량', '첫 고전력 관측 기준 (분)')
duration_group = pd.cut(peak_events['지속분'], [0,15,30,45,np.inf], labels=['15분','30분','45분','60분 이상'])
duration_group.value_counts(sort=False).plot.bar(ax=axes[1,0])
axes[1,0].set(title='고전력 사건 지속시간', ylabel='사건 수')
long_df.iloc[peak_starts]['시간'].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=axes[1,1])
axes[1,1].set(title='원본 시간구간별 고전력 시작 횟수', xlabel='원본 시간', ylabel='사건 수')
plt.tight_layout(); plt.show()
display(peak_events.sort_values('지속분', ascending=False).head(15))
save_table(peak_events, 'peak_events')
save_table(peak_windows, 'peak_event_windows', index=False)


In [ ]:
positive = df.loc[df['생산량'].gt(0)].copy()
matched_summary = positive.groupby(['생산량구간','시간'], observed=True).agg(
    관측수=('전력평균계산','size'), 중앙값=('전력평균계산','median'),
    상위90=('전력평균계산', lambda x: x.quantile(.9)),
    평균생산량=('생산량','mean')).reset_index()
count_grid = matched_summary.pivot(index='생산량구간', columns='시간', values='관측수').reindex(columns=range(24)).fillna(0)
fig, axes = plt.subplots(3,1, figsize=(16,11))
for ax, metric in zip(axes[:2], ['중앙값','상위90']):
    grid = matched_summary.pivot(index='생산량구간', columns='시간', values=metric).reindex(columns=range(24))
    sns.heatmap(grid, mask=count_grid.lt(10), cmap='YlOrRd', annot=True, fmt='.0f', ax=ax)
    ax.set_title(f'생산량 구간 × 시간: 전력 {metric} (n≥10)')
sns.heatmap(count_grid, cmap='Blues', annot=True, fmt='.0f', ax=axes[2])
axes[2].set_title('조합별 표본 수')
plt.tight_layout(); plt.show()
display(matched_summary.sort_values(['생산량구간','시간']))
save_table(matched_summary, 'production_bin_hour_power', index=False)
print('추가 분석 CSV 저장 위치:', CSV_DIR)


In [ ]:
daily_profiles = df.pivot(index='날짜', columns='시간', values=POWER_COLS).sort_index()
duplicate_days = daily_profiles.index[daily_profiles.duplicated(keep=False)]
representative_days = daily_profiles.index[~daily_profiles.duplicated(keep='first')]
suspect_dates = pd.to_datetime(['2021-07-13','2021-07-15'])
sensitivity_sets = {
    '전체': df,
    '이상 날짜 제외': df.loc[~df['날짜'].isin(suspect_dates)],
    '네 전력 모두 0 제외': df.loc[~df[POWER_COLS].eq(0).all(axis=1)],
    '동일 일별패턴 첫 날짜만': df.loc[df['날짜'].isin(representative_days)]
}
fixed_threshold = long_df['전력값'].quantile(.95)
sensitivity_rows = []
fig, axes = plt.subplots(1,2, figsize=(15,5))
monthly_sensitivity = []
for label, part in sensitivity_sets.items():
    flag = part[POWER_COLS].ge(fixed_threshold)
    sensitivity_rows.append({'조건':label, '시간수':len(part), '날짜수':part['날짜'].nunique(),
        '평균전력':part['전력평균계산'].mean(), '피크비율':flag.to_numpy().mean(),
        '평균생산량':part['생산량'].mean(), '주말비율':part['주말'].mean()})
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0], label=label)
    monthly = flag.mean(axis=1).groupby(part['월']).mean()
    monthly.mul(100).plot(ax=axes[1], marker='o', label=label)
    monthly_sensitivity.append(monthly.rename(label))
axes[0].set(title='제외 조건별 시간대 평균전력', ylabel='전력값')
axes[1].set(title=f'동일 기준 {fixed_threshold:g} 이상: 월별 비율', ylabel='%')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
sensitivity_summary = pd.DataFrame(sensitivity_rows).set_index('조건')
display(sensitivity_summary)
save_table(sensitivity_summary, 'sensitivity_summary')
save_table(pd.concat(monthly_sensitivity, axis=1), 'sensitivity_monthly_peak')


In [ ]:
repeat_eda = df.copy()
repeat_eda['패턴집단'] = np.where(repeat_eda['날짜'].isin(duplicate_days), '반복 패턴 날짜', '고유 패턴 날짜')
repeat_eda['피크비율'] = repeat_eda[POWER_COLS].ge(fixed_threshold).mean(axis=1)
repeat_summary = repeat_eda.groupby('패턴집단').agg(날짜수=('날짜','nunique'), 시간수=('날짜','size'),
    평균전력=('전력평균계산','mean'), 평균생산량=('생산량','mean'), 피크비율=('피크비율','mean'))
display(repeat_summary)
fig, axes = plt.subplots(2,2, figsize=(15,9))
for label, part in repeat_eda.groupby('패턴집단'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0,0], label=label)
    part.groupby('시간')['피크비율'].mean().mul(100).plot(ax=axes[0,1], label=label)
axes[0,0].set(title='패턴 집단별 시간대 평균전력', ylabel='전력값')
axes[0,1].set(title='패턴 집단별 고전력 비율', ylabel='%')
for ax in axes[0]: ax.legend()
daily_groups = repeat_eda.drop_duplicates('날짜')
pd.crosstab(daily_groups['월'], daily_groups['패턴집단']).plot.bar(ax=axes[1,0])
axes[1,0].set(title='집단별 월 구성', ylabel='날짜 수')
pd.crosstab(daily_groups['요일번호'], daily_groups['패턴집단']).plot.bar(ax=axes[1,1])
axes[1,1].set(title='집단별 요일 구성 (0=월, 6=일)', ylabel='날짜 수')
plt.tight_layout(); plt.show()
# 동일 프로파일 날짜 목록: 해시 대신 실제 전력 열 전체로 그룹화
profile_ids = daily_profiles.groupby(list(daily_profiles.columns), sort=False).ngroup().rename('프로파일ID')
profile_catalog = profile_ids.reset_index()
profile_catalog['같은프로파일날짜수'] = profile_catalog.groupby('프로파일ID')['날짜'].transform('size')
display(profile_catalog.loc[profile_catalog['같은프로파일날짜수'].gt(1)].head(20))
save_table(repeat_summary, 'repeated_vs_unique_summary')
save_table(profile_catalog, 'daily_profile_catalog', index=False)


In [ ]:
# 16번에서 만든 profile_catalog와 원본 시간별 df를 연결합니다.
repeat_detail = df.merge(
    profile_catalog[['날짜','프로파일ID','같은프로파일날짜수']],
    on='날짜', how='left', validate='many_to_one'
)
repeat_detail = repeat_detail.loc[repeat_detail['같은프로파일날짜수'].gt(1)].copy()
COMPARE_COLS = ['생산량','기온','풍속','습도','강수량','전기요금(계절)','공장인원','인건비']
weekday_names = ['월','화','수','목','금','토','일']
group_rows, match_rows, gap_rows = [], [], []
for profile_id, part in repeat_detail.groupby('프로파일ID'):
    dates = pd.DatetimeIndex(part['날짜'].unique()).sort_values()
    gaps = dates.to_series().diff().dt.days.dropna().astype(int)
    for previous, current in zip(dates[:-1], dates[1:]):
        gap_rows.append({'프로파일ID':profile_id, '이전날짜':previous,
                         '다음날짜':current, '간격일':(current-previous).days})
    group_rows.append({
        '프로파일ID':profile_id, '반복날짜수':len(dates),
        '날짜목록':', '.join(dates.strftime('%Y-%m-%d')),
        '요일목록':', '.join(weekday_names[i] for i in sorted(set(dates.dayofweek))),
        '최소간격일':gaps.min(), '중앙간격일':gaps.median(), '최대간격일':gaps.max(),
        '7일간격비율':gaps.eq(7).mean(),
        '7일배수간격비율':gaps.mod(7).eq(0).mean()
    })
    # 96개 전력값 일치 여부를 독립적으로 확인합니다.
    power_matrix = part.pivot(index='날짜', columns='시간', values=POWER_COLS)
    assert len(power_matrix.drop_duplicates()) == 1
    for col in COMPARE_COLS:
        matrix = part.pivot(index='날짜', columns='시간', values=col).reindex(columns=range(24))
        unique_patterns = len(matrix.drop_duplicates())
        match_rows.append({'프로파일ID':profile_id, '변수':col,
            '반복날짜수':len(dates), '고유24시간패턴수':unique_patterns,
            '모든날짜동일':unique_patterns==1, '결측수':int(matrix.isna().sum().sum()),
            '날짜간최대시간별차이':(matrix.max(axis=0)-matrix.min(axis=0)).max()})
repeat_group_summary = pd.DataFrame(group_rows).sort_values(
    ['반복날짜수','프로파일ID'], ascending=[False,True])
repeat_variable_matches = pd.DataFrame(match_rows)
repeat_date_gaps = pd.DataFrame(gap_rows)
display(repeat_group_summary)
match_matrix = repeat_variable_matches.pivot(index='프로파일ID', columns='변수', values='모든날짜동일')
display(match_matrix.reindex(repeat_group_summary['프로파일ID']).rename_axis(columns='변수: True=24시간 값 모두 동일'))
fig, ax = plt.subplots(figsize=(12, max(5, len(match_matrix)*.22)))
sns.heatmap(match_matrix.reindex(repeat_group_summary['프로파일ID'])[COMPARE_COLS].astype(int),
            vmin=0, vmax=1, cmap=['#eeeeee','#287da8'], cbar=False, ax=ax)
ax.set(title='동일 전력 그룹 안의 변수 일치: 파랑=동일, 회색=다름', ylabel='프로파일ID')
plt.tight_layout(); plt.show()
save_table(repeat_group_summary, 'repeat_group_summary', index=False)
save_table(repeat_variable_matches, 'repeat_variable_matches', index=False)
save_table(repeat_date_gaps, 'repeat_date_gaps', index=False)
save_table(repeat_detail, 'repeat_group_hourly_data', index=False)


In [ ]:
# 다른 그룹을 보려면 None 대신 위 표의 프로파일ID를 입력하세요. 예: 4
SELECT_PROFILE_ID = None
selected_profile = int(repeat_group_summary.iloc[0]['프로파일ID']) if SELECT_PROFILE_ID is None else SELECT_PROFILE_ID
selected = repeat_detail.loc[repeat_detail['프로파일ID'].eq(selected_profile)].sort_values(['날짜','시간'])
assert not selected.empty, '반복그룹 목록에 있는 프로파일ID를 선택하세요.'
selected_dates = sorted(selected['날짜'].unique())
daily_comparison = selected.groupby('날짜').agg(
    요일번호=('요일번호','first'), 총생산량=('생산량',lambda s:s.sum(min_count=1)),
    평균기온=('기온','mean'), 평균습도=('습도','mean'), 평균풍속=('풍속','mean'),
    평균강수량관측값=('강수량','mean'), 계절계수=('전기요금(계절)','first'),
    시간수정수=('시간수정','sum'))
daily_comparison['요일'] = daily_comparison.pop('요일번호').map(dict(enumerate(weekday_names)))
print(f'선택 그룹: {selected_profile} / {len(selected_dates)}일')
display(daily_comparison)
display(repeat_variable_matches.loc[repeat_variable_matches['프로파일ID'].eq(selected_profile)])
# 전력 96개는 완전히 같으므로 대표 날짜 하나만 표시합니다.
representative = selected.loc[selected['날짜'].eq(selected_dates[0])].sort_values('시간')
fig, ax = plt.subplots(figsize=(14,3.5))
ax.plot(np.arange(1,97)*.25, representative[POWER_COLS].to_numpy().ravel())
ax.set(title=f'그룹 {selected_profile}: 공통 전력 곡선 ({len(selected_dates)}일 동일)',
       xlabel='하루 시작 후 시간 (구간 끝)', ylabel='전력값', xticks=range(0,25,2))
plt.tight_layout(); plt.show()
# 변수별로 날짜를 색으로 구분합니다. 값이 같으면 선이 겹칩니다.
colors = plt.cm.turbo(np.linspace(.05,.95,len(selected_dates)))
for col in COMPARE_COLS:
    fig, ax = plt.subplots(figsize=(14,4))
    for date, color in zip(selected_dates, colors):
        part = selected.loc[selected['날짜'].eq(date)]
        ax.plot(part['시간'],part[col],color=color,lw=1,alpha=.8,
                label=pd.Timestamp(date).strftime('%Y-%m-%d'))
    ax.set(title=f'그룹 {selected_profile}: 날짜별 {col}',xlabel='시간',ylabel=col,xticks=range(0,24,2))
    ax.legend(loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,ncol=max(1,(len(selected_dates)+19)//20))
    plt.tight_layout(); plt.show()
save_table(daily_comparison, f'repeat_profile_{selected_profile}_daily_summary')
print('생산량과 기상이 달라도 전력은 동일할 수 있습니다. 일치 결과만으로 데이터 제작 의도를 확정하지 않습니다.')


In [ ]:
within_hour = df[['날짜시간','시간','생산량','생산여부']].copy()
within_hour['시간내변화'] = df['60분'] - df['15분']
within_hour['시간내범위'] = df[POWER_COLS].max(axis=1) - df[POWER_COLS].min(axis=1)
quarter_summary = pd.DataFrame({'평균':df[POWER_COLS].mean(), '중앙값':df[POWER_COLS].median(),
    '피크비율':df[POWER_COLS].ge(fixed_threshold).mean()})
fig, axes = plt.subplots(2,2, figsize=(14,8))
quarter_summary[['평균','중앙값']].plot.bar(ax=axes[0,0])
axes[0,0].set_title('15·30·45·60분 전력 수준')
quarter_summary['피크비율'].mul(100).plot.bar(ax=axes[0,1])
axes[0,1].set(title='분 구간별 고전력 비율', ylabel='%')
sns.boxplot(data=within_hour, x='시간', y='시간내변화', showfliers=False, ax=axes[1,0])
axes[1,0].axhline(0, color='gray', linestyle='--')
axes[1,0].set_title('시간대별 60분−15분 (이상점 표시 생략)')
axes[1,1].scatter(within_hour['생산량'], within_hour['시간내범위'], s=9, alpha=.25)
axes[1,1].set(title='생산량과 시간 내 변동 범위', xlabel='생산량', ylabel='최대−최소 전력')
plt.tight_layout(); plt.show()
display(quarter_summary)
save_table(quarter_summary, 'quarter_position_summary')
save_table(within_hour, 'within_hour_variation', index=False)


In [ ]:
ratio_eda = df.loc[df['생산량'].gt(0) & df[POWER_COLS].sum(axis=1).gt(0)].copy()
ratio_eda['생산전력비율'] = ratio_eda['생산량'] / ratio_eda[POWER_COLS].sum(axis=1)
fig, axes = plt.subplots(1,3, figsize=(18,5))
sns.boxplot(data=ratio_eda, x='시간', y='생산전력비율', showfliers=False, ax=axes[0])
sns.boxplot(data=ratio_eda, x='월', y='생산전력비율', showfliers=False, ax=axes[1])
sns.boxplot(data=ratio_eda, x='생산량구간', y='생산전력비율', showfliers=False, ax=axes[2])
for ax,title in zip(axes, ['시간대별 비율','월별 비율','생산량 구간별 비율']):
    ax.set_title(title + ' (이상점 표시 생략)')
axes[2].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()
ratio_summary = ratio_eda.groupby('시간')['생산전력비율'].agg(['count','median','mean'])
display(ratio_summary)
save_table(ratio_summary, 'production_power_ratio_by_hour')
save_table(ratio_eda[['날짜시간','생산량','생산전력비율']], 'production_power_ratio', index=False)


In [ ]:
# ============================================================
# 1. 전력 시차 상관관계
# ============================================================

# long_df는 관측시각 기준으로 정렬되어 있어야 함
long_df = long_df.sort_values('관측시각').reset_index(drop=True)

# 15분 간격의 고정 시차
lag_steps = {
    '15분 전': 1,
    '30분 전': 2,
    '1시간 전': 4,
    '24시간 전': 96,
    '1주 전': 672,
    '2주 전': 1344,
    '3주 전': 2016,
    '4주 전': 2688
}

lag_frame = pd.DataFrame({
    '관측시각': long_df['관측시각'],
    '현재전력': long_df['전력값']
})

# 고정된 15분 단위 시차 생성
for label, steps in lag_steps.items():
    lag_frame[label] = long_df['전력값'].shift(steps)


# ------------------------------------------------------------
# 달력 기준 한 달 전 같은 시각의 전력
# ------------------------------------------------------------

# 관측시각별 전력 조회용 Series
power_by_time = long_df.set_index('관측시각')['전력값']

# 각 현재 시각에서 달력 기준으로 한 달 전 시각 계산
previous_month_time = (
    long_df['관측시각']
    .apply(lambda x: x - pd.DateOffset(months=1))
)

# 한 달 전 같은 시각의 전력 연결
lag_frame['한 달 전'] = power_by_time.reindex(
    previous_month_time
).to_numpy()


# ------------------------------------------------------------
# 모든 시차를 동일한 관측 구간에서 비교
# ------------------------------------------------------------

comparison_cols = list(lag_steps.keys()) + ['한 달 전']

lag_common = lag_frame[
    ['현재전력'] + comparison_cols
].dropna()

lag_summary = pd.DataFrame({
    'Pearson': [
        lag_common['현재전력'].corr(lag_common[col])
        for col in comparison_cols
    ],
    'Spearman': [
        lag_common['현재전력'].corr(
            lag_common[col],
            method='spearman'
        )
        for col in comparison_cols
    ],
    '관측수': len(lag_common)
}, index=comparison_cols)


# ------------------------------------------------------------
# 시차별 산점도
# ------------------------------------------------------------

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flat

for ax, label in zip(axes, comparison_cols):
    ax.hexbin(
        lag_common[label],
        lag_common['현재전력'],
        gridsize=35,
        mincnt=1,
        cmap='Blues'
    )

    ax.set(
        title=(
            f"{label}: "
            f"Pearson={lag_summary.loc[label, 'Pearson']:.3f}"
        ),
        xlabel=label,
        ylabel='현재전력'
    )

# 사용하지 않은 축만 숨김
for ax in list(axes)[len(comparison_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

display(lag_summary)


# ============================================================
# 2. 생산량 시차와 전력의 상관관계
# ============================================================

# 비교 시각을 통일하고 양끝 6시간 제외
production_lags = pd.DataFrame({
    lag: df['생산량'].shift(lag)
    for lag in range(-6, 7)
})

production_lags['전력'] = df['전력평균계산']
production_lags = production_lags.dropna()

cross_corr = pd.Series({
    lag: production_lags[lag].corr(
        production_lags['전력'],
        method='spearman'
    )
    for lag in range(-6, 7)
}, name='Spearman')

fig, ax = plt.subplots(figsize=(11, 4))

cross_corr.plot.bar(ax=ax)

ax.set(
    title='생산량 시차와 현재 전력의 관계',
    xlabel='생산량 시차 (시간): 양수=과거, 음수=미래',
    ylabel='상관계수'
)

plt.tight_layout()
plt.show()

display(cross_corr.to_frame())


# ============================================================
# 3. 결과 저장
# ============================================================

save_table(
    lag_summary,
    'power_lag_correlations'
)

save_table(
    cross_corr.to_frame(),
    'production_power_cross_correlations'
)

# 2. 공통 전처리와 LightGBM 모델

In [ ]:
import os, sys, platform, random, json, time, types, io, contextlib, warnings, shutil, tempfile
os.environ["PYTHONHASHSEED"] = "42"
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import lightgbm as lgb
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
_fonts = {f.name for f in fm.fontManager.ttflist}
_kf = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"] if f in _fonts), None)
if _kf: plt.rcParams["font.family"] = _kf
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ROOT = PROJECT_ROOT
assert (ROOT / "dataset/okm_augumented_2021.csv").exists(), "프로젝트 루트(data/ 폴더가 있는 곳)에서 실행하세요"
OUTDIR = ROOT / "results/team_submission_runtime/outputs"; (OUTDIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTDIR / "run_log.txt").write_text("", encoding="utf-8")


def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(OUTDIR / "run_log.txt", "a", encoding="utf-8") as f: f.write(line + "\n")


T_START = time.time()
log(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | lightgbm {lgb.__version__} | torch {torch.__version__} | seed {SEED}")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Jupyter 없이 스크립트로 검증할 때 사용

ROOT = PROJECT_ROOT
if not (ROOT / 'dataset/okm_augumented_2021.csv').exists():
    ROOT = PROJECT_ROOT
DATA_PATH = ROOT / 'dataset/okm_augumented_2021.csv'
OUT = ROOT / 'results/preprocessing_24h'
POWER_COLS = ['15분', '30분', '45분', '60분']
MINUTES = [15, 30, 45, 60]
HORIZON = 24
VALID_START = pd.Timestamp('2021-07-01')
TEST_START = pd.Timestamp('2021-08-01')
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
# 기존 전처리에서 사용한 대한민국 2021년 공휴일 달력을 유지합니다.
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])
raw = pd.read_csv(DATA_PATH)
print('원본:', raw.shape)

In [ ]:
data = raw.copy()
data['date'] = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
data['original_hour'] = data['시간']
row_hour = data.groupby('date', sort=False).cumcount()
valid_hour = data['시간'].between(0, 23) & data['시간'].eq(data['시간'].round())
assert data.groupby('date').size().eq(24).all(), '날짜별 행 수 확인 필요'
assert data.loc[valid_hour, '시간'].eq(row_hour[valid_hour]).all(), '원본 행 순서 확인 필요'
data['time_repaired'] = ~valid_hour
data.loc[~valid_hour, '시간'] = row_hour[~valid_hour]
data['origin'] = data['date'] + pd.to_timedelta(data['시간'], unit='h')
data = data.sort_values('origin').reset_index(drop=True)
assert data['origin'].is_unique
assert data['origin'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert np.isfinite(data[POWER_COLS + ['생산량']].to_numpy(dtype=float)).all()
assert data[POWER_COLS + ['생산량']].ge(0).all().all()
data['suspect_date'] = data['date'].isin(SUSPECT_DATES)
# 11시 행 전체를 포함: 행 시작시각 기준 [8/28 18시, 8/29 12시).
data['special_period'] = data['origin'].ge('2021-08-28 18:00') & data['origin'].lt('2021-08-29 12:00')
calendar = pd.DataFrame({
    'month': data['origin'].dt.month,
    'day': data['origin'].dt.day,
    'hour': data['origin'].dt.hour,
    'weekday': data['origin'].dt.dayofweek,
    'weekend': data['origin'].dt.dayofweek.ge(5).astype(int),
    'holiday': data['date'].isin(HOLIDAYS).astype(int),
})
display(data.loc[data['time_repaired'], ['date', 'original_hour', '시간']].head())
print('복구 행:', int(data['time_repaired'].sum()))
print('범위:', data['origin'].min(), '~', data['origin'].max())

In [ ]:
# 시간축의 연속성을 검증했으므로 정수 위치로 정확한 시간 블록을 조회합니다.
starts = np.arange(336, len(data) - HORIZON + 1)
target_pos = starts[:, None] + np.arange(HORIZON)
w1_pos, w2_pos = target_pos - 168, target_pos - 336
power = data[POWER_COLS].to_numpy(dtype=float)
production = data['생산량'].to_numpy(dtype=float)
holiday = calendar['holiday'].to_numpy()
plan = production[target_pos]
sequence_parts = [plan[..., None]]
sequence_names = ['planned_production']
for name in calendar.columns:
    sequence_parts.append(calendar[name].to_numpy()[target_pos][..., None])
    sequence_names.append(name)
for week, pos in [(1, w1_pos), (2, w2_pos)]:
    sequence_parts.extend([
        power[pos], production[pos][..., None], holiday[pos][..., None],
        (plan - production[pos])[..., None],
        ((plan > 0) != (production[pos] > 0))[..., None],
        (holiday[target_pos] != holiday[pos])[..., None],
    ])
    sequence_names.extend([
        *[f'w{week}_power_{m}' for m in MINUTES],
        f'w{week}_production', f'w{week}_holiday',
        f'w{week}_production_difference', f'w{week}_production_state_mismatch',
        f'w{week}_holiday_mismatch',
    ])
sequence = np.concatenate(sequence_parts, axis=2).astype(float)
recent = power[starts - 1]
targets = power[target_pos]
recent_names = [f'recent_previous_hour_{m}' for m in MINUTES]
feature_names = recent_names + [f'h{h:02d}_{name}' for h in range(HORIZON) for name in sequence_names]
target_names = [f'h{h:02d}_power_{m}' for h in range(HORIZON) for m in MINUTES]
sample_ids = pd.Index(starts, name='sample_id')
X = pd.DataFrame(np.concatenate([recent, sequence.reshape(len(starts), -1)], axis=1), index=sample_ids, columns=feature_names)
y = pd.DataFrame(targets.reshape(len(starts), -1), index=sample_ids, columns=target_names)
print('X:', X.shape, '| y:', y.shape, '| sequence:', sequence.shape)

In [ ]:
meta = pd.DataFrame(index=sample_ids)
meta['origin'] = data['origin'].iloc[starts].to_numpy()
meta['target_end'] = meta['origin'] + pd.Timedelta(hours=HORIZON)
meta['origin_hour'] = meta['origin'].dt.hour
suspect = data['suspect_date'].to_numpy()
special = data['special_period'].to_numpy()
meta['target_suspect'] = suspect[target_pos].any(axis=1)
meta['recent_suspect'] = suspect[starts - 1]
meta['w1_suspect'] = suspect[w1_pos].any(axis=1)
meta['w2_suspect'] = suspect[w2_pos].any(axis=1)
meta['primary_eligible'] = ~meta['target_suspect']
meta['history_clean_eligible'] = ~meta[['target_suspect', 'recent_suspect', 'w1_suspect', 'w2_suspect']].any(axis=1)
meta['special_period_overlap'] = special[target_pos].any(axis=1)
meta['split'] = 'boundary_excluded'
meta.loc[meta['target_end'].le(VALID_START), 'split'] = 'train'
meta.loc[meta['origin'].ge(VALID_START) & meta['target_end'].le(TEST_START), 'split'] = 'validation'
meta.loc[meta['origin'].ge(TEST_START), 'split'] = 'test'
meta['final_retrain_candidate'] = meta['target_end'].le(TEST_START)
split_masks = {name: meta['split'].eq(name).to_numpy() for name in ['train', 'validation', 'test']}
split_masks['final_retrain_candidates'] = meta['final_retrain_candidate'].to_numpy()

# 기준은 겹친 윈도우가 아니라 1~6월 원본의 각 관측값을 딱 한 번 사용합니다.
threshold_source = data.loc[data['origin'].lt(VALID_START), POWER_COLS].to_numpy().ravel()
peak_threshold = float(np.quantile(threshold_source, 0.95))
hour_masks = {
    'suspect_target': suspect[target_pos],
    'special_period': special[target_pos],
    'weekend_production': (calendar['weekend'].to_numpy()[target_pos] == 1) & (plan > 0),
    'holiday_production': (holiday[target_pos] == 1) & (plan > 0),
    'zero_production': plan == 0,
    'production_both_weeks_zero': (plan > 0) & (production[w1_pos] == 0) & (production[w2_pos] == 0),
}
target_masks = {name: np.repeat(values, 4, axis=1) for name, values in hour_masks.items()}
for name, lo, hi in [('horizon_0_3', 0, 3), ('horizon_3_12', 3, 12), ('horizon_12_24', 12, 24)]:
    hours = (np.arange(24) >= lo) & (np.arange(24) < hi)
    target_masks[name] = np.broadcast_to(np.repeat(hours, 4), y.shape).copy()
target_masks['actual_peak'] = y.to_numpy() >= peak_threshold
split_summary = meta.groupby('split').agg(samples=('origin', 'size'), first_origin=('origin', 'min'), last_origin=('origin', 'max'), primary_samples=('primary_eligible', 'sum'), history_clean_samples=('history_clean_eligible', 'sum'))
display(split_summary)
print('고정 피크 기준(q95):', peak_threshold)

In [ ]:
assert X.index.equals(y.index) and y.index.equals(meta.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y.to_numpy()).all()
assert X.columns.is_unique and y.columns.is_unique
assert meta['origin'].iloc[0] == pd.Timestamp('2021-01-15')
assert meta.loc[meta['split'].eq('train'), 'origin'].max() == pd.Timestamp('2021-06-30')
for pos in [w1_pos, w2_pos]:
    latest_observed = data['origin'].to_numpy()[pos[:, -1]] + np.timedelta64(60, 'm')
    assert (latest_observed <= meta['origin'].to_numpy()).all()
assert (data['origin'].to_numpy()[starts - 1] + np.timedelta64(60, 'm') == meta['origin'].to_numpy()).all()
lookup = data.set_index('origin')
for i in [0, len(starts) // 2, len(starts) - 1]:
    origin = meta['origin'].iloc[i]
    times = pd.date_range(origin, periods=24, freq='h')
    np.testing.assert_array_equal(targets[i], lookup.loc[times, POWER_COLS].to_numpy())
    np.testing.assert_array_equal(recent[i], lookup.loc[origin - pd.Timedelta(hours=1), POWER_COLS].to_numpy(dtype=float))
    for week in [1, 2]:
        cols = [sequence_names.index(f'w{week}_power_{m}') for m in MINUTES]
        np.testing.assert_array_equal(sequence[i][:, cols], lookup.loc[times - pd.Timedelta(weeks=week), POWER_COLS].to_numpy())
for name, mask in split_masks.items():
    assert mask.any(), f'{name}: 빈 분할'
assert meta.loc[split_masks['train'], 'target_end'].le(VALID_START).all()
assert meta.loc[split_masks['validation'], 'origin'].ge(VALID_START).all()
assert meta.loc[split_masks['validation'], 'target_end'].le(TEST_START).all()
assert meta.loc[split_masks['test'], 'origin'].ge(TEST_START).all()
assert meta['target_end'].max() == data['origin'].max() + pd.Timedelta(hours=1)
assert len(threshold_source) == 181 * 24 * 4
np.testing.assert_array_equal(X.iloc[:, 4:].to_numpy().reshape(sequence.shape), sequence)
print('원본 대응·과거 정보 시각·정답 경계·배열 순서 검증 통과')

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for name, mask in split_masks.items():
    for prefix, frame in [('X', X), ('y', y), ('metadata', meta)]:
        frame.loc[mask].to_csv(OUT / f'{prefix}_{name}.csv', encoding='utf-8-sig', index=True)
    np.savez_compressed(
        OUT / f'arrays_{name}.npz',
        sample_id=sample_ids.to_numpy()[mask], sequence=sequence[mask], recent=recent[mask],
        y=targets[mask], baseline_w1=power[w1_pos][mask], baseline_w2=power[w2_pos][mask],
        baseline_weekly_mean=((power[w1_pos] + power[w2_pos]) / 2)[mask],
        **{key: value[mask] for key, value in target_masks.items()},
    )
meta.to_csv(OUT / 'all_sample_metadata.csv', encoding='utf-8-sig')
split_summary.to_csv(OUT / 'split_summary.csv', encoding='utf-8-sig')
data.to_csv(OUT / 'hourly_review.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'feature': feature_names}).to_csv(OUT / 'feature_catalog.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([
    {'target': f'h{h:02d}_power_{m}', 'hour_offset': h, 'minute_in_hour': m, 'observation_offset_minutes': h * 60 + m}
    for h in range(24) for m in MINUTES
]).to_csv(OUT / 'target_catalog.csv', index=False, encoding='utf-8-sig')
config = {
    'source_file': str(DATA_PATH.relative_to(ROOT)),
    'source_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'horizon_hours': 24, 'stride_hours': 1, 'history_hours': [168, 336],
    'feature_columns': feature_names, 'sequence_features': sequence_names,
    'recent_features': recent_names, 'target_columns': target_names,
    'flatten_order': 'hour first, then feature / power minute (15,30,45,60)',
    'peak_threshold': peak_threshold, 'peak_comparison': '>=',
    'threshold_source': 'unique raw observations Jan 1 through June 30; fixed after retraining',
    'threshold_observations': len(threshold_source),
    'validation_start': str(VALID_START), 'test_start': str(TEST_START),
    'holiday_dates': [str(d.date()) for d in HOLIDAYS],
    'suspect_dates': [str(d.date()) for d in SUSPECT_DATES],
    'primary_evaluation_mask': 'primary_eligible',
    'strict_evaluation_mask': 'history_clean_eligible',
    'special_period_row_interval': '[2021-08-28 18:00, 2021-08-29 12:00)',
    'operating_hour': None, 'final_retrain_suspect_policy': 'not yet selected; candidates only',
    'availability': '24h production plan known at origin; previous hour 60-minute power immediately available',
    'scaling': None, 'missing_value_policy': 'stop if required production or power is missing',
    'weather': 'excluded', 'random_seed_for_downstream_models': 42,
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
# 저장 후 다시 읽어 팀원이 불러올 때도 행 순서와 값이 유지되는지 확인합니다.
for name, mask in split_masks.items():
    saved_x = pd.read_csv(OUT / f'X_{name}.csv', index_col='sample_id')
    saved_y = pd.read_csv(OUT / f'y_{name}.csv', index_col='sample_id')
    assert saved_x.index.equals(saved_y.index)
    np.testing.assert_allclose(saved_x.to_numpy(), X.loc[mask].to_numpy())
    with np.load(OUT / f'arrays_{name}.npz') as saved:
        np.testing.assert_array_equal(saved['sample_id'], saved_x.index.to_numpy())
        np.testing.assert_allclose(saved['y'].reshape(len(saved_y), -1), saved_y.to_numpy())
print('저장 및 재로딩 검증 완료:', OUT)

In [ ]:
_ss = pd.read_csv(ROOT / "results/preprocessing_24h/split_summary.csv"); display(_ss)
log("전처리 완료: results/preprocessing_24h/ (arrays_*.npz, metadata_*.csv, hourly_review.csv, config.json)")

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = PROJECT_ROOT
PREP = ROOT / "results/preprocessing_24h"
PEAK = 176.0
_SEQ_DEFAULT = ["planned_production", "month", "day", "hour", "weekday", "weekend", "holiday", "w1_power_15", "w1_power_30", "w1_power_45", "w1_power_60",
                "w1_production", "w1_holiday", "w1_production_difference", "w1_production_state_mismatch", "w1_holiday_mismatch",
                "w2_power_15", "w2_power_30", "w2_power_45", "w2_power_60", "w2_production", "w2_holiday", "w2_production_difference",
                "w2_production_state_mismatch", "w2_holiday_mismatch"]
# 전처리 결과가 있으면 config.json을 읽고, 없으면(전처리 전) 기본값으로 모듈을 불러올 수 있게 한다.
CFG = json.loads((PREP / "config.json").read_text(encoding="utf-8")) if (PREP / "config.json").exists() else {}
SEQ = CFG.get("sequence_features", _SEQ_DEFAULT)
assert float(CFG.get("peak_threshold", PEAK)) == PEAK

# 현재 공식 Q1 하이퍼파라미터(03 §10, 06과 동일)
Q1_PARAMS = dict(
    objective="quantile", alpha=0.90, learning_rate=0.03, num_leaves=31, max_depth=-1,
    min_child_samples=50, subsample=0.85, subsample_freq=1, colsample_bytree=1.0,
    reg_alpha=1.0, reg_lambda=10.0, n_estimators=272, random_state=42, n_jobs=-1, verbose=-1,
)

REC = [f"rec_{m}" for m in (15, 30, 45, 60)]
CAL = ["month", "day", "hour", "weekday", "weekend", "holiday"]
HIST = ["w1_same", "w2_same", "w1_production", "w2_production", "w1_holiday", "w2_holiday",
        "w1_holiday_mismatch", "w2_holiday_mismatch", "minute", "lead", "hour_offset"]
PLAN = ["planned_production", "w1_production_difference", "w2_production_difference",
        "w1_production_state_mismatch", "w2_production_state_mismatch"]
EXTRA = ["two_week_mean_same", "rec_mean1h", "rec_std1h", "rec_change1h"]
Q1_FEATURES = REC + CAL + HIST + PLAN + EXTRA          # 현재 공식 Q1 30개


def as_bool(s):
    if s.dtype == bool:
        return s.to_numpy()
    return s.astype(str).str.lower().map({"true": True, "false": False}).to_numpy(bool)


def long_features(arrays, rows=None):
    """공식 Q1과 같은 15분 칸 단위 피처 표(창 × 96칸). 03 §10 / 06 셀 7과 동일 로직."""
    seq = arrays["sequence"] if rows is None else arrays["sequence"][rows]
    recent = arrays["recent"] if rows is None else arrays["recent"][rows]
    seq = seq.astype(np.float32); recent = recent.astype(np.float32); n = len(seq)
    idx = {name: i for i, name in enumerate(SEQ)}
    rep = lambda a: np.repeat(a, 4, axis=1).reshape(-1)
    v = {}
    for j, m in enumerate((15, 30, 45, 60)):
        v[f"rec_{m}"] = np.broadcast_to(recent[:, j, None], (n, 96)).reshape(-1)
    for name in [x for x in SEQ if "_power_" not in x]:
        v[name] = rep(seq[:, :, idx[name]])
    w1 = seq[:, :, [idx[f"w1_power_{m}"] for m in (15, 30, 45, 60)]].reshape(n, 96)
    w2 = seq[:, :, [idx[f"w2_power_{m}"] for m in (15, 30, 45, 60)]].reshape(n, 96)
    v["w1_same"] = w1.reshape(-1); v["w2_same"] = w2.reshape(-1)
    v["two_week_mean_same"] = ((w1 + w2) / 2).reshape(-1)
    v["minute"] = np.tile(np.array([15, 30, 45, 60], np.float32), n * 24)
    v["lead"] = np.tile(np.arange(96, dtype=np.float32), n)
    v["hour_offset"] = np.tile(np.repeat(np.arange(24, dtype=np.float32), 4), n)
    for name, x in {"rec_mean1h": recent.mean(axis=1), "rec_std1h": recent.std(axis=1),
                    "rec_change1h": recent[:, -1] - recent[:, 0]}.items():
        v[name] = np.broadcast_to(x[:, None], (n, 96)).reshape(-1)
    return pd.DataFrame(v, dtype=np.float32)


# ---------------------------------------------------------------- 3단 규칙
def rule_inputs(F):
    return (F["planned_production"].to_numpy() > 0, F["w1_production"].to_numpy() > 0, F["w2_production"].to_numpy() > 0)


def observation_table(sets):
    """주어진 (arrays, meta) 목록에서 '시각별 정답 관측' 표를 만든다(중복 시각 제거, 의심 칸 제외).
    열: ts(15분 구간 끝 시각), day(관측이 확정된 날), plan_on, hour, minute, y."""
    rows = []
    for a, m in sets:
        n = len(m); seq = a["sequence"].astype(np.float32)
        y = a["y"].reshape(n, 96)
        sus = a["suspect_target"].reshape(n, 96).astype(bool)
        h = np.tile(np.repeat(np.arange(24), 4), n); q = np.tile(np.arange(4), n * 24)
        win = np.repeat(np.arange(n), 96)
        origin = pd.to_datetime(m["origin"].to_numpy())[win]
        ts = origin + pd.to_timedelta(h, "h") + pd.to_timedelta(15 * (q + 1), "m")
        ih = SEQ.index("hour"); ip = SEQ.index("planned_production")
        df = pd.DataFrame(dict(ts=ts, plan_on=(seq[win, h, ip] > 0).astype(np.int8),
                               hour=seq[win, h, ih].astype(np.int8), minute=((q + 1) * 15).astype(np.int8),
                               y=y.reshape(-1), sus=sus.reshape(-1)))
        rows.append(df[~df.sus].drop(columns="sus"))
    obs = pd.concat(rows, ignore_index=True).drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
    obs["day"] = (obs.ts - pd.Timedelta(minutes=1)).dt.floor("D")
    return obs


class OOFProfile:
    """기준시각 이전에 이미 확정된 관측만으로 (가동 여부, 시, 분) 평균을 만든다.
    창의 기준일 D에 대해 '관측 day < D'(= D 00:00 이전 관측)만 사용 → 학습 정답이 자기 자신에게 섞이지 않는다."""
    def __init__(self, obs):
        self.days = np.sort(obs.day.unique())
        key = (obs.plan_on.astype(int) * 10000 + obs.hour.astype(int) * 100 + (obs.minute.astype(int) // 15)).to_numpy()
        self.keys = np.unique(key); kidx = {k: i for i, k in enumerate(self.keys)}
        di = {d: i for i, d in enumerate(self.days)}
        S = np.zeros((len(self.days), len(self.keys))); C = np.zeros_like(S)
        for k, d, y in zip(key, obs.day.to_numpy(), obs.y.to_numpy()):
            S[di[d], kidx[k]] += y; C[di[d], kidx[k]] += 1
        self.cS = np.cumsum(S, 0); self.cC = np.cumsum(C, 0); self.kidx = kidx
        self.gS = np.cumsum([obs[obs.day == d].groupby("plan_on").y.sum().reindex([0, 1]).fillna(0).to_numpy() for d in self.days], axis=0)
        self.gC = np.cumsum([obs[obs.day == d].groupby("plan_on").y.count().reindex([0, 1]).fillna(0).to_numpy() for d in self.days], axis=0)

    def lookup(self, origin_day, plan_on, hour, minute):
        """origin_day(셀별 창 기준일), 반환: 프로파일 값(없으면 plan_on 전체 평균, 그것도 없으면 NaN)."""
        pos = np.searchsorted(self.days, np.asarray(origin_day, dtype="datetime64[ns]"), side="left") - 1   # day < D 의 마지막 인덱스
        ok = pos >= 0; p = np.where(ok, pos, 0)
        key = plan_on.astype(int) * 10000 + hour.astype(int) * 100 + (minute.astype(int) // 15)
        ki = np.array([self.kidx.get(k, -1) for k in key]); has = ok & (ki >= 0)
        k0 = np.where(ki >= 0, ki, 0)
        cnt = np.where(has, self.cC[p, k0], 0); s = np.where(has, self.cS[p, k0], 0)
        prof = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
        gc = np.where(ok, self.gC[p, plan_on.astype(int)], 0); gs = np.where(ok, self.gS[p, plan_on.astype(int)], 0)
        glob = np.where(gc > 0, gs / np.maximum(gc, 1), np.nan)
        return np.where(np.isfinite(prof), prof, glob)


def rule3_oof(F, meta, profile):
    """3단 규칙 값과 단계(0: 1주 전 사용, 1: 2주 전 사용, 2: 평균 프로파일). 프로파일은 창 기준일 이전 관측만."""
    plan_on, w1_on, w2_on = rule_inputs(F)
    n = len(meta)
    origin_day = np.repeat(pd.to_datetime(meta["origin"]).dt.floor("D").to_numpy(), 96)
    fb = profile.lookup(origin_day, plan_on.astype(np.int8), F["hour"].to_numpy(np.int8), F["minute"].to_numpy(np.int8))
    w1 = F["w1_same"].to_numpy(); w2 = F["w2_same"].to_numpy()
    step = np.where(plan_on == w1_on, 0, np.where(plan_on == w2_on, 1, 2)).astype(np.float32)
    val = np.where(step == 0, w1, np.where(step == 1, w2, fb)).astype(np.float32)
    return val, step


def rule3_official(F_tr, y_tr_long, F_eval):
    """현재 공식 3단 규칙(학습 구간 전체 평균 프로파일, 학습 정답이 섞임) — OOF와 비교용."""
    pt, _, _ = rule_inputs(F_tr)
    pf = pd.DataFrame(dict(p=pt.astype(np.int8), h=F_tr["hour"].to_numpy(np.int8), m=F_tr["minute"].to_numpy(np.int8), y=y_tr_long))
    prof = pf.groupby(["p", "h", "m"]).y.mean(); glob = pf.groupby("p").y.mean()

    def f(F):
        po, w1o, w2o = rule_inputs(F)
        keys = pd.MultiIndex.from_arrays([po.astype(np.int8), F["hour"].to_numpy(np.int8), F["minute"].to_numpy(np.int8)])
        fb = prof.reindex(keys).to_numpy(); miss = ~np.isfinite(fb)
        if miss.any(): fb[miss] = pd.Series(po[miss].astype(np.int8)).map(glob).to_numpy()
        step = np.where(po == w1o, 0, np.where(po == w2o, 1, 2)).astype(np.float32)
        return np.where(step == 0, F["w1_same"].to_numpy(), np.where(step == 1, F["w2_same"].to_numpy(), fb)).astype(np.float32), step
    return f(F_tr), f(F_eval)


# ---------------------------------------------------------------- 지표
def metrics(y, p, thr=PEAK, origin_hour=None):
    a = y >= thr; q = p >= thr
    tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    e = p - y
    out = dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()),
               precision=pr, recall=rc, F1=f(1), F2=f(2), TP=tp, FP=fp, FN=fn, 칸=int(y.size))
    return out


def fp_per_day_00(y, p, origin_hour, thr=PEAK):
    """매일 00시 기준 24시간 예측만 모아 하루당 잘못된 경보 칸 수(비교 대상이 고르게 하루 1번 예측)."""
    sel = origin_hour == 0
    a = y[sel] >= thr; q = p[sel] >= thr
    return float((~a & q).sum() / max(sel.sum(), 1))


def tag_cells(F, meta):
    """상태 태그(예측 시점에 아는 계획 기준). 칸 단위 1차원 배열."""
    po, w1o, _ = rule_inputs(F)
    hol = F["holiday"].to_numpy() > 0
    tag = np.full(len(po), "정상 반복", dtype=object)
    tag[w1o & ~po] = "가동→정지"; tag[~w1o & po] = "정지→가동"; tag[po & hol] = "공휴일 가동"
    return tag



W_GRID = np.round(np.arange(0, 1.01, 0.1), 2)      # 고정 GRU 비중 w: pred = w*GRU + (1-w)*Q1

In [ ]:
ST_COLS = ["st_day_on", "st_day_total", "st_day_known_h", "st_prev_day_total", "st_prev_day_on", "st_off_run", "st_first_op",
           "st_hrs_since_prod", "st_win_plan_total", "st_win_on_hours", "st_win_all_off", "st_win_vs_w1", "st_win_w1_total", "st_win_w2_total", "st_hol_prod"]


class HourlyIndex:
    """hourly_review.csv(시간별 원자료)에서 생산량 배열과 날짜 색인을 만든다."""
    def __init__(self, hourly):
        self.P = hourly["생산량"].to_numpy(float)
        self.row_of = {t: i for i, t in enumerate(hourly["origin"])}
        self.date_d = hourly["origin"].dt.floor("D").to_numpy()
        self.day_first = {}
        for i, d in enumerate(self.date_d):
            a, b = self.day_first.get(d, (i, i)); self.day_first[d] = (a, i)
        self.days = sorted(self.day_first); self.day_idx = {d: k for k, d in enumerate(self.days)}


def state_features(hi, origins, seq, Fcells, P=None):
    """창(기준시각)마다 상태 전환 피처를 만든다. 반환: 길이 (창 수 x 96)의 DataFrame.
    P: 시간별 생산량 배열(기본은 실제 자료). 미래 값 교란 검사에서 바꾼 배열을 넣어 볼 수 있다."""
    P = hi.P if P is None else P
    cs = np.concatenate([[0.0], np.cumsum(P)])
    DT = np.array([cs[hi.day_first[d][1] + 1] - cs[hi.day_first[d][0]] for d in hi.days])
    zrun = np.zeros(len(hi.days), int)
    for k in range(len(hi.days)):
        zrun[k] = (zrun[k - 1] + 1 if k > 0 else 1) if DT[k] == 0 else 0
    last_pos = np.full(len(P), -1); lp = -1
    for i, v in enumerate(P):
        if v > 0: lp = i
        last_pos[i] = lp
    ip = SEQ.index("planned_production"); i1 = SEQ.index("w1_production"); i2 = SEQ.index("w2_production")
    n = len(origins); out = {c: np.full((n, 96), np.nan, np.float32) for c in ST_COLS}
    for w, o in enumerate(pd.to_datetime(origins)):
        r0 = hi.row_of[o]; dA = hi.date_d[r0]; a0, a1 = hi.day_first[dA]; kA = hi.day_idx[dA]
        win = P[r0:r0 + 24]
        totA = cs[a1 + 1] - cs[a0]
        b0 = a1 + 1; b1 = min(r0 + 23, len(P) - 1)
        totB = (cs[b1 + 1] - cs[b0]) if b1 >= b0 else 0.0
        knownB = max(b1 - b0 + 1, 0)
        prevA = (cs[a0] - cs[a0 - 24]) if a0 >= 24 else np.nan
        hs = (r0 - last_pos[r0 - 1]) if r0 > 0 and last_pos[r0 - 1] >= 0 else 336
        hs = min(hs, 336)
        w1t = seq[w, :, i1].sum(); w2t = seq[w, :, i2].sum(); pt = seq[w, :, ip].sum()
        for h in range(24):
            inA = (r0 + h) <= a1
            tot, kh, prev = (totA, 24, prevA) if inA else (totB, knownB, totA)
            run = zrun[kA - 1] if inA and kA > 0 else (zrun[kA] if not inA else 0)
            sl = slice(h * 4, h * 4 + 4)
            out["st_day_on"][w, sl] = float(tot > 0); out["st_day_total"][w, sl] = tot; out["st_day_known_h"][w, sl] = kh
            out["st_prev_day_total"][w, sl] = prev; out["st_prev_day_on"][w, sl] = (float(prev > 0) if np.isfinite(prev) else np.nan)
            out["st_off_run"][w, sl] = min(run, 14); out["st_first_op"][w, sl] = float(tot > 0 and run >= 2)
            out["st_hrs_since_prod"][w, sl] = hs
            out["st_win_plan_total"][w, sl] = pt; out["st_win_on_hours"][w, sl] = float((win > 0).sum()); out["st_win_all_off"][w, sl] = float(pt == 0)
            out["st_win_vs_w1"][w, sl] = pt - w1t; out["st_win_w1_total"][w, sl] = w1t; out["st_win_w2_total"][w, sl] = w2t
    df = pd.DataFrame({c: v.reshape(-1) for c, v in out.items()})
    df["st_hol_prod"] = ((Fcells["holiday"].to_numpy() > 0) & (Fcells["planned_production"].to_numpy() > 0)).astype(np.float32)
    return df

In [ ]:
GRID = np.arange(100.0, 231.0, 1.0)                     # 경보 기준 후보(예측값에 거는 임계값)


def m2(y, p, t, oh=None):
    """실제 피크는 y >= 176, 경보는 p >= t (t는 경보 기준)."""
    a = y >= PEAK; q = p >= t; tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    out = dict(MAE=float(np.abs(p - y).mean()), bias=float((p - y).mean()), precision=pr, recall=rc, F1=f(1), F2=f(2), TP=tp, FP=fp, FN=fn)
    if oh is not None: out["FP칸/일(00시)"] = float((~a & q)[oh == 0].sum() / max((oh == 0).sum(), 1))
    return out


def curve(y, p, grid=GRID):
    a = (y >= PEAK).reshape(-1); pr = p.reshape(-1); pa = np.sort(pr[a]); pn = np.sort(pr[~a])
    tp = len(pa) - np.searchsorted(pa, grid, "left"); fp = len(pn) - np.searchsorted(pn, grid, "left")
    prec = tp / np.maximum(tp + fp, 1); rec = tp / max(len(pa), 1)
    return pd.DataFrame(dict(t=grid, precision=prec, recall=rec, F2=5 * prec * rec / np.maximum(4 * prec + rec, 1e-12), FP=fp, TP=tp))


def choose(y, p, policy):
    """경보 기준 고르기. 'R85' = 재현율 ≥ .85 중 정밀도 최대(같으면 더 높은 기준), 'R90' 동일, 'F2' = F2 최대, 'T176' = 176 고정."""
    if policy == "T176": return 176.0
    cv = curve(y, p)
    if policy == "F2": return float(cv.t[int(cv.F2.to_numpy().argmax())])
    target = int(policy[1:]) / 100; ok = cv.recall >= target                    # "R85" -> 0.85, "R95" -> 0.95
    if not ok.any(): return float(cv.t.iloc[0])                       # 목표 재현율을 못 맞추면 가장 낮은 기준(재현율 최대)
    sub = cv[ok]; best = sub.precision.max(); return float(sub[sub.precision >= best - 1e-12].t.max())

In [ ]:
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

HIDDEN, LAYERS, DROPOUT, BATCH, LR, WD = 32, 2, 0.15, 64, 1e-3, 1e-4
PEAK_WEIGHT, EPOCHS, SEED = 3.0, 9, 42


class TunedGRURegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=29, hidden_size=HIDDEN, num_layers=LAYERS, batch_first=True, dropout=DROPOUT, bidirectional=True)
        self.output = nn.Linear(2 * HIDDEN, 4)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.output(out)


def _combine(sequence, recent):
    """시간별 입력 25개 + 직전 1시간 전력 4개를 24시간에 반복해 붙인다 → (n, 24, 29)."""
    rep = np.repeat(recent[:, None, :], sequence.shape[1], axis=1)
    return np.concatenate([sequence, rep], axis=2).astype(np.float32)


def train_predict(seq_tr, rec_tr, y_tr, base_tr, seq_ev, rec_ev, base_ev, peak=176.0, epochs=EPOCHS, seed=SEED, threads=None):
    """학습 창으로 학습하고 평가 창을 예측한다. 입력 y/baseline은 (n, 24, 4). 반환: (n_ev, 24, 4) 전력 예측(0 미만은 0)."""
    X_tr = _combine(seq_tr, rec_tr); X_ev = _combine(seq_ev, rec_ev)
    y_tr = y_tr.astype(np.float32); res_tr = y_tr - base_tr.astype(np.float32)
    xm = X_tr.mean(axis=(0, 1), keepdims=True); xs = X_tr.std(axis=(0, 1), keepdims=True); xs[xs < 1e-6] = 1.0
    rm = res_tr.mean(axis=(0, 1), keepdims=True); rs = res_tr.std(axis=(0, 1), keepdims=True); rs[rs < 1e-6] = 1.0
    Xs = ((X_tr - xm) / xs).astype(np.float32); Xe = ((X_ev - xm) / xs).astype(np.float32); Rs = ((res_tr - rm) / rs).astype(np.float32)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.set_num_threads(threads or min(8, os.cpu_count() or 1))
    gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xs), torch.from_numpy(Rs), torch.from_numpy(y_tr)), batch_size=BATCH, shuffle=True, generator=gen)
    model = TunedGRURegressor(); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); tot = n = 0
        for bx, bt, ba in loader:
            opt.zero_grad(); pred = model(bx)
            w = torch.where(ba >= peak, torch.as_tensor(PEAK_WEIGHT), torch.as_tensor(1.0))
            loss = (w * (pred - bt) ** 2).mean(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
            tot += loss.item() * len(bx); n += len(bx)
        hist.append(tot / n)
    model.eval()
    with torch.inference_mode():
        sp = model(torch.from_numpy(Xe)).numpy()
    return np.clip(base_ev.astype(np.float32) + sp * rs + rm, 0, None).astype(np.float32), hist

In [ ]:
def build_windows(prep=None, strict=True):
    """학습 가능한 창 + 테스트 창과 공통 피처 표. prep: 전처리 산출물 폴더."""
    prep = PREP if prep is None else Path(prep)
    fm_ = pd.read_csv(prep / "metadata_final_retrain_candidates.csv", parse_dates=["origin", "target_end"]); keep = as_bool(fm_["primary_eligible"])
    mt_ = pd.read_csv(prep / "metadata_test.csv", parse_dates=["origin", "target_end"])
    mv_ = pd.read_csv(prep / "metadata_validation.csv", parse_dates=["origin", "target_end"]); kv = as_bool(mv_["primary_eligible"])
    af, at, av = (np.load(prep / f"arrays_{n}.npz") for n in ("final_retrain_candidates", "test", "validation"))
    cat = lambda k: np.concatenate([af[k][keep], at[k]])
    W = types.SimpleNamespace()
    W.seq, W.rec = cat("sequence"), cat("recent")
    W.y3, W.base3 = cat("y").astype(np.float32), cat("baseline_w1").astype(np.float32)          # (N, 24, 4)
    W.N = len(W.seq); W.y = W.y3.reshape(W.N, 96); W.base = W.base3.reshape(W.N, 96); W.sus = cat("suspect_target")
    W.meta = pd.concat([fm_.loc[keep], mt_], ignore_index=True)[["origin", "target_end"]]
    idx = {o: i for i, o in enumerate(W.meta.origin)}
    W.ev_val = np.array([idx[o] for o in mv_.loc[kv, "origin"] if o in idx]); W.ev_test = np.array([idx[o] for o in mt_["origin"]])
    if strict:
        assert W.meta.origin.is_monotonic_increasing and W.N == 4635 + 1057, f"창 수 {W.N}"
        assert len(W.ev_val) == 627 and len(W.ev_test) == 1057
        assert np.array_equal(W.y[W.ev_val], av["y"][kv].reshape(-1, 96)), "검증 창 정답 불일치"
    W.EV = np.concatenate([W.ev_val, W.ev_test])
    W.hi = HourlyIndex(pd.read_csv(prep / "hourly_review.csv", parse_dates=["origin"]))
    W.F = long_features(dict(sequence=W.seq, recent=W.rec))
    st = state_features(W.hi, W.meta["origin"], W.seq, W.F)
    for c in ST_COLS: W.F[c] = st[c].to_numpy()
    prof = OOFProfile(observation_table([(dict(sequence=W.seq, y=W.y, suspect_target=W.sus), W.meta)]))
    r, s = rule3_oof(W.F, W.meta, prof)
    W.F["rule3_oof"] = r; W.F["rule_step"] = s; W.F["rule3_oof"] = W.F["rule3_oof"].fillna(W.F["w1_same"])
    W.BASE = Q1_FEATURES; W.L13 = W.BASE + ["rule3_oof", "rule_step"] + ST_COLS               # 공식 30피처 / 새 Q1 피처(47개)
    W.tr_val = (W.meta.target_end <= pd.Timestamp("2021-07-01")).to_numpy()                 # val 단계 학습 창
    W.tr_test = (W.meta.target_end <= pd.Timestamp("2021-08-01")).to_numpy()                # test 단계 학습 창
    W.mon = W.meta.origin.dt.month.to_numpy()
    return W


W = build_windows()
log(f"창 {W.N}개 | 학습 창 val {int(W.tr_val.sum())} / test {int(W.tr_test.sum())} | 평가 창 val {len(W.ev_val)} + test {len(W.ev_test)} | 피처 공식 {len(W.BASE)}개 / 새 Q1 {len(W.L13)}개")

In [ ]:
STAGES = {"val": (W.tr_val, W.ev_val, False), "test": (W.tr_test, W.ev_test, True)}      # (학습 마스크, 평가 창 위치, 예측을 0 이상으로 자를지)
GRU = {}
for stg, (trm, ev, _) in STAGES.items():
    t0 = time.time()
    p, hist = train_predict(W.seq[trm], W.rec[trm], W.y3[trm], W.base3[trm], W.seq[ev], W.rec[ev], W.base3[ev], seed=SEED)
    GRU[stg] = p.reshape(-1, 96)
    log(f"GRU({stg}) 학습 창 {int(trm.sum())} → 예측 창 {len(ev)} | 학습 {time.time() - t0:.0f}초 | 마지막 epoch 손실 {hist[-1]:.4f}")

In [ ]:
def fit_lgb(cols, alpha, tr_mask, te_mask, clip, weight=None):
    prm = dict(Q1_PARAMS); prm["alpha"] = alpha; rows_tr = np.repeat(tr_mask, 96); rows_te = np.repeat(te_mask, 96)
    m_ = lgb.LGBMRegressor(**prm).fit(W.F.loc[rows_tr, cols], W.y.reshape(-1)[rows_tr], sample_weight=None if weight is None else weight[rows_tr])
    p = m_.predict(W.F.loc[rows_te, cols]).reshape(-1, 96).astype(np.float32)
    return np.clip(p, 0, None) if clip else p


Q1_CUR = {}
for stg, (trm, ev, clip) in STAGES.items():
    te = np.zeros(W.N, bool); te[ev] = True; t0 = time.time()
    Q1_CUR[stg] = fit_lgb(W.BASE, 0.90, trm, te, clip); log(f"현재 Q1({stg}, 공식 30피처, q=0.90) {time.time() - t0:.0f}초")

WEEKS = pd.date_range("2021-05-31", "2021-09-13", freq="7D")
NEW_Q = 0.75
# 이례적 휴무 구간(연속 3평일 이상 생산 0)을 찾는다. 재학습 때 이미 끝난 구간의 창은 가중치 0.1로 낮춘다(안전장치).
_dp = pd.Series(W.hi.P).groupby(W.hi.date_d).sum(); _off = (_dp <= 0) & (pd.DatetimeIndex(_dp.index).dayofweek < 5); STREAKS = []; _run = []
for _d, _o in _off.items():
    if _o: _run.append(_d)
    else:
        if len(_run) >= 3: STREAKS.append((_run[0], _run[-1]))
        _run = []
if len(_run) >= 3: STREAKS.append((_run[0], _run[-1]))
log(f"이례적 휴무 구간(연속 3평일 이상 생산 0): {[(str(a.date()), str(b.date())) for a, b in STREAKS]}")
Q1_NEW = np.full((W.N, 96), np.nan, np.float32); t0 = time.time(); schedule = []
for S in WEEKS:
    tr_m = (W.meta.target_end <= S).to_numpy(); te_m = ((W.meta.origin >= S) & (W.meta.origin < S + pd.Timedelta(days=7))).to_numpy()
    if te_m.sum() == 0: continue
    assert W.meta.target_end[tr_m].max() <= S, "학습 창이 주 시작 이후에 끝나는 타깃을 포함"       # 미래 정보 금지 확인
    known = np.zeros(W.N, bool)                                                                   # 재학습 시점에 이미 끝난 휴무 구간과 겹치는 창
    for a, b in STREAKS:
        if b + pd.Timedelta(days=1) <= S: known |= ((W.meta.origin < b + pd.Timedelta(days=1)) & (W.meta.target_end > a)).to_numpy()
    Q1_NEW[te_m] = fit_lgb(W.L13, NEW_Q, tr_m, te_m, True, weight=np.repeat(np.where(known, 0.1, 1.0), 96))
    schedule.append(dict(주시작=S.date(), 학습창=int(tr_m.sum()), 예측창=int(te_m.sum()), 가중치낮춘창=int((known & tr_m).sum())))
log(f"새 Q1(L1+L3, q={NEW_Q}) 주간 재학습 {len(schedule)}주 {time.time() - t0:.0f}초")
display(pd.DataFrame(schedule).T)

In [ ]:
WK_OF = ((W.meta.origin - WEEKS[0]) // pd.Timedelta(days=7)).to_numpy()


HIST_LOG = []                                    # (주 시작, 이력으로 쓴 창의 가장 늦은 타깃 종료 시각) — 자체 검증에서 확인


def window_thresholds(pred, policy="R85-최근4주", target="R85"):
    """모든 창에 대한 경보 기준 벡터. 그 주 시작 전에 타깃이 끝난 창의 (예측, 정답)만 이력으로 쓴다."""
    thr = np.full(W.N, 176.0)
    if policy == "T176": return thr
    for w in np.unique(WK_OF[W.EV]):
        S = WEEKS[0] + pd.Timedelta(days=7) * int(w)
        hist = np.isfinite(pred[:, 0]) & (W.meta.target_end <= S).to_numpy() & (W.meta.origin >= WEEKS[0]).to_numpy()
        if policy == "R85-최근4주": hist &= (W.meta.origin >= S - pd.Timedelta(days=28)).to_numpy()
        if hist.sum() > 0:
            thr[WK_OF == w] = choose(W.y[hist], pred[hist], target); HIST_LOG.append((S, W.meta.target_end[hist].max()))
    return thr


THR_NEW = window_thresholds(Q1_NEW)
_wk = pd.DataFrame({"주시작": [(WEEKS[0] + pd.Timedelta(days=7) * int(w)).date() for w in np.unique(WK_OF[W.EV])],
                    "경보기준": [THR_NEW[WK_OF == w][0] for w in np.unique(WK_OF[W.EV])]})
display(_wk.set_index("주시작").T)

In [ ]:
RULE = W.F["rule3_oof"].to_numpy().reshape(W.N, 96).astype(np.float32)
GATE = (W.F["st_day_on"].to_numpy().reshape(W.N, 96) == 0)                  # 목표 날 생산계획 없음(휴무일)
ON_ROWS = (~GATE).reshape(-1)                                                  # 잔차 모델 학습에 쓰는 칸 = 가동일 칸
RES_TARGET = (W.y - RULE).reshape(-1).astype(np.float64); CURVE_RES = np.full((W.N, 96), np.nan, np.float32); t0 = time.time()
for S in WEEKS:
    tr_m = (W.meta.target_end <= S).to_numpy(); te_m = ((W.meta.origin >= S) & (W.meta.origin < S + pd.Timedelta(days=7))).to_numpy()
    if te_m.sum() == 0: continue
    prm = dict(Q1_PARAMS); prm["objective"] = "regression_l1"; rows_tr = np.repeat(tr_m, 96) & ON_ROWS; rows_te = np.repeat(te_m, 96)      # 학습은 가동일 칸만(휴무일 칸은 아래에서 프로파일을 쓰므로 모델이 배울 필요가 없다)
    m_ = lgb.LGBMRegressor(**prm).fit(W.F.loc[rows_tr, W.L13], RES_TARGET[rows_tr])
    CURVE_RES[te_m] = np.clip(m_.predict(W.F.loc[rows_te, W.L13]).reshape(-1, 96) + RULE[te_m], 0, None)
HRLY = pd.read_csv(PREP / "hourly_review.csv", parse_dates=["origin"]).copy(); HRLY["date"] = HRLY.origin.dt.floor("D"); HRLY["hr"] = HRLY.origin.dt.hour
_PW = ["15분", "30분", "45분", "60분"]; _dp = HRLY.groupby("date")["생산량"].sum(); _off = _dp[_dp == 0].index                # 하루 생산이 0인 날(휴무일)
_byday = {d: g.sort_values("hr")[_PW].to_numpy() for d, g in HRLY.groupby("date") if len(g) == 24}
IDLE = np.full((W.N, 96), np.nan, np.float32); _cache = {}; dates_ = W.meta.origin.dt.floor("D").to_numpy(); hr0_ = W.meta.origin.dt.hour.to_numpy()
for i in range(W.N):
    D = pd.Timestamp(dates_[i])
    if D not in _cache:
        ds = [d for d in _off if d < D and d in _byday]                                    # 기준일 이전의 휴무일만 사용(미래 정보 없음)
        _cache[D] = np.median(np.stack([_byday[d] for d in ds]), axis=0) if len(ds) >= 3 else None
    if _cache[D] is not None: IDLE[i] = _cache[D][(hr0_[i] + np.repeat(np.arange(24), 4)) % 24, np.tile(np.arange(4), 24)]
CURVE = np.where(GATE & np.isfinite(IDLE), IDLE, np.where(GATE, RULE, CURVE_RES)).astype(np.float32)
log(f"곡선(3단 규칙 + 잔차 L1, 휴무일은 프로파일) 주간 재학습 {time.time() - t0:.0f}초 | 휴무일 칸 비율 7월 {GATE[W.ev_val].mean():.3f} / 8~9월 {GATE[W.ev_test].mean():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
THR_WATCH = window_thresholds(Q1_NEW, target="R95")
LAB = (W.y >= PEAK); PROB = np.full((W.N, 96), np.nan, np.float32)
for w in np.unique(WK_OF[W.EV]):
    S = WEEKS[0] + pd.Timedelta(days=7) * int(w); wk_ = WK_OF == w
    hist = np.isfinite(Q1_NEW[:, 0]) & (W.meta.target_end <= S).to_numpy() & (W.meta.origin >= WEEKS[0]).to_numpy() & (W.meta.origin >= S - pd.Timedelta(days=28)).to_numpy()
    lr = LogisticRegression(C=1e4, max_iter=200).fit((Q1_NEW[hist].reshape(-1, 1) - 176.0) / 20.0, LAB[hist].reshape(-1))
    PROB[wk_] = lr.predict_proba((Q1_NEW[wk_].reshape(-1, 1) - 176.0) / 20.0)[:, 1].reshape(-1, 96)
log("2단계 경보 기준(주의)와 피크 확률(Platt, 최근 4주) 계산 완료")

In [ ]:
def summary_metrics(y_, p_, t):
    """실제 피크 y>=176, 경보 p>=t(창별 경보 기준). F1·F2·정확도와 경보를 안 울릴 때의 정확도(함정)를 함께 낸다."""
    t = np.broadcast_to(np.asarray(t, float).reshape(-1, 1), y_.shape) if np.ndim(t) else t
    a = y_ >= 176.0; q = p_ >= t; tp = int((a & q).sum()); fp = int((~a & q).sum()); fn = int((a & ~q).sum()); tn = int((~a & ~q).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    f = lambda b: (1 + b * b) * pr * rc / (b * b * pr + rc) if pr + rc else 0.0
    e = p_ - y_
    return dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()), precision=pr, recall=rc, F1=f(1), F2=f(2),
                accuracy=(tp + tn) / y_.size, no_alarm_accuracy=float((~a).mean()), peak_rate=float(a.mean()), TP=tp, FP=fp, FN=fn, TN=tn, cells=int(y_.size))


POS = {"val": W.ev_val, "test": W.ev_test}; PRED = {}
for stg in ("val", "test"):
    ev = POS[stg]
    PRED[stg] = dict(baseline_w1=W.base[ev], gru_tuned=GRU[stg], q1_current=Q1_CUR[stg], q1_new=Q1_NEW[ev])
    PRED[stg]["ens_current"] = 0.5 * GRU[stg] + 0.5 * Q1_CUR[stg]
    PRED[stg]["ens_new_w05"] = 0.5 * GRU[stg] + 0.5 * Q1_NEW[ev]
    PRED[stg]["ens_new_w02"] = 0.2 * GRU[stg] + 0.8 * Q1_NEW[ev]
    PRED[stg]["rule3"] = RULE[ev]; PRED[stg]["curve_final"] = CURVE[ev]; PRED[stg]["prob_peak"] = PROB[ev]
SPECS = [("baseline_w1@176", "baseline_w1", False), ("q1_current@176", "q1_current", False), ("gru_tuned@176", "gru_tuned", False),
         ("ens_current@176", "ens_current", False), ("q1_new@176", "q1_new", False), ("q1_new@R85-최근4주", "q1_new", True),
         ("ens_new_w05@176", "ens_new_w05", False), ("ens_new_w02@176", "ens_new_w02", False)]
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    for name, key, adaptive in SPECS:
        t = THR_NEW[POS[stg]] if adaptive else 176.0
        rows.append(dict(구간=label, 모델=name, **summary_metrics(W.y[POS[stg]], PRED[stg][key], t)))
METRICS = pd.DataFrame(rows); METRICS.to_csv(OUTDIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")
show = ["MAE", "bias", "precision", "recall", "F1", "F2", "accuracy", "no_alarm_accuracy", "FP", "FN"]
display(METRICS[METRICS.구간 == "7월(검증)"].set_index("모델")[show].round(3).rename_axis("7월(검증)"))
display(METRICS[METRICS.구간 != "7월(검증)"].set_index("모델")[show].round(3).rename_axis("8~9월(개발 검증)"))

# ---- 곡선(수치 예측) 지표
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    yy = W.y[POS[stg]]
    for name, key in (("baseline_w1", "baseline_w1"), ("rule3(3단 규칙 값)", "rule3"), ("gru_tuned", "gru_tuned"), ("ens_current", "ens_current"), ("ens_new_w05", "ens_new_w05"), ("curve_final(규칙+잔차, 휴무일은 프로파일)", "curve_final")):
        e = PRED[stg][key] - yy; rows.append(dict(구간=label, 모델=name, MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean())))
CURVE_M = pd.DataFrame(rows); CURVE_M.to_csv(OUTDIR / "metrics_curve.csv", index=False, encoding="utf-8-sig")
display(CURVE_M.pivot(index="모델", columns="구간", values=["MAE", "RMSE", "bias"]).round(2))

# ---- 2단계 경보 지표
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    ev = POS[stg]; yy = W.y[ev]; pr = PRED[stg]["q1_new"]; ta = THR_NEW[ev][:, None]; tw = THR_WATCH[ev][:, None]
    a = yy >= 176; A = pr >= ta; Wt = (pr >= tw) & ~A; AW = A | Wt; prec = lambda m_: float((a & m_).sum() / max(m_.sum(), 1))
    rows.append(dict(구간=label, 경보_정밀도=prec(A), 경보_재현율=float((a & A).sum() / a.sum()), 경보_칸비율=float(A.mean() * 100), 주의만_칸비율=float(Wt.mean() * 100), 주의만_정밀도=prec(Wt),
                     경보와주의_재현율=float((a & AW).sum() / a.sum()), 경보와주의_정밀도=prec(AW), 경보가놓친피크_중_주의가잡은_비율=float((a & ~A & Wt).sum() / max((a & ~A).sum(), 1) * 100), 놓친피크_경보=int((a & ~A).sum()), 놓친피크_경보와주의=int((a & ~AW).sum())))
LEVELS = pd.DataFrame(rows); LEVELS.to_csv(OUTDIR / "metrics_alarm_levels.csv", index=False, encoding="utf-8-sig"); display(LEVELS.set_index("구간").T.round(3))

# ---- 확률 품질
def ap_score(a_, s_):
    o = np.argsort(-s_, kind="stable"); a_o = a_[o]; tp = np.cumsum(a_o); fp = np.cumsum(~a_o); prec_ = tp / (tp + fp); rec_ = tp / max(a_.sum(), 1)
    return float(np.sum(np.diff(np.concatenate([[0], rec_])) * prec_))
rows = []
for stg, label in (("val", "7월(검증)"), ("test", "8~9월(개발 검증)")):
    a = (W.y[POS[stg]] >= 176).reshape(-1); s_ = np.clip(PRED[stg]["prob_peak"].reshape(-1).astype(float), 1e-6, 1 - 1e-6); o = np.argsort(s_); ece = 0.0
    for ch in np.array_split(o, 10): ece += len(ch) / len(s_) * abs(s_[ch].mean() - a[ch].mean())
    rows.append(dict(구간=label, Brier=float(np.mean((s_ - a) ** 2)), LogLoss=float(-np.mean(a * np.log(s_) + (~a) * np.log(1 - s_))), ECE=float(ece), AP=ap_score(a, s_), 평균_예측확률=float(s_.mean()), 실제_피크비율=float(a.mean())))
PROBM = pd.DataFrame(rows); PROBM.to_csv(OUTDIR / "metrics_probability.csv", index=False, encoding="utf-8-sig"); display(PROBM.set_index("구간").T.round(4))

for stg, fname in (("test", "test_predictions_8_9월.csv"), ("val", "validation_predictions_7월.csv")):
    ev = POS[stg]; n = len(ev); origin = np.repeat(W.meta.origin.to_numpy()[ev], 96); step = np.tile(np.arange(96), n)
    ttime = pd.to_datetime(origin) + pd.to_timedelta(step // 4, "h") + pd.to_timedelta(15 * (step % 4 + 1), "m")
    thr = np.repeat(THR_NEW[ev], 96)
    df = pd.DataFrame(dict(기준시각=origin, 예측대상시각=ttime, 예측칸=step, 실제전력=W.y[ev].reshape(-1)))
    for k, v in PRED[stg].items(): df[k] = v.reshape(-1).round(3)
    df["최종_경보기준"] = 176.0
    df["최종_피크경보"] = (df["q1_new"] >= 176.0).astype(int)
    df["새Q1_경보기준(주별)"] = thr; df["실제피크(>=176)"] = (df["실제전력"] >= 176).astype(int)
    df["경보_새Q1(주별 기준)"] = (df["q1_new"] >= thr).astype(int); df["경보_현재Q1(176)"] = (df["q1_current"] >= 176).astype(int)
    thr_w = np.repeat(THR_WATCH[ev], 96); df["주의기준(주별)"] = thr_w
    df["경보단계(0정상/1주의/2경보)"] = np.where(df["q1_new"] >= thr, 2, np.where(df["q1_new"] >= thr_w, 1, 0))
    df.to_csv(OUTDIR / fname, index=False, encoding="utf-8-sig"); log(f"저장: outputs/{fname} ({len(df):,}행)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
d = METRICS[METRICS.구간 != "7월(검증)"].set_index("모델"); nm = [s[0] for s in SPECS]
ax[0].bar(range(len(nm)), d.loc[nm, "F2"], color="#0f766e"); ax[0].set_xticks(range(len(nm))); ax[0].set_xticklabels(nm, rotation=40, ha="right", fontsize=8); ax[0].set_ylim(0, 1); ax[0].set_title("8~9월(개발 검증) 피크 F2")
ax[1].bar(range(len(nm)), d.loc[nm, "MAE"], color="#d97706"); ax[1].set_xticks(range(len(nm))); ax[1].set_xticklabels(nm, rotation=40, ha="right", fontsize=8); ax[1].set_title("8~9월(개발 검증) MAE (낮을수록 좋음)")
ws = np.unique(WK_OF[W.EV]); xs, rs, th = [], [], []
for w in ws:
    k = WK_OF[W.EV] == w; yy = W.y[W.EV][k]; pp_ = Q1_NEW[W.EV][k]; tt = THR_NEW[W.EV][k]; a_ = yy >= 176
    xs.append(WEEKS[0] + pd.Timedelta(days=7) * int(w)); th.append(tt[0]); rs.append(((a_ & (pp_ >= tt[:, None])).sum() / a_.sum()) if a_.sum() >= 50 else np.nan)
ax[2].plot(xs, rs, marker="o", color="#0f766e", label="주별 실제 재현율"); ax[2].axhline(0.85, color="k", ls=":"); ax[2].set_ylabel("재현율"); a2 = ax[2].twinx(); a2.step(xs, th, where="post", color="#d97706", label="경보 기준")
a2.set_ylabel("경보 기준"); ax[2].set_title("새 Q1: 주별 경보 기준과 재현율"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); fig.savefig(OUTDIR / "figures/summary.png", dpi=130); plt.show()

In [ ]:
PREPROCESS_SRC = r'''
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Jupyter 없이 스크립트로 검증할 때 사용

ROOT = Path.cwd()
if not (ROOT / 'dataset/okm_augumented_2021.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'dataset/okm_augumented_2021.csv'
OUT = ROOT / 'results/preprocessing_24h'
POWER_COLS = ['15분', '30분', '45분', '60분']
MINUTES = [15, 30, 45, 60]
HORIZON = 24
VALID_START = pd.Timestamp('2021-07-01')
TEST_START = pd.Timestamp('2021-08-01')
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
# 기존 전처리에서 사용한 대한민국 2021년 공휴일 달력을 유지합니다.
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])
raw = pd.read_csv(DATA_PATH)
print('원본:', raw.shape)

data = raw.copy()
data['date'] = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
data['original_hour'] = data['시간']
row_hour = data.groupby('date', sort=False).cumcount()
valid_hour = data['시간'].between(0, 23) & data['시간'].eq(data['시간'].round())
assert data.groupby('date').size().eq(24).all(), '날짜별 행 수 확인 필요'
assert data.loc[valid_hour, '시간'].eq(row_hour[valid_hour]).all(), '원본 행 순서 확인 필요'
data['time_repaired'] = ~valid_hour
data.loc[~valid_hour, '시간'] = row_hour[~valid_hour]
data['origin'] = data['date'] + pd.to_timedelta(data['시간'], unit='h')
data = data.sort_values('origin').reset_index(drop=True)
assert data['origin'].is_unique
assert data['origin'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert np.isfinite(data[POWER_COLS + ['생산량']].to_numpy(dtype=float)).all()
assert data[POWER_COLS + ['생산량']].ge(0).all().all()
data['suspect_date'] = data['date'].isin(SUSPECT_DATES)
# 11시 행 전체를 포함: 행 시작시각 기준 [8/28 18시, 8/29 12시).
data['special_period'] = data['origin'].ge('2021-08-28 18:00') & data['origin'].lt('2021-08-29 12:00')
calendar = pd.DataFrame({
    'month': data['origin'].dt.month,
    'day': data['origin'].dt.day,
    'hour': data['origin'].dt.hour,
    'weekday': data['origin'].dt.dayofweek,
    'weekend': data['origin'].dt.dayofweek.ge(5).astype(int),
    'holiday': data['date'].isin(HOLIDAYS).astype(int),
})
display(data.loc[data['time_repaired'], ['date', 'original_hour', '시간']].head())
print('복구 행:', int(data['time_repaired'].sum()))
print('범위:', data['origin'].min(), '~', data['origin'].max())

# 시간축의 연속성을 검증했으므로 정수 위치로 정확한 시간 블록을 조회합니다.
starts = np.arange(336, len(data) - HORIZON + 1)
target_pos = starts[:, None] + np.arange(HORIZON)
w1_pos, w2_pos = target_pos - 168, target_pos - 336
power = data[POWER_COLS].to_numpy(dtype=float)
production = data['생산량'].to_numpy(dtype=float)
holiday = calendar['holiday'].to_numpy()
plan = production[target_pos]
sequence_parts = [plan[..., None]]
sequence_names = ['planned_production']
for name in calendar.columns:
    sequence_parts.append(calendar[name].to_numpy()[target_pos][..., None])
    sequence_names.append(name)
for week, pos in [(1, w1_pos), (2, w2_pos)]:
    sequence_parts.extend([
        power[pos], production[pos][..., None], holiday[pos][..., None],
        (plan - production[pos])[..., None],
        ((plan > 0) != (production[pos] > 0))[..., None],
        (holiday[target_pos] != holiday[pos])[..., None],
    ])
    sequence_names.extend([
        *[f'w{week}_power_{m}' for m in MINUTES],
        f'w{week}_production', f'w{week}_holiday',
        f'w{week}_production_difference', f'w{week}_production_state_mismatch',
        f'w{week}_holiday_mismatch',
    ])
sequence = np.concatenate(sequence_parts, axis=2).astype(float)
recent = power[starts - 1]
targets = power[target_pos]
recent_names = [f'recent_previous_hour_{m}' for m in MINUTES]
feature_names = recent_names + [f'h{h:02d}_{name}' for h in range(HORIZON) for name in sequence_names]
target_names = [f'h{h:02d}_power_{m}' for h in range(HORIZON) for m in MINUTES]
sample_ids = pd.Index(starts, name='sample_id')
X = pd.DataFrame(np.concatenate([recent, sequence.reshape(len(starts), -1)], axis=1), index=sample_ids, columns=feature_names)
y = pd.DataFrame(targets.reshape(len(starts), -1), index=sample_ids, columns=target_names)
print('X:', X.shape, '| y:', y.shape, '| sequence:', sequence.shape)

meta = pd.DataFrame(index=sample_ids)
meta['origin'] = data['origin'].iloc[starts].to_numpy()
meta['target_end'] = meta['origin'] + pd.Timedelta(hours=HORIZON)
meta['origin_hour'] = meta['origin'].dt.hour
suspect = data['suspect_date'].to_numpy()
special = data['special_period'].to_numpy()
meta['target_suspect'] = suspect[target_pos].any(axis=1)
meta['recent_suspect'] = suspect[starts - 1]
meta['w1_suspect'] = suspect[w1_pos].any(axis=1)
meta['w2_suspect'] = suspect[w2_pos].any(axis=1)
meta['primary_eligible'] = ~meta['target_suspect']
meta['history_clean_eligible'] = ~meta[['target_suspect', 'recent_suspect', 'w1_suspect', 'w2_suspect']].any(axis=1)
meta['special_period_overlap'] = special[target_pos].any(axis=1)
meta['split'] = 'boundary_excluded'
meta.loc[meta['target_end'].le(VALID_START), 'split'] = 'train'
meta.loc[meta['origin'].ge(VALID_START) & meta['target_end'].le(TEST_START), 'split'] = 'validation'
meta.loc[meta['origin'].ge(TEST_START), 'split'] = 'test'
meta['final_retrain_candidate'] = meta['target_end'].le(TEST_START)
split_masks = {name: meta['split'].eq(name).to_numpy() for name in ['train', 'validation', 'test']}
split_masks['final_retrain_candidates'] = meta['final_retrain_candidate'].to_numpy()

# 기준은 겹친 윈도우가 아니라 1~6월 원본의 각 관측값을 딱 한 번 사용합니다.
threshold_source = data.loc[data['origin'].lt(VALID_START), POWER_COLS].to_numpy().ravel()
peak_threshold = float(np.quantile(threshold_source, 0.95))
hour_masks = {
    'suspect_target': suspect[target_pos],
    'special_period': special[target_pos],
    'weekend_production': (calendar['weekend'].to_numpy()[target_pos] == 1) & (plan > 0),
    'holiday_production': (holiday[target_pos] == 1) & (plan > 0),
    'zero_production': plan == 0,
    'production_both_weeks_zero': (plan > 0) & (production[w1_pos] == 0) & (production[w2_pos] == 0),
}
target_masks = {name: np.repeat(values, 4, axis=1) for name, values in hour_masks.items()}
for name, lo, hi in [('horizon_0_3', 0, 3), ('horizon_3_12', 3, 12), ('horizon_12_24', 12, 24)]:
    hours = (np.arange(24) >= lo) & (np.arange(24) < hi)
    target_masks[name] = np.broadcast_to(np.repeat(hours, 4), y.shape).copy()
target_masks['actual_peak'] = y.to_numpy() >= peak_threshold
split_summary = meta.groupby('split').agg(samples=('origin', 'size'), first_origin=('origin', 'min'), last_origin=('origin', 'max'), primary_samples=('primary_eligible', 'sum'), history_clean_samples=('history_clean_eligible', 'sum'))
display(split_summary)
print('고정 피크 기준(q95):', peak_threshold)

assert X.index.equals(y.index) and y.index.equals(meta.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y.to_numpy()).all()
assert X.columns.is_unique and y.columns.is_unique
assert meta['origin'].iloc[0] == pd.Timestamp('2021-01-15')
assert meta.loc[meta['split'].eq('train'), 'origin'].max() == pd.Timestamp('2021-06-30')
for pos in [w1_pos, w2_pos]:
    latest_observed = data['origin'].to_numpy()[pos[:, -1]] + np.timedelta64(60, 'm')
    assert (latest_observed <= meta['origin'].to_numpy()).all()
assert (data['origin'].to_numpy()[starts - 1] + np.timedelta64(60, 'm') == meta['origin'].to_numpy()).all()
lookup = data.set_index('origin')
for i in [0, len(starts) // 2, len(starts) - 1]:
    origin = meta['origin'].iloc[i]
    times = pd.date_range(origin, periods=24, freq='h')
    np.testing.assert_array_equal(targets[i], lookup.loc[times, POWER_COLS].to_numpy())
    np.testing.assert_array_equal(recent[i], lookup.loc[origin - pd.Timedelta(hours=1), POWER_COLS].to_numpy(dtype=float))
    for week in [1, 2]:
        cols = [sequence_names.index(f'w{week}_power_{m}') for m in MINUTES]
        np.testing.assert_array_equal(sequence[i][:, cols], lookup.loc[times - pd.Timedelta(weeks=week), POWER_COLS].to_numpy())
for name, mask in split_masks.items():
    assert mask.any(), f'{name}: 빈 분할'
assert meta.loc[split_masks['train'], 'target_end'].le(VALID_START).all()
assert meta.loc[split_masks['validation'], 'origin'].ge(VALID_START).all()
assert meta.loc[split_masks['validation'], 'target_end'].le(TEST_START).all()
assert meta.loc[split_masks['test'], 'origin'].ge(TEST_START).all()
assert meta['target_end'].max() == data['origin'].max() + pd.Timedelta(hours=1)
assert len(threshold_source) == 181 * 24 * 4
np.testing.assert_array_equal(X.iloc[:, 4:].to_numpy().reshape(sequence.shape), sequence)
print('원본 대응·과거 정보 시각·정답 경계·배열 순서 검증 통과')

OUT.mkdir(parents=True, exist_ok=True)
for name, mask in split_masks.items():
    for prefix, frame in [('X', X), ('y', y), ('metadata', meta)]:
        frame.loc[mask].to_csv(OUT / f'{prefix}_{name}.csv', encoding='utf-8-sig', index=True)
    np.savez_compressed(
        OUT / f'arrays_{name}.npz',
        sample_id=sample_ids.to_numpy()[mask], sequence=sequence[mask], recent=recent[mask],
        y=targets[mask], baseline_w1=power[w1_pos][mask], baseline_w2=power[w2_pos][mask],
        baseline_weekly_mean=((power[w1_pos] + power[w2_pos]) / 2)[mask],
        **{key: value[mask] for key, value in target_masks.items()},
    )
meta.to_csv(OUT / 'all_sample_metadata.csv', encoding='utf-8-sig')
split_summary.to_csv(OUT / 'split_summary.csv', encoding='utf-8-sig')
data.to_csv(OUT / 'hourly_review.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'feature': feature_names}).to_csv(OUT / 'feature_catalog.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([
    {'target': f'h{h:02d}_power_{m}', 'hour_offset': h, 'minute_in_hour': m, 'observation_offset_minutes': h * 60 + m}
    for h in range(24) for m in MINUTES
]).to_csv(OUT / 'target_catalog.csv', index=False, encoding='utf-8-sig')
config = {
    'source_file': str(DATA_PATH.relative_to(ROOT)),
    'source_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'horizon_hours': 24, 'stride_hours': 1, 'history_hours': [168, 336],
    'feature_columns': feature_names, 'sequence_features': sequence_names,
    'recent_features': recent_names, 'target_columns': target_names,
    'flatten_order': 'hour first, then feature / power minute (15,30,45,60)',
    'peak_threshold': peak_threshold, 'peak_comparison': '>=',
    'threshold_source': 'unique raw observations Jan 1 through June 30; fixed after retraining',
    'threshold_observations': len(threshold_source),
    'validation_start': str(VALID_START), 'test_start': str(TEST_START),
    'holiday_dates': [str(d.date()) for d in HOLIDAYS],
    'suspect_dates': [str(d.date()) for d in SUSPECT_DATES],
    'primary_evaluation_mask': 'primary_eligible',
    'strict_evaluation_mask': 'history_clean_eligible',
    'special_period_row_interval': '[2021-08-28 18:00, 2021-08-29 12:00)',
    'operating_hour': None, 'final_retrain_suspect_policy': 'not yet selected; candidates only',
    'availability': '24h production plan known at origin; previous hour 60-minute power immediately available',
    'scaling': None, 'missing_value_policy': 'stop if required production or power is missing',
    'weather': 'excluded', 'random_seed_for_downstream_models': 42,
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
# 저장 후 다시 읽어 팀원이 불러올 때도 행 순서와 값이 유지되는지 확인합니다.
for name, mask in split_masks.items():
    saved_x = pd.read_csv(OUT / f'X_{name}.csv', index_col='sample_id')
    saved_y = pd.read_csv(OUT / f'y_{name}.csv', index_col='sample_id')
    assert saved_x.index.equals(saved_y.index)
    np.testing.assert_allclose(saved_x.to_numpy(), X.loc[mask].to_numpy())
    with np.load(OUT / f'arrays_{name}.npz') as saved:
        np.testing.assert_array_equal(saved['sample_id'], saved_x.index.to_numpy())
        np.testing.assert_allclose(saved['y'].reshape(len(saved_y), -1), saved_y.to_numpy())
print('저장 및 재로딩 검증 완료:', OUT)
'''

In [ ]:
checks = []
def add_check(name, ok, detail=""): checks.append(dict(검사=name, 결과="PASS" if ok else "FAIL", 설명=detail))

# 1) 기준 수치 대조
ref = pd.read_csv(ROOT / "reference/expected_metrics.csv"); GRU_DEP = {"gru_tuned@176", "ens_current@176", "ens_new_w05@176"}
for _, r in ref.iterrows():
    g = METRICS[(METRICS.구간 == r["구간"]) & (METRICS.모델 == r["모델"])].iloc[0]; dep = r["모델"] in GRU_DEP
    wide = dep and r["구간"].startswith("7월")                   # 7월 GRU는 원래 튜닝 노트북의 학습 경로로 만들어져 처음부터 학습한 결과와 조금 다름
    pr_ = .02 if wide else (.01 if dep else .002)
    tol = dict(precision=pr_, recall=pr_, F2=pr_, MAE=.3 if wide else (.2 if dep else .02), FP=300 if wide else (200 if dep else 20))
    diffs = {k: abs(float(g[k]) - float(r[k])) for k in tol}
    add_check(f"기준 수치 대조 | {r['구간']} | {r['모델']}", all(diffs[k] <= tol[k] for k in tol),
              f"F2 {float(g['F2']):.3f} (기준 {r['F2']:.3f}), 정밀도 {float(g['precision']):.3f} ({r['precision']:.3f}), 재현율 {float(g['recall']):.3f} ({r['recall']:.3f}), MAE {float(g['MAE']):.2f} ({r['MAE']:.2f})")

# 1-2) 곡선·2단계 경보·확률 기준 수치 대조
ex = pd.read_csv(ROOT / "reference/expected_extra.csv")


def got_value(item, per):
    cm = CURVE_M[CURVE_M.구간 == per]
    if item == "curve_final MAE": return float(cm[cm.모델.str.startswith("curve_final")].MAE.iloc[0])
    if item == "rule3 MAE": return float(cm[cm.모델.str.startswith("rule3")].MAE.iloc[0])
    if item == "gru MAE": return float(cm[cm.모델 == "gru_tuned"].MAE.iloc[0])
    L = LEVELS[LEVELS.구간 == per].iloc[0]; Pm = PROBM[PROBM.구간 == per].iloc[0]
    return float({"경보 정밀도": L.경보_정밀도, "경보 재현율": L.경보_재현율, "경보와주의 재현율": L.경보와주의_재현율, "놓친 피크(경보와주의)": L.놓친피크_경보와주의, "Brier": Pm.Brier, "ECE": Pm.ECE}[item])


for _, r in ex.iterrows():
    gv = got_value(r["항목"], r["구간"]); add_check(f"기준 수치 대조 | {r['구간']} | {r['항목']}", abs(gv - r["값"]) <= r["허용오차"], f"{gv:.4f} (기준 {r['값']:.4f}, 허용 ±{r['허용오차']})")

# 2) 구조 점검
add_check("창 개수", W.N == 5692 and len(W.ev_val) == 627 and len(W.ev_test) == 1057, f"전체 {W.N}, 평가 val {len(W.ev_val)} / test {len(W.ev_test)}")
add_check("학습 창 개수(val/test 단계)", int(W.tr_val.sum()) == 3985 and int(W.tr_test.sum()) == 4635, f"{int(W.tr_val.sum())} / {int(W.tr_test.sum())}")
add_check("val 학습 창이 7/1 이전에 끝남", W.meta.target_end[W.tr_val].max() <= pd.Timestamp("2021-07-01"), str(W.meta.target_end[W.tr_val].max()))
add_check("test 학습 창이 8/1 이전에 끝남", W.meta.target_end[W.tr_test].max() <= pd.Timestamp("2021-08-01"), str(W.meta.target_end[W.tr_test].max()))
add_check("새 Q1 예측이 평가 창에서 모두 유한값", bool(np.isfinite(Q1_NEW[W.EV]).all()), f"{int(np.isfinite(Q1_NEW[W.EV]).all(axis=1).sum())}/{len(W.EV)} 창")
add_check("경보 기준 이력 창이 모두 해당 주 시작 전에 끝남", len(HIST_LOG) > 0 and all(t_end <= S for S, t_end in HIST_LOG), f"{len(HIST_LOG)}개 평가 주의 이력 창 검사 (8~9월 결과는 기준 선택에 쓰이지 않음)")

# 3a) 미래 생산량 교란: 기준시각+24시간 이후 생산량을 바꿔도 상태 전환 피처가 같은가
rng = np.random.default_rng(0); pick = rng.choice(W.N, 60, replace=False); bad = 0
for w_ in pick:
    sl = slice(w_ * 96, (w_ + 1) * 96)
    base = state_features(W.hi, W.meta["origin"].iloc[[w_]], W.seq[[w_]], W.F.iloc[sl])
    P2 = W.hi.P.copy(); r0 = W.hi.row_of[pd.Timestamp(W.meta["origin"].iloc[w_])]; P2[r0 + 24:] = rng.uniform(0, 3000, len(P2) - r0 - 24)
    pert = state_features(W.hi, W.meta["origin"].iloc[[w_]], W.seq[[w_]], W.F.iloc[sl], P=P2)
    if not np.allclose(base.to_numpy(), pert.to_numpy(), equal_nan=True): bad += 1
add_check("미래 생산량 교란 검사(상태 전환 피처)", bad == 0, f"무작위 {len(pick)}개 창 중 피처가 바뀐 창 {bad}개")

# 3b) 미래 전력 교란: 8/15 이후 전력 원자료를 바꾸고 전처리부터 다시 실행
CUT = pd.Timestamp("2021-08-15")


def run_preprocessing_in(workdir):
    old = os.getcwd(); os.chdir(workdir)
    try:
        g = {"__name__": "__leakcheck__", "display": lambda *a, **k: None}
        with contextlib.redirect_stdout(io.StringIO()): exec(compile(PREPROCESS_SRC, "01_preprocessing(검사용 사본)", "exec"), g)
    finally: os.chdir(old)


tmp = Path(tempfile.mkdtemp(prefix="leakcheck_")); (tmp / "dataset").mkdir()
rawp = pd.read_csv(ROOT / "dataset/okm_augumented_2021.csv"); pcols = [c for c in rawp.columns if c.endswith("분")]; mask = rawp["날짜"] >= 20210815
rawp[pcols] = rawp[pcols].astype(float); rr = np.random.default_rng(1); rawp.loc[mask, pcols] = rr.uniform(20, 230, (int(mask.sum()), len(pcols))).round(0)
if "평균" in rawp.columns: rawp["평균"] = rawp["평균"].astype(float); rawp.loc[mask, "평균"] = rawp.loc[mask, pcols].mean(axis=1)
rawp.to_csv(tmp / "dataset/okm_augumented_2021.csv", index=False); run_preprocessing_in(tmp)
W2 = build_windows(tmp / "results/preprocessing_24h", strict=False)
o1 = {o: i for i, o in enumerate(W.meta.origin)}; o2 = {o: i for i, o in enumerate(W2.meta.origin)}
early = [o for o in W.meta.origin if o < CUT and o in o2]; late = [o for o in W.meta.origin if o >= CUT + pd.Timedelta(days=1) and o in o2]
same = lambda a, b, ids1, ids2: all(np.allclose(a[i1], b[i2], equal_nan=True) for i1, i2 in zip(ids1, ids2))
e1 = [o1[o] for o in early]; e2 = [o2[o] for o in early]
f_early = np.allclose(W.F.to_numpy().reshape(W.N, 96, -1)[e1], W2.F.to_numpy().reshape(W2.N, 96, -1)[e2], equal_nan=True)
f_seq = np.allclose(W.seq[e1], W2.seq[e2], equal_nan=True) and np.allclose(W.rec[e1], W2.rec[e2], equal_nan=True)
l1 = [o1[o] for o in late]; l2 = [o2[o] for o in late]
f_late_changed = not np.allclose(W.rec[l1], W2.rec[l2], equal_nan=True)
add_check("미래 전력 교란 검사(전처리부터 재실행): 입력 배열", f_seq, f"8/15 이전 기준시각 창 {len(early)}개의 sequence·recent 동일")
add_check("미래 전력 교란 검사: 모델 피처 전체(공식 30 + 3단 규칙 + 상태 전환)", f_early, f"8/15 이전 기준시각 창 {len(early)}개 × 96칸 × {W.F.shape[1]}피처 동일")
add_check("교란 검사가 실제로 작동함(8/15 이후 창은 달라짐)", f_late_changed, f"8/16 이후 창 {len(late)}개의 직전 전력 입력이 바뀜")
shutil.rmtree(tmp, ignore_errors=True)

CHK = pd.DataFrame(checks); CHK.to_csv(OUTDIR / "verification_checks.csv", index=False, encoding="utf-8-sig")
display(CHK.assign(설명=CHK.설명.str.slice(0, 120)))
npass = int((CHK.결과 == "PASS").sum()); log(f"자체 검증 {npass}/{len(CHK)} PASS | 전체 실행 시간 {(time.time() - T_START) / 60:.1f}분")
if npass != len(CHK): print("※ FAIL 항목이 있습니다. 위 표의 설명을 확인하세요.")

## 2.1 최종 제출용 예측 파일

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = PROJECT_ROOT
expected = ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv'
if not expected.exists():
    expected = ROOT / 'results/model_outputs/test_predictions_8_9월.csv'
df = pd.read_csv(expected).sort_values(['기준시각', '예측칸']).reset_index(drop=True)
if not df.groupby('기준시각')['예측칸'].apply(lambda s: np.array_equal(s.to_numpy(), np.arange(96))).all():
    raise ValueError('각 예측창은 step 0~95를 한 번씩 포함해야 합니다.')
df['최종_경보기준'] = 176.0
df['최종_피크경보'] = (df['q1_new'] >= 176.0).astype(int)
OUT = ROOT / 'results/model_outputs'
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / 'test_predictions_8_9월.csv', index=False, encoding='utf-8-sig')
submission = ROOT / 'submission'
submission.mkdir(parents=True, exist_ok=True)
export = df[['기준시각', '예측대상시각', '예측칸', 'curve_final', 'q1_new',
             '최종_경보기준', '최종_피크경보']].copy()
export.to_csv(submission / 'final_predictions_fixed176.csv', index=False, encoding='utf-8-sig')
(OUT / 'selected_alarm_policy.json').write_text(json.dumps({
    'point_model': 'curve_final', 'peak_prediction': 'q1_new',
    'alarm_policy': 'fixed176', 'alarm_threshold': 176.0,
    'weekly_alarm_role': 'comparison_only'
}, ensure_ascii=False, indent=2), encoding='utf-8')
display(export.head())
print('예측창:', export['기준시각'].nunique(), '| 15분 예측:', len(export))
print('제출용 출력에는 실제 전력·주별 경보·확률을 포함하지 않습니다.')


# 3. GRU 실험과 동일 조건 비교

In [ ]:
from pathlib import Path
import copy
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ROOT = PROJECT_ROOT
if not (ROOT / 'results/preprocessing_24h/config.json').exists():
    ROOT = PROJECT_ROOT
PREP = ROOT / 'results/preprocessing_24h'
OUT = ROOT / 'results/gru_24h_team_preprocessing'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
device = torch.device('cpu')
torch.set_num_threads(min(8, os.cpu_count() or 1))
BASE_PARAMS = {
    'hidden_size': 32, 'num_layers': 2, 'dropout': 0.15,
    'bidirectional': True, 'batch_size': 64, 'learning_rate': 1e-3,
    'weight_decay': 1e-4, 'max_epochs': 100, 'patience': 10,
    'peak_weight': 1.0,
}
print('PyTorch:', torch.__version__, '| device:', device)
print('저장 위치:', OUT.resolve())


In [ ]:
prep_config = json.loads((PREP / 'config.json').read_text(encoding='utf-8'))
SEQUENCE_NAMES = prep_config['sequence_features']
RECENT_NAMES = prep_config['recent_features']
PEAK_THRESHOLD = float(prep_config['peak_threshold'])

def load_arrays(split):
    with np.load(PREP / f'arrays_{split}.npz') as saved:
        return {name: saved[name].copy() for name in
                ['sequence', 'recent', 'y', 'baseline_w1', 'sample_id']}

train, validation = load_arrays('train'), load_arrays('validation')
meta_validation = pd.read_csv(
    PREP / 'metadata_validation.csv', index_col='sample_id',
    parse_dates=['origin', 'target_end'],
)
np.testing.assert_array_equal(validation['sample_id'], meta_validation.index.to_numpy())
primary_rows = meta_validation['primary_eligible'].astype(bool).to_numpy()
strict_rows = meta_validation['history_clean_eligible'].astype(bool).to_numpy()
assert primary_rows.any() and np.all(~strict_rows | primary_rows)

for arrays in [train, validation]:
    n = len(arrays['sample_id'])
    assert arrays['sequence'].shape == (n, 24, len(SEQUENCE_NAMES))
    assert arrays['recent'].shape == (n, len(RECENT_NAMES))
    assert arrays['y'].shape == arrays['baseline_w1'].shape == (n, 24, 4)
    for name in ['sequence', 'recent', 'y', 'baseline_w1']:
        assert np.isfinite(arrays[name]).all(), name

feature_sets = {'team_common_25': list(SEQUENCE_NAMES)}
assert len(SEQUENCE_NAMES) == 25
print('학습:', train['y'].shape, '| 검증:', validation['y'].shape)
print('primary:', primary_rows.sum(), '| history_clean:', strict_rows.sum(),
      '| 피크 기준:', PEAK_THRESHOLD)
display(pd.DataFrame([
    {'feature_set': name, 'sequence_features': len(names),
     'GRU_input_channels': len(names) + len(RECENT_NAMES)}
    for name, names in feature_sets.items()
]))


In [ ]:
def make_input(arrays, names):
    indices = [SEQUENCE_NAMES.index(name) for name in names]
    recent = np.repeat(arrays['recent'][:, None, :], 24, axis=1)
    return np.concatenate(
        [arrays['sequence'][:, :, indices], recent], axis=2
    ).astype(np.float32)

class GRURegressor(nn.Module):
    def __init__(self, input_features, hidden_size=32, num_layers=2,
                 dropout=0.15, bidirectional=True, scalar_output=False):
        super().__init__()
        self.scalar_output = scalar_output
        self.gru = nn.GRU(
            input_features, hidden_size, num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        width = hidden_size * (2 if bidirectional else 1)
        self.output = nn.Linear(width, 1 if scalar_output else 4)

    def forward(self, x):
        sequence, hidden = self.gru(x)
        if self.scalar_output:
            summary = (torch.cat([hidden[-2], hidden[-1]], dim=1)
                       if self.gru.bidirectional else hidden[-1])
            return self.output(summary).squeeze(-1)
        return self.output(sequence)

def element_loss(prediction, target, objective, alpha):
    error = target - prediction
    if objective == 'mse':
        return error.square()
    if objective == 'quantile':
        return torch.maximum(alpha * error, (alpha - 1) * error)
    raise ValueError(objective)

RUN_CACHE = {}  # 같은 입력·설정의 중복 학습 방지


In [ ]:
def fit_gru(names, role, params=None, objective='mse', alpha=0.9,
            train_mask=None, valid_mask=None, tag='unified'):
    params = {**BASE_PARAMS, **(params or {})}
    cache_key = (
        tuple(names), role, json.dumps(params, sort_keys=True),
        objective, float(alpha), tag,
    )
    if cache_key in RUN_CACHE:
        return RUN_CACHE[cache_key]

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    started = time.perf_counter()
    xtr_raw, xva_raw = make_input(train, names), make_input(validation, names)
    x_mean = xtr_raw.mean(axis=(0, 1), keepdims=True)
    x_std = np.maximum(xtr_raw.std(axis=(0, 1), keepdims=True), 1e-6)
    xtr, xva = (xtr_raw - x_mean) / x_std, (xva_raw - x_mean) / x_std

    target_tr = train['y'] - train['baseline_w1']
    target_va = validation['y'] - validation['baseline_w1']
    tmask = (np.ones_like(target_tr, dtype=bool)
             if train_mask is None else np.asarray(train_mask, dtype=bool))
    vmask = (np.ones_like(target_va, dtype=bool)
             if valid_mask is None else np.asarray(valid_mask, dtype=bool))
    vmask = vmask & primary_rows[:, None, None]
    assert tmask.shape == target_tr.shape and vmask.shape == target_va.shape
    assert tmask.any() and vmask.any()

    target_mean = float(target_tr[tmask].mean())
    target_std = max(float(target_tr[tmask].std()), 1e-6)
    scaled_tr = ((target_tr - target_mean) / target_std).astype(np.float32)
    scaled_va = ((target_va - target_mean) / target_std).astype(np.float32)
    train_weights = tmask.astype(np.float32)
    if objective == 'mse':
        train_weights *= np.where(
            train['y'] >= PEAK_THRESHOLD, params['peak_weight'], 1.0
        ).astype(np.float32)

    active_rows = tmask.reshape(len(tmask), -1).any(axis=1)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(xtr[active_rows]),
            torch.from_numpy(scaled_tr[active_rows]),
            torch.from_numpy(train_weights[active_rows]),
        ),
        batch_size=params['batch_size'], shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    architecture = {
        'input_features': xtr.shape[-1],
        **{key: params[key] for key in
           ['hidden_size', 'num_layers', 'dropout', 'bidirectional']},
        'scalar_output': False,
    }
    model = GRURegressor(**architecture).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params['learning_rate'],
        weight_decay=params['weight_decay'],
    )
    valid_x = torch.from_numpy(xva).to(device)
    valid_target = torch.from_numpy(scaled_va).to(device)
    valid_mask_tensor = torch.from_numpy(vmask.astype(np.float32)).to(device)

    best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
    history = []
    print(f'학습 시작: {role} / {tag} / {xtr.shape[-1]}채널 / {objective}')
    for epoch in range(1, params['max_epochs'] + 1):
        model.train()
        train_sum, weight_sum = 0.0, 0.0
        for batch_x, batch_target, batch_weight in loader:
            batch_x, batch_target, batch_weight = [
                value.to(device) for value in (batch_x, batch_target, batch_weight)
            ]
            optimizer.zero_grad()
            raw = element_loss(model(batch_x), batch_target, objective, alpha)
            loss = (raw * batch_weight).sum() / batch_weight.sum()
            if not torch.isfinite(loss):
                raise RuntimeError('학습 손실이 유한하지 않습니다.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_sum += float(loss.detach()) * float(batch_weight.sum())
            weight_sum += float(batch_weight.sum())

        model.eval()
        with torch.no_grad():
            valid_raw = element_loss(model(valid_x), valid_target, objective, alpha)
            valid_loss = float(
                (valid_raw * valid_mask_tensor).sum() / valid_mask_tensor.sum()
            )
        history.append({
            'epoch': epoch, 'train_loss': train_sum / weight_sum,
            'valid_loss': valid_loss,
        })
        if valid_loss < best_loss - 1e-6:
            best_loss, best_epoch, stale = valid_loss, epoch, 0
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= params['patience']:
                break

    model.load_state_dict(best_state)
    model.eval()
    parts = []
    with torch.no_grad():
        for start in range(0, len(xva), params['batch_size']):
            parts.append(model(
                torch.from_numpy(xva[start:start + params['batch_size']]).to(device)
            ).cpu().numpy())
    predicted_residual = np.concatenate(parts) * target_std + target_mean
    prediction = np.maximum(
        validation['baseline_w1'] + predicted_residual, 0
    ).astype(np.float32)
    run = {
        'names': list(names), 'params': params, 'objective': objective,
        'alpha': float(alpha), 'architecture': architecture,
        'state_dict': best_state, 'x_mean': x_mean, 'x_std': x_std,
        'target_mean': target_mean, 'target_std': target_std,
        'prediction': prediction, 'history': pd.DataFrame(history),
        'best_epoch': best_epoch, 'best_valid_loss': best_loss,
        'train_positions': int(tmask.sum()),
        'validation_positions': int(vmask.sum()),
        'seconds': time.perf_counter() - started,
    }
    RUN_CACHE[cache_key] = run
    print(f'완료: best epoch {best_epoch} / {run["seconds"]:.1f}초')
    return run


In [ ]:
def evaluate(actual, prediction, mask=None):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    if mask is not None:
        mask = np.broadcast_to(mask, actual.shape)
        actual, prediction = actual[mask], prediction[mask]
    actual, prediction = actual.reshape(-1), prediction.reshape(-1)
    error = prediction - actual
    actual_peak, predicted_peak = (
        actual >= PEAK_THRESHOLD, prediction >= PEAK_THRESHOLD
    )
    tp = int((actual_peak & predicted_peak).sum())
    fp = int((~actual_peak & predicted_peak).sum())
    fn = int((actual_peak & ~predicted_peak).sum())
    tn = int((~actual_peak & ~predicted_peak).sum())
    return {
        'observations': len(actual),
        'mae': float(np.abs(error).mean()) if len(error) else np.nan,
        'rmse': float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan,
        'mean_error': float(error.mean()) if len(error) else np.nan,
        'true_positive': tp, 'false_positive': fp,
        'false_negative': fn, 'true_negative': tn,
        'peak_precision': tp / (tp + fp) if tp + fp else np.nan,
        'peak_recall': tp / (tp + fn) if tp + fn else np.nan,
        'peak_f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
        'peak_f2': 5 * tp / (5 * tp + fp + 4 * fn) if 5 * tp + fp + 4 * fn else np.nan,
    }

def maximum_metrics(actual, prediction):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    error = prediction - actual
    return {
        'windows': len(actual),
        'maximum_mae': float(np.abs(error).mean()) if len(error) else np.nan,
        'maximum_rmse': float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan,
        'maximum_bias': float(error.mean()) if len(error) else np.nan,
        'underprediction_rate': float((error < 0).mean()) if len(error) else np.nan,
    }

def window_metrics(actual, prediction):
    actual, prediction = actual.reshape(-1, 96), prediction.reshape(-1, 96)
    actual_max, predicted_max = actual.max(axis=1), prediction.max(axis=1)
    distance = (
        np.abs(np.arange(96)[None, :] - prediction.argmax(axis=1)[:, None]) * 15
    )
    timing = np.where(
        actual == actual_max[:, None], distance, np.inf
    ).min(axis=1)
    return {
        **maximum_metrics(actual_max, predicted_max),
        'maximum_time_mae_minutes': float(timing.mean()),
        'maximum_time_within_60m': float((timing <= 60).mean()),
    }

def result_table(runs):
    return pd.DataFrame([
        {
            'role': role, 'candidate': candidate,
            'channels': len(run['names']) + len(RECENT_NAMES),
            'best_epoch': run['best_epoch'],
            **evaluate(
                validation['y'][primary_rows],
                run['prediction'][primary_rows],
            ),
        }
        for (role, candidate), run in runs.items()
    ])

def show_roles(table, id_columns):
    for role, metrics in [
        ('point', ['mae', 'rmse']),
        ('peak_f1', [
            'peak_precision', 'peak_recall', 'peak_f1',
            'false_positive', 'false_negative',
        ]),
    ]:
        print('\n수치예측 (MAE 기준)' if role == 'point' else '\n피크경보 (F1 기준)')
        display(
            table.loc[table['role'].eq(role), id_columns + metrics].round(4)
        )

def select_id(table, role, column='candidate'):
    rows = table.loc[table['role'].eq(role)]
    if role == 'point':
        return rows.sort_values('mae', kind='stable').iloc[0][column]
    return rows.sort_values(
        ['peak_f1', 'peak_precision', 'mae'],
        ascending=[False, False, True], kind='stable',
    ).iloc[0][column]

def plot_confusions(predictions, path):
    fig, axes = plt.subplots(
        1, len(predictions), figsize=(6 * len(predictions), 4.8), squeeze=False
    )
    for ax, (label, prediction) in zip(axes.ravel(), predictions.items()):
        score = evaluate(
            validation['y'][primary_rows], prediction[primary_rows]
        )
        matrix = np.array([
            [score['true_negative'], score['false_positive']],
            [score['false_negative'], score['true_positive']],
        ])
        ax.imshow(matrix, cmap='Blues')
        for (row, col), value in np.ndenumerate(matrix):
            ax.text(
                col, row, f'{value:,}\n({value / matrix.sum():.2%})',
                ha='center', va='center',
                color='white' if value > matrix.max() / 2 else 'black',
            )
        ax.set(
            xticks=[0, 1], yticks=[0, 1],
            xticklabels=['예측 비피크', '예측 피크'],
            yticklabels=['실제 비피크', '실제 피크'],
            xlabel='예측', ylabel='실제',
            title=(
                f'{label}\nPrecision {score["peak_precision"]:.3f} / '
                f'Recall {score["peak_recall"]:.3f} / F1 {score["peak_f1"]:.3f}'
            ),
        )
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()


In [ ]:
basic_runs = {
    ('point', 'L2'): fit_gru(
        feature_sets['team_common_25'], 'point', objective='mse'
    ),
    ('peak_f1', 'quantile_0.90'): fit_gru(
        feature_sets['team_common_25'], 'peak_f1',
        objective='quantile', alpha=0.90,
    ),
}
basic_results = result_table(basic_runs)
show_roles(basic_results, ['candidate', 'channels', 'best_epoch'])
display(pd.DataFrame([{
    'model': 'previous_week_baseline',
    **evaluate(
        validation['y'][primary_rows],
        validation['baseline_w1'][primary_rows],
    ),
}]).round(4))
basic_results.to_csv(
    OUT / 'basic_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)


In [ ]:
basic_selected = {
    role: basic_runs[(role, select_id(basic_results, role))]
    for role in ['point', 'peak_f1']
}
for role, run in basic_selected.items():
    print(
        role, '| objective:', run['objective'],
        '| alpha:', run['alpha'], '| best epoch:', run['best_epoch']
    )


In [ ]:
feature_runs = {}
for role, base in basic_selected.items():
    for candidate, names in feature_sets.items():
        feature_runs[(role, candidate)] = fit_gru(
            names, role, base['params'], base['objective'], base['alpha']
        )
feature_results = result_table(feature_runs)
show_roles(feature_results, ['candidate', 'channels', 'best_epoch'])
feature_results.to_csv(
    OUT / 'feature_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)
feature_selected = {
    role: feature_runs[(role, select_id(feature_results, role))]
    for role in basic_selected
}

def search_stage(stage, current, candidates):
    runs = {}
    for role, base in current.items():
        for candidate, changes in candidates.items():
            runs[(role, candidate)] = fit_gru(
                base['names'], role,
                {**base['params'], **changes},
                base['objective'], base['alpha'],
            )
    table = result_table(runs)
    show_roles(table, ['candidate', 'channels', 'best_epoch'])
    table.to_csv(
        OUT / f'{stage}_validation.csv',
        index=False, encoding='utf-8-sig',
    )
    selected = {
        role: runs[(role, select_id(table, role))]
        for role in current
    }
    return selected, table

structure_selected, structure_results = search_stage(
    'structure', feature_selected, {
        'current': {},
        'hidden64': {'hidden_size': 64},
        'one_layer': {'num_layers': 1, 'dropout': 0.0},
    }
)
training_selected, training_results = search_stage(
    'training', structure_selected, {
        'current': {},
        'lr0005': {'learning_rate': 5e-4},
        'batch128': {'batch_size': 128},
    }
)


In [ ]:
loss_runs = {}
point_base = training_selected['point']
for weight in [1.0, 2.0, 5.0]:
    loss_runs[('point', f'weight_{weight:g}')] = fit_gru(
        point_base['names'], 'point',
        {**point_base['params'], 'peak_weight': weight},
        objective='mse',
    )
risk_base = training_selected['peak_f1']
for alpha in [0.85, 0.90, 0.95]:
    loss_runs[('peak_f1', f'quantile_{alpha:.2f}')] = fit_gru(
        risk_base['names'], 'peak_f1', risk_base['params'],
        objective='quantile', alpha=alpha,
    )
loss_results = result_table(loss_runs)
show_roles(loss_results, ['candidate', 'channels', 'best_epoch'])
loss_results.to_csv(
    OUT / 'loss_settings_validation.csv',
    index=False, encoding='utf-8-sig',
)
tuned_selected = {
    role: loss_runs[(role, select_id(loss_results, role))]
    for role in basic_selected
}


In [ ]:
plot_confusions({
    '수치예측 최적 조합 (MAE 선택)': tuned_selected['point']['prediction'],
    '피크경보 최적 조합 (F1 선택)': tuned_selected['peak_f1']['prediction'],
}, OUT / 'tuned_confusions_validation.png')


In [ ]:
def sequence_feature(arrays, name):
    return arrays['sequence'][:, :, SEQUENCE_NAMES.index(name)]

positive_plans = sequence_feature(train, 'planned_production')
positive_plans = positive_plans[positive_plans > 0]
production_split = float(np.median(positive_plans))

def make_regime_masks(arrays):
    planned = sequence_feature(arrays, 'planned_production')
    day_off = (
        (sequence_feature(arrays, 'weekend') > 0) |
        (sequence_feature(arrays, 'holiday') > 0)
    )
    hour_masks = {
        'unified': {'all': np.ones_like(planned, dtype=bool)},
        'production_binary': {
            'production_off': planned <= 0,
            'production_on': planned > 0,
        },
        'production_level': {
            'production_off': planned <= 0,
            'production_low': (
                (planned > 0) & (planned <= production_split)
            ),
            'production_high': planned > production_split,
        },
        'calendar': {
            'weekday': ~day_off,
            'weekend_or_holiday': day_off,
        },
    }
    result = {
        strategy: {
            group: np.repeat(mask[:, :, None], 4, axis=2)
            for group, mask in groups.items()
        }
        for strategy, groups in hour_masks.items()
    }
    for groups in result.values():
        assert np.all(np.stack(list(groups.values())).sum(axis=0) == 1)
    return result

train_regimes = make_regime_masks(train)
validation_regimes = make_regime_masks(validation)

def fit_routed(role, reference, strategy, tag_prefix='regime'):
    if strategy == 'unified':
        return {
            'prediction': reference['prediction'],
            'models': {'all': reference},
            'fallbacks': {},
            'strategy': strategy,
        }
    prediction = np.full_like(validation['y'], np.nan, dtype=np.float32)
    models, fallbacks = {}, {}
    for group, train_mask in train_regimes[strategy].items():
        valid_mask = validation_regimes[strategy][group]
        eligible_valid = valid_mask & primary_rows[:, None, None]
        if not train_mask.any() or not eligible_valid.any():
            run = reference
            fallbacks[group] = '통합 모델 사용: 학습 또는 검증 위치 없음'
        else:
            run = fit_gru(
                reference['names'], role, reference['params'],
                reference['objective'], reference['alpha'],
                train_mask=train_mask, valid_mask=valid_mask,
                tag=f'{tag_prefix}/{strategy}/{group}',
            )
        models[group] = run
        prediction[valid_mask] = run['prediction'][valid_mask]
    assert np.isfinite(prediction).all()
    return {
        'prediction': prediction, 'models': models,
        'fallbacks': fallbacks, 'strategy': strategy,
    }

regime_runs, regime_rows, regime_group_rows = {}, [], []
for role, reference in tuned_selected.items():
    for strategy in train_regimes:
        routed = fit_routed(role, reference, strategy)
        regime_runs[(role, strategy)] = routed
        regime_rows.append({
            'role': role, 'strategy': strategy,
            **evaluate(
                validation['y'][primary_rows],
                routed['prediction'][primary_rows],
            ),
        })
        for group, mask in validation_regimes[strategy].items():
            regime_group_rows.append({
                'role': role, 'strategy': strategy, 'group': group,
                'fallback': routed['fallbacks'].get(group, ''),
                **evaluate(
                    validation['y'], routed['prediction'],
                    mask & primary_rows[:, None, None],
                ),
            })

regime_results = pd.DataFrame(regime_rows)
regime_group_results = pd.DataFrame(regime_group_rows)
selected_strategy = {
    role: select_id(regime_results, role, 'strategy')
    for role in tuned_selected
}
selected_routed = {
    role: regime_runs[(role, selected_strategy[role])]
    for role in tuned_selected
}
selected_predictions = {
    role: routed['prediction']
    for role, routed in selected_routed.items()
}
show_roles(regime_results, ['strategy'])
print('저/고생산 경계:', production_split)
print('역할별 선택:', selected_strategy)
display(regime_group_results.round(4))
regime_results.to_csv(
    OUT / 'regime_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)
regime_group_results.to_csv(
    OUT / 'regime_groups_validation.csv',
    index=False, encoding='utf-8-sig',
)


In [ ]:
plan_on = sequence_feature(validation, 'planned_production') > 0
w1_on = sequence_feature(validation, 'w1_production') > 0
condition_hours = {
    'all': np.ones_like(plan_on, dtype=bool),
    'both_off': ~plan_on & ~w1_on,
    'current_only_production': plan_on & ~w1_on,
    'previous_week_only_production': ~plan_on & w1_on,
    'both_on': plan_on & w1_on,
}
conditions = {
    name: np.repeat(mask[:, :, None], 4, axis=2)
    for name, mask in condition_hours.items()
}
evaluation_rows = {'primary': primary_rows, 'history_clean': strict_rows}
diagnostic_predictions = {
    'baseline': validation['baseline_w1'],
    **selected_predictions,
}
condition_records, maximum_records = [], []
for subset, rows in evaluation_rows.items():
    for role, prediction in diagnostic_predictions.items():
        maximum_records.append({
            'subset': subset, 'role': role,
            **window_metrics(
                validation['y'][rows], prediction[rows]
            ),
        })
        for condition, mask in conditions.items():
            condition_records.append({
                'subset': subset, 'role': role,
                'condition': condition,
                **evaluate(
                    validation['y'], prediction,
                    mask & rows[:, None, None],
                ),
            })
condition_results = pd.DataFrame(condition_records)
maximum_results = pd.DataFrame(maximum_records)
for subset in evaluation_rows:
    print('\n', subset)
    show_roles(
        condition_results.loc[
            condition_results['subset'].eq(subset)
        ],
        ['condition', 'observations'],
    )
display(maximum_results.round(4))
condition_results.to_csv(
    OUT / 'conditions_validation.csv',
    index=False, encoding='utf-8-sig',
)
maximum_results.to_csv(
    OUT / 'maximum_validation.csv',
    index=False, encoding='utf-8-sig',
)


In [ ]:
stage_records = []
for stage, runs in [
    ('basic', basic_selected),
    ('features', feature_selected),
    ('structure', structure_selected),
    ('training', training_selected),
    ('loss_settings', tuned_selected),
]:
    for role, run in runs.items():
        stage_records.append({
            'stage': stage, 'role': role,
            **evaluate(
                validation['y'][primary_rows],
                run['prediction'][primary_rows],
            ),
        })
for role, prediction in selected_predictions.items():
    stage_records.append({
        'stage': 'regime', 'role': role,
        **evaluate(
            validation['y'][primary_rows],
            prediction[primary_rows],
        ),
    })
progress_results = pd.DataFrame(stage_records)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, (role, metric, title) in enumerate([
    ('point', 'mae', '수치예측 MAE'),
    ('peak_f1', 'peak_f1', '피크경보 F1'),
]):
    table = progress_results.loc[progress_results['role'].eq(role)]
    axes[0, col].plot(table['stage'], table[metric], marker='o')
    axes[0, col].set_title(title + ' / 단계별')
    axes[0, col].tick_params(axis='x', rotation=30)
    table = regime_results.loc[regime_results['role'].eq(role)]
    axes[1, col].bar(table['strategy'], table[metric])
    axes[1, col].set_title(title + ' / 운영 조건 분리')
    axes[1, col].tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(
    OUT / 'progress_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

plot_confusions({
    '최종 수치예측 (MAE 선택)': selected_predictions['point'],
    '최종 피크경보 (F1 선택)': selected_predictions['peak_f1'],
}, OUT / 'selected_confusions_validation.png')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, role, metric, title in [
    (axes[0, 0], 'point', 'mae', '수치예측 조건별 MAE'),
    (axes[0, 1], 'peak_f1', 'peak_f1', '피크경보 조건별 F1'),
]:
    table = condition_results.loc[
        condition_results['subset'].eq('primary') &
        condition_results['role'].eq(role)
    ]
    ax.bar(table['condition'], table[metric])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)

maximum_plot = maximum_results.loc[
    maximum_results['subset'].eq('primary') &
    maximum_results['role'].isin(['baseline', 'point'])
]
for ax, metric, title in [
    (axes[1, 0], 'maximum_mae', '24시간 최대전력 MAE'),
    (axes[1, 1], 'maximum_bias', '최대전력 편향: 음수는 과소예측'),
]:
    ax.bar(maximum_plot['role'], maximum_plot[metric])
    ax.set_title(title)
    ax.axhline(0, color='black', linewidth=0.8)
fig.tight_layout()
fig.savefig(
    OUT / 'conditions_and_maximum_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

eligible = np.flatnonzero(primary_rows)
example_index = eligible[
    np.argmax(validation['y'][primary_rows].max(axis=(1, 2)))
]
actual = validation['y'][example_index].reshape(-1)
point = selected_predictions['point'][example_index].reshape(-1)
risk = selected_predictions['peak_f1'][example_index].reshape(-1)
hours = np.arange(1, 97) / 4
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(hours, actual, label='실제')
axes[0].plot(
    hours, validation['baseline_w1'][example_index].reshape(-1),
    label='전주 기준', alpha=0.6,
)
axes[0].plot(hours, point, label='수치예측 (MAE)')
axes[0].axhline(
    PEAK_THRESHOLD, color='red', linestyle='--', label='피크 기준'
)
axes[0].set_title(
    f'primary 최고 최대전력 사례 / origin '
    f'{meta_validation.iloc[example_index]["origin"]}'
)
axes[0].legend()
axes[1].plot(hours, actual, label='실제')
axes[1].plot(hours, risk, label='피크경보 분위수 예측')
axes[1].fill_between(
    hours, 0, 1, where=risk >= PEAK_THRESHOLD,
    transform=axes[1].get_xaxis_transform(),
    alpha=0.15, color='red', label='경보',
)
axes[1].axhline(PEAK_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('예측 구간 내 시간')
axes[1].legend()
fig.tight_layout()
fig.savefig(
    OUT / 'selected_example_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()


In [ ]:
def save_routed(role, routed, directory):
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {
        'role': role,
        'strategy': routed['strategy'],
        'selection_metric': 'MAE' if role == 'point' else 'F1',
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
        'seed': SEED,
        'peak_threshold': PEAK_THRESHOLD,
        'production_split': production_split,
        'source_sha256': prep_config['source_sha256'],
        'recent_features': RECENT_NAMES,
        'routing': (
            '각 예측 대상 시간의 생산계획·주말·공휴일로 분기; '
            '전문 모델은 24시간 전체 입력을 읽고 담당 위치의 손실만 학습'
        ),
        'fallbacks': routed['fallbacks'],
        'groups': {},
    }
    for group, run in routed['models'].items():
        stem = f'{role}_{group}'
        weight_path = directory / f'{stem}.pt'
        scaler_path = directory / f'{stem}_scalers.npz'
        torch.save(run['state_dict'], weight_path)
        np.savez_compressed(
            scaler_path,
            x_mean=run['x_mean'], x_std=run['x_std'],
            target_mean=run['target_mean'], target_std=run['target_std'],
        )
        run['history'].to_csv(
            directory / f'{stem}_history.csv',
            index=False, encoding='utf-8-sig',
        )
        manifest['groups'][group] = {
            'sequence_features': run['names'],
            'params': run['params'],
            'architecture': run['architecture'],
            'objective': run['objective'],
            'alpha': run['alpha'],
            'best_epoch': run['best_epoch'],
            'best_valid_loss': run['best_valid_loss'],
            'train_positions': run['train_positions'],
            'validation_positions': run['validation_positions'],
            'weights': weight_path.name,
            'scalers': scaler_path.name,
        }
        restored = torch.load(
            weight_path, map_location='cpu', weights_only=True
        )
        assert set(restored) == set(run['state_dict'])
        with np.load(scaler_path) as saved:
            assert saved['x_mean'].shape == run['x_mean'].shape
            assert saved['x_std'].shape == run['x_std'].shape
    (directory / f'{role}_selection.json').write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return manifest

selection_manifests = {
    role: save_routed(role, routed, OUT / 'selected_models')
    for role, routed in selected_routed.items()
}
np.savez_compressed(
    OUT / 'selected_predictions_validation.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y=validation['y'].reshape(-1, 96),
    baseline=validation['baseline_w1'].reshape(-1, 96),
    **{
        role: prediction.reshape(-1, 96)
        for role, prediction in selected_predictions.items()
    },
)
progress_results.to_csv(
    OUT / 'progress_validation.csv',
    index=False, encoding='utf-8-sig',
)
print('역할별 모델·스케일러·예측 저장 확인:', OUT.resolve())


In [ ]:
for role in ['point', 'peak_f1']:
    score = evaluate(
        validation['y'][primary_rows],
        selected_predictions[role][primary_rows],
    )
    run = tuned_selected[role]
    print(
        f'{role}: 전략={selected_strategy[role]}, '
        f'입력={len(run["names"]) + len(RECENT_NAMES)}채널'
    )
    if role == 'point':
        print(f'  MAE={score["mae"]:.4f}, RMSE={score["rmse"]:.4f}')
    else:
        print(
            f'  Precision={score["peak_precision"]:.4f}, '
            f'Recall={score["peak_recall"]:.4f}, '
            f'F1={score["peak_f1"]:.4f}, '
            f'FP={score["false_positive"]}, '
            f'FN={score["false_negative"]}'
        )
print('7월 검증 선택 후보이며 최종 테스트 성능은 아직 평가하지 않았습니다.')


In [ ]:
EXTRA_OUT = OUT / 'additional_experiments'
EXTRA_OUT.mkdir(parents=True, exist_ok=True)
extra_variants = {
    'reference': False,
    'weekly_power_only': True,
}

def simplified_names(reference_names, weekly_only):
    names = list(reference_names)
    if weekly_only:
        names = [
            name for name in names
            if not name.startswith(('w1_', 'w2_'))
            or name.startswith(('w1_power_', 'w2_power_'))
        ]
    return names

extra_features, extra_routed, extra_predictions = {}, {}, {}
extra_rows = []
for role, reference in tuned_selected.items():
    seen = {}
    for variant, switches in extra_variants.items():
        names = simplified_names(reference['names'], switches)
        signature = tuple(names)
        extra_features[(role, variant)] = names
        same_as = seen.get(signature, '')
        if same_as:
            routed = extra_routed[(role, same_as)]
        elif variant == 'reference':
            routed = selected_routed[role]
        else:
            unified = fit_gru(
                names, role, reference['params'],
                reference['objective'], reference['alpha'],
                tag=f'ablation/{role}/{variant}/unified',
            )
            routed = fit_routed(
                role, unified, selected_strategy[role],
                tag_prefix=f'ablation/{variant}',
            )
        seen.setdefault(signature, variant)
        extra_routed[(role, variant)] = routed
        extra_predictions[(role, variant)] = routed['prediction']
        for subset, rows in [
            ('primary', primary_rows),
            ('history_clean', strict_rows),
        ]:
            extra_rows.append({
                'role': role, 'variant': variant,
                'subset': subset,
                'strategy': selected_strategy[role],
                'input_channels': len(names) + len(RECENT_NAMES),
                'same_input_as': same_as,
                **evaluate(
                    validation['y'][rows],
                    routed['prediction'][rows],
                ),
            })

extra_results = pd.DataFrame(extra_rows)
extra_results.to_csv(
    EXTRA_OUT / 'input_ablation_metrics.csv',
    index=False, encoding='utf-8-sig',
)
extra_selected = {}
for role in ['point', 'peak_f1']:
    table = extra_results.loc[
        extra_results['role'].eq(role) &
        extra_results['subset'].eq('primary')
    ].copy()
    extra_selected[role] = select_id(table, role, 'variant')
    columns = [
        'variant', 'strategy', 'input_channels', 'same_input_as'
    ]
    columns += (
        ['mae', 'rmse'] if role == 'point'
        else [
            'peak_precision', 'peak_recall', 'peak_f1',
            'false_positive', 'false_negative',
        ]
    )
    print('\n수치예측 (MAE 기준)' if role == 'point'
          else '\n피크경보 (F1 기준)')
    display(table[columns].round(4))
print('역할별 입력 선택:', extra_selected)
print('history_clean은 후보 선택에 사용하지 않습니다.')
display(extra_results.loc[
    extra_results['subset'].eq('history_clean'),
    [
        'role', 'variant', 'mae',
        'peak_precision', 'peak_recall', 'peak_f1',
    ],
].round(4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, role, metric, title in [
    (axes[0], 'point', 'mae', '수치예측 MAE (낮을수록 좋음)'),
    (axes[1], 'peak_f1', 'peak_f1', '피크경보 F1 (높을수록 좋음)'),
]:
    table = extra_results.loc[
        extra_results['role'].eq(role) &
        extra_results['subset'].eq('primary')
    ]
    colors = [
        'tab:orange' if value == extra_selected[role] else 'tab:blue'
        for value in table['variant']
    ]
    ax.bar(table['variant'], table[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)
    for index, value in enumerate(table[metric]):
        ax.text(index, value, f'{value:.4f}', ha='center', va='bottom')

risk_prediction = extra_predictions[
    ('peak_f1', extra_selected['peak_f1'])
]
risk_score = evaluate(
    validation['y'][primary_rows],
    risk_prediction[primary_rows],
)
matrix = np.array([
    [risk_score['true_negative'], risk_score['false_positive']],
    [risk_score['false_negative'], risk_score['true_positive']],
])
axes[2].imshow(matrix, cmap='Blues')
for (row, col), value in np.ndenumerate(matrix):
    axes[2].text(
        col, row, f'{value:,}\n({value / matrix.sum():.2%})',
        ha='center', va='center',
        color='white' if value > matrix.max() / 2 else 'black',
    )
axes[2].set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=['예측 비피크', '예측 피크'],
    yticklabels=['실제 비피크', '실제 피크'],
    xlabel='예측', ylabel='실제',
    title=(
        f'F1 선택: {extra_selected["peak_f1"]}\n'
        f'피크 기준 {PEAK_THRESHOLD:g}'
    ),
)
fig.tight_layout()
fig.savefig(
    EXTRA_OUT / 'input_ablation_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

extra_condition_rows = []
for (role, variant), prediction in extra_predictions.items():
    for condition, mask in conditions.items():
        extra_condition_rows.append({
            'role': role, 'variant': variant,
            'condition': condition,
            **evaluate(
                validation['y'], prediction,
                mask & primary_rows[:, None, None],
            ),
        })
pd.DataFrame(extra_condition_rows).to_csv(
    EXTRA_OUT / 'input_ablation_conditions.csv',
    index=False, encoding='utf-8-sig',
)
np.savez_compressed(
    EXTRA_OUT / 'input_ablation_predictions.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y=validation['y'].reshape(-1, 96),
    **{
        f'{role}__{variant}': prediction.reshape(-1, 96)
        for (role, variant), prediction in extra_predictions.items()
    },
)
extra_manifests = {
    role: save_routed(
        role,
        extra_routed[(role, extra_selected[role])],
        EXTRA_OUT / 'selected_models',
    )
    for role in ['point', 'peak_f1']
}
(EXTRA_OUT / 'input_ablation_selection.json').write_text(
    json.dumps({
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
        'selected_variants': extra_selected,
        'manifests': extra_manifests,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)


In [ ]:
maximum_names = extra_features[
    ('point', extra_selected['point'])
]
maximum_params = tuned_selected['point']['params']
xtr_raw = make_input(train, maximum_names)
xva_raw = make_input(validation, maximum_names)
maximum_x_mean = xtr_raw.mean(axis=(0, 1), keepdims=True)
maximum_x_std = np.maximum(
    xtr_raw.std(axis=(0, 1), keepdims=True), 1e-6
)
xtr = ((xtr_raw - maximum_x_mean) / maximum_x_std).astype(np.float32)
xva = ((xva_raw - maximum_x_mean) / maximum_x_std).astype(np.float32)
maximum_y_train = train['y'].reshape(len(train['y']), -1).max(axis=1)
maximum_y_valid = validation['y'].reshape(
    len(validation['y']), -1
).max(axis=1)
maximum_target_mean = float(maximum_y_train.mean())
maximum_target_std = max(float(maximum_y_train.std()), 1e-6)
scaled_y_train = (
    (maximum_y_train - maximum_target_mean) / maximum_target_std
).astype(np.float32)
scaled_y_valid = (
    (maximum_y_valid - maximum_target_mean) / maximum_target_std
).astype(np.float32)

torch.manual_seed(SEED)
np.random.seed(SEED)
maximum_architecture = {
    'input_features': xtr.shape[-1],
    **{
        key: maximum_params[key]
        for key in [
            'hidden_size', 'num_layers',
            'dropout', 'bidirectional',
        ]
    },
    'scalar_output': True,
}
maximum_model = GRURegressor(**maximum_architecture).to(device)
maximum_optimizer = torch.optim.AdamW(
    maximum_model.parameters(),
    lr=maximum_params['learning_rate'],
    weight_decay=maximum_params['weight_decay'],
)
maximum_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(xtr),
        torch.from_numpy(scaled_y_train),
    ),
    batch_size=maximum_params['batch_size'],
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
valid_x = torch.from_numpy(xva).to(device)
valid_target = torch.from_numpy(scaled_y_valid).to(device)
valid_mask = torch.from_numpy(primary_rows).to(device)

best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
maximum_history = []
for epoch in range(1, maximum_params['max_epochs'] + 1):
    maximum_model.train()
    train_sum, train_count = 0.0, 0
    for batch_x, batch_y in maximum_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        maximum_optimizer.zero_grad()
        loss = (maximum_model(batch_x) - batch_y).square().mean()
        loss.backward()
        nn.utils.clip_grad_norm_(maximum_model.parameters(), 1.0)
        maximum_optimizer.step()
        train_sum += float(loss.detach()) * len(batch_x)
        train_count += len(batch_x)

    maximum_model.eval()
    with torch.no_grad():
        valid_loss = float(
            (maximum_model(valid_x)[valid_mask] -
             valid_target[valid_mask]).square().mean()
        )
    maximum_history.append({
        'epoch': epoch,
        'train_loss': train_sum / train_count,
        'valid_loss': valid_loss,
    })
    if valid_loss < best_loss - 1e-6:
        best_loss, best_epoch, stale = valid_loss, epoch, 0
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in maximum_model.state_dict().items()
        }
    else:
        stale += 1
        if stale >= maximum_params['patience']:
            break

maximum_model.load_state_dict(best_state)
maximum_model.eval()
parts = []
with torch.no_grad():
    for start in range(0, len(xva), maximum_params['batch_size']):
        parts.append(maximum_model(
            torch.from_numpy(
                xva[start:start + maximum_params['batch_size']]
            ).to(device)
        ).cpu().numpy())
direct_maximum = np.maximum(
    np.concatenate(parts) * maximum_target_std +
    maximum_target_mean,
    0,
).astype(np.float32)

maximum_predictions = {
    'previous_week_max': validation['baseline_w1'].reshape(
        len(validation['y']), -1
    ).max(axis=1),
    'selected_point_curve_max': selected_predictions['point'].reshape(
        len(validation['y']), -1
    ).max(axis=1),
    'ablation_selected_curve_max': extra_predictions[
        ('point', extra_selected['point'])
    ].reshape(len(validation['y']), -1).max(axis=1),
    'direct_maximum_gru': direct_maximum,
}
maximum_rows = []
for subset, rows in [
    ('primary', primary_rows),
    ('history_clean', strict_rows),
]:
    for model_name, prediction in maximum_predictions.items():
        maximum_rows.append({
            'subset': subset, 'model': model_name,
            **maximum_metrics(
                maximum_y_valid[rows], prediction[rows]
            ),
        })
direct_maximum_results = pd.DataFrame(maximum_rows)
display(direct_maximum_results.round(4))
print(
    '편향 = 예측 최대값 - 실제 최대값. '
    '음수이면 최대전력을 낮게 예측한 것입니다.'
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
table = direct_maximum_results.loc[
    direct_maximum_results['subset'].eq('primary')
]
axes[0].bar(table['model'], table['maximum_mae'])
axes[0].set_title('24시간 최대전력 MAE')
axes[0].tick_params(axis='x', rotation=30)
axes[1].scatter(
    maximum_y_valid[primary_rows],
    direct_maximum[primary_rows],
    alpha=0.3, s=14,
)
bounds = [
    min(
        maximum_y_valid[primary_rows].min(),
        direct_maximum[primary_rows].min(),
    ),
    max(
        maximum_y_valid[primary_rows].max(),
        direct_maximum[primary_rows].max(),
    ),
]
axes[1].plot(bounds, bounds, 'k--')
axes[1].set(
    xlabel='실제 최대전력',
    ylabel='직접 예측 최대전력',
    title='대각선 아래: 최대전력 과소예측',
)
fig.tight_layout()
fig.savefig(
    EXTRA_OUT / 'direct_maximum_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

direct_maximum_results.to_csv(
    EXTRA_OUT / 'maximum_metrics.csv',
    index=False, encoding='utf-8-sig',
)
pd.DataFrame(maximum_history).to_csv(
    EXTRA_OUT / 'direct_maximum_history.csv',
    index=False, encoding='utf-8-sig',
)
torch.save(
    best_state, EXTRA_OUT / 'direct_maximum_gru.pt'
)
np.savez_compressed(
    EXTRA_OUT / 'direct_maximum_scalers.npz',
    x_mean=maximum_x_mean, x_std=maximum_x_std,
    target_mean=maximum_target_mean,
    target_std=maximum_target_std,
)
np.savez_compressed(
    EXTRA_OUT / 'maximum_predictions.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y_maximum=maximum_y_valid,
    **maximum_predictions,
)
(EXTRA_OUT / 'direct_maximum_config.json').write_text(
    json.dumps({
        'sequence_features': maximum_names,
        'recent_features': RECENT_NAMES,
        'architecture': maximum_architecture,
        'params': maximum_params,
        'best_epoch': best_epoch,
        'best_valid_loss': best_loss,
        'target': 'maximum power over next 24 hours (96 quarters)',
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('추가실험 저장 위치:', EXTRA_OUT.resolve())


## 3.1 LightGBM과 동일 조건의 GRU 비교

In [ ]:
# 02_aligned_gru.py의 계산을 노트북 내부에서 직접 실행합니다.
AW_ROOT = ROOT
AW_PREP = AW_ROOT / "results/preprocessing_24h"
AW_OUT = AW_ROOT / "results/gru_24h_team_preprocessing/aligned_with_team_lgbm"
AW_OUT.mkdir(parents=True, exist_ok=True)
CFG = json.loads((AW_PREP / "config.json").read_text(encoding="utf-8"))
SEQ = CFG["sequence_features"]
log = print
AW_PEAK = 176.0
AW_WEEKS = pd.date_range("2021-05-31", "2021-07-26", freq="7D")
AW_EPOCHS = 9
AW_BATCH = 64
AW_LR = 1e-3
AW_WD = 1e-4
AW_SEED = 42
assert len(W.BASE) == 30 and len(W.L13) == 47
assert len(W.ev_val) == 627 and W.y[W.ev_val].size == 60192
assert W.meta.loc[W.ev_val, "origin"].dt.month.eq(7).all()

def aw_gru_input(columns):
    values = W.F[columns].to_numpy(np.float32)
    values = values.reshape(W.N, 24, 4, len(columns))
    return values.reshape(W.N, 24, 4 * len(columns))

AW_X30 = aw_gru_input(W.BASE)
AW_X47 = aw_gru_input(W.L13)
assert AW_X30.shape == (W.N, 24, 120)
assert AW_X47.shape == (W.N, 24, 188)
assert np.isfinite(AW_X30).all() and np.isfinite(AW_X47).all()

class AWGRU(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.gru = nn.GRU(
            input_features, 32, num_layers=2, batch_first=True,
            dropout=0.15, bidirectional=True,
        )
        self.output = nn.Linear(64, 4)

    def forward(self, values):
        sequence_output, _ = self.gru(values)
        return self.output(sequence_output)

def aw_fit_predict(
    x_train, target_train, x_predict, objective,
    alpha=0.5, cell_weight=None,
):
    if cell_weight is None:
        cell_weight = np.ones_like(target_train, dtype=np.float32)
    cell_weight = cell_weight.astype(np.float32)
    active = cell_weight > 0
    if not active.any():
        raise ValueError("학습 대상 칸이 없습니다.")

    x_mean = x_train.mean(axis=(0, 1), keepdims=True)
    x_std = x_train.std(axis=(0, 1), keepdims=True)
    x_std[x_std < 1e-6] = 1.0
    target_mean = float(target_train[active].mean())
    target_std = float(target_train[active].std())
    if target_std < 1e-6:
        target_std = 1.0

    x_train = ((x_train - x_mean) / x_std).astype(np.float32)
    x_predict = ((x_predict - x_mean) / x_std).astype(np.float32)
    target_train = (
        (target_train - target_mean) / target_std
    ).astype(np.float32)

    np.random.seed(AW_SEED)
    torch.manual_seed(AW_SEED)
    generator = torch.Generator().manual_seed(AW_SEED)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(x_train),
            torch.from_numpy(target_train),
            torch.from_numpy(cell_weight),
        ),
        batch_size=AW_BATCH,
        shuffle=True,
        generator=generator,
    )
    model = AWGRU(x_train.shape[-1])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=AW_LR, weight_decay=AW_WD
    )
    history = []
    for _ in range(AW_EPOCHS):
        model.train()
        loss_sum = 0.0
        weight_sum = 0.0
        for batch_x, batch_target, batch_weight in loader:
            optimizer.zero_grad()
            prediction = model(batch_x)
            error = batch_target - prediction
            if objective == "quantile":
                element = torch.maximum(
                    alpha * error, (alpha - 1) * error
                )
            elif objective == "l1":
                element = error.abs()
            else:
                raise ValueError(objective)
            denominator = batch_weight.sum().clamp_min(1.0)
            loss = (element * batch_weight).sum() / denominator
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_sum += float((element * batch_weight).sum())
            weight_sum += float(batch_weight.sum())
        history.append(loss_sum / max(weight_sum, 1.0))

    model.eval()
    parts = []
    with torch.inference_mode():
        for start in range(0, len(x_predict), 512):
            parts.append(
                model(torch.from_numpy(x_predict[start:start + 512])).numpy()
            )
    prediction = np.concatenate(parts) * target_std + target_mean
    return prediction.astype(np.float32), history

# 02와 같은 이례적 휴무 구간과 사후 안전장치.
aw_daily_production = pd.Series(W.hi.P).groupby(W.hi.date_d).sum()
aw_weekday_off = (
    aw_daily_production.le(0)
    & (pd.DatetimeIndex(aw_daily_production.index).dayofweek < 5)
)
AW_STREAKS = []
aw_run = []
for aw_day, aw_is_off in aw_weekday_off.items():
    if aw_is_off:
        aw_run.append(aw_day)
    else:
        if len(aw_run) >= 3:
            AW_STREAKS.append((aw_run[0], aw_run[-1]))
        aw_run = []
if len(aw_run) >= 3:
    AW_STREAKS.append((aw_run[0], aw_run[-1]))

def aw_known_off_windows(week_start):
    known = np.zeros(W.N, bool)
    for first_day, last_day in AW_STREAKS:
        if last_day + pd.Timedelta(days=1) <= week_start:
            known |= (
                W.meta["origin"].lt(last_day + pd.Timedelta(days=1))
                & W.meta["target_end"].gt(first_day)
            ).to_numpy()
    return known

AW_SCHEDULE = []

def aw_weekly_predict(
    candidate, inputs, target, objective, alpha=0.5,
    training_cells=None, use_off_weight=True,
):
    prediction = np.full_like(target, np.nan, dtype=np.float32)
    for week_start in AW_WEEKS:
        train_rows = W.meta["target_end"].le(week_start).to_numpy()
        predict_rows = (
            W.meta["origin"].ge(week_start)
            & W.meta["origin"].lt(week_start + pd.Timedelta(days=7))
        ).to_numpy()
        if not predict_rows.any():
            continue
        assert W.meta.loc[train_rows, "target_end"].max() <= week_start
        weights = np.ones_like(target[train_rows], dtype=np.float32)
        known = aw_known_off_windows(week_start)
        if use_off_weight:
            weights *= np.where(
                known[train_rows, None, None], 0.1, 1.0
            ).astype(np.float32)
        if training_cells is not None:
            weights *= training_cells[train_rows].astype(np.float32)

        started = time.time()
        week_prediction, history = aw_fit_predict(
            inputs[train_rows], target[train_rows], inputs[predict_rows],
            objective, alpha=alpha, cell_weight=weights,
        )
        prediction[predict_rows] = week_prediction
        AW_SCHEDULE.append({
            "candidate": candidate,
            "week_start": str(week_start.date()),
            "training_windows": int(train_rows.sum()),
            "prediction_windows": int(predict_rows.sum()),
            "downweighted_windows": int(
                (known & train_rows).sum() if use_off_weight else 0
            ),
            "epochs": AW_EPOCHS,
            "last_train_loss": history[-1],
            "seconds": time.time() - started,
        })
    return prediction

# 피크경보: 현재 Q1과 개선 Q1에 대응하는 GRU 후보.
aw_static_q090, aw_static_history = aw_fit_predict(
    AW_X30[W.tr_val], W.y3[W.tr_val],
    AW_X30[W.ev_val], "quantile", alpha=0.90,
)
aw_static_q090 = np.clip(aw_static_q090.reshape(-1, 96), 0, None)
AW_SCHEDULE.append({
    "candidate": "gru_current_static_30_q090",
    "week_start": "2021-07-01",
    "training_windows": int(W.tr_val.sum()),
    "prediction_windows": int(len(W.ev_val)),
    "downweighted_windows": 0,
    "epochs": AW_EPOCHS,
    "last_train_loss": aw_static_history[-1],
    "seconds": np.nan,
})

AW_PEAK_SPECS = [
    ("gru_official30_weekly_q075", AW_X30, 0.75, True),
    ("gru_enhanced47_weekly_q075_no_off_weight", AW_X47, 0.75, False),
    ("gru_enhanced47_weekly_q075", AW_X47, 0.75, True),
    ("gru_enhanced47_weekly_q080", AW_X47, 0.80, True),
    ("gru_enhanced47_weekly_q090", AW_X47, 0.90, True),
]
AW_PEAK_PREDICTIONS = {}
for candidate, inputs, alpha, use_off_weight in AW_PEAK_SPECS:
    AW_PEAK_PREDICTIONS[candidate] = np.clip(
        aw_weekly_predict(
            candidate, inputs, W.y3, "quantile",
            alpha=alpha, use_off_weight=use_off_weight,
        ).reshape(W.N, 96),
        0, None,
    )

# 수치예측: 3단 규칙 잔차, 학습 범위, 휴무일 처리를 분리해 비교.
AW_RULE = W.F["rule3_oof"].to_numpy().reshape(W.N, 96).astype(np.float32)
AW_GATE = W.F["st_day_on"].to_numpy().reshape(W.N, 24, 4) == 0
AW_RESIDUAL_TARGET = (
    W.y - AW_RULE
).reshape(W.N, 24, 4).astype(np.float32)
aw_curve_all = aw_weekly_predict(
    "gru_curve_residual_all_cells", AW_X47,
    AW_RESIDUAL_TARGET, "l1",
    training_cells=None, use_off_weight=False,
).reshape(W.N, 96) + AW_RULE
aw_curve_on = aw_weekly_predict(
    "gru_curve_residual_operating_cells", AW_X47,
    AW_RESIDUAL_TARGET, "l1",
    training_cells=~AW_GATE, use_off_weight=False,
).reshape(W.N, 96) + AW_RULE
aw_curve_all = np.clip(aw_curve_all, 0, None)
aw_curve_on = np.clip(aw_curve_on, 0, None)

# 02와 같은 과거 휴무일 시간별 중앙값.
aw_hourly = pd.read_csv(
    AW_PREP / "hourly_review.csv", parse_dates=["origin"]
)
aw_hourly["date"] = aw_hourly["origin"].dt.floor("D")
aw_hourly["hour"] = aw_hourly["origin"].dt.hour
aw_power_columns = ["15분", "30분", "45분", "60분"]
aw_day_total = aw_hourly.groupby("date")["생산량"].sum()
aw_off_days = aw_day_total[aw_day_total.eq(0)].index
aw_by_day = {
    day: group.sort_values("hour")[aw_power_columns].to_numpy()
    for day, group in aw_hourly.groupby("date") if len(group) == 24
}
AW_IDLE = np.full((W.N, 96), np.nan, np.float32)
aw_idle_cache = {}
aw_dates = W.meta["origin"].dt.floor("D").to_numpy()
aw_origin_hours = W.meta["origin"].dt.hour.to_numpy()
for index in range(W.N):
    day = pd.Timestamp(aw_dates[index])
    if day not in aw_idle_cache:
        previous_off = [
            value for value in aw_off_days
            if value < day and value in aw_by_day
        ]
        aw_idle_cache[day] = (
            np.median(
                np.stack([aw_by_day[value] for value in previous_off]),
                axis=0,
            ) if len(previous_off) >= 3 else None
        )
    profile = aw_idle_cache[day]
    if profile is not None:
        AW_IDLE[index] = profile[
            (
                aw_origin_hours[index] + np.repeat(np.arange(24), 4)
            ) % 24,
            np.tile(np.arange(4), 24),
        ]
aw_gate_flat = AW_GATE.reshape(W.N, 96)
aw_curve_on_rule_off = np.where(
    aw_gate_flat, AW_RULE, aw_curve_on
).astype(np.float32)
aw_curve_final = np.where(
    aw_gate_flat & np.isfinite(AW_IDLE),
    AW_IDLE,
    aw_curve_on_rule_off,
).astype(np.float32)

def aw_metrics(target, prediction, threshold=AW_PEAK):
    threshold_values = (
        np.broadcast_to(
            np.asarray(threshold, float).reshape(-1, 1), target.shape
        ) if np.ndim(threshold) else threshold
    )
    actual = target >= AW_PEAK
    alarm = prediction >= threshold_values
    tp = int((actual & alarm).sum())
    fp = int((~actual & alarm).sum())
    fn = int((actual & ~alarm).sum())
    tn = int((~actual & ~alarm).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    f2 = (
        5 * precision * recall / (4 * precision + recall)
        if precision + recall else 0.0
    )
    error = prediction - target
    return {
        "observations": int(target.size),
        "mae": float(np.abs(error).mean()),
        "rmse": float(np.sqrt((error ** 2).mean())),
        "bias": float(error.mean()),
        "precision": precision, "recall": recall,
        "f1": f1, "f2": f2,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

# 개선 Q1과 같은 최근 4주 R85/R95 경보 기준.
AW_WEEK_OF = (
    (W.meta["origin"] - AW_WEEKS[0]) // pd.Timedelta(days=7)
).to_numpy()
def aw_weekly_thresholds(prediction, target_policy):
    thresholds = np.full(W.N, AW_PEAK, dtype=float)
    for week in np.unique(AW_WEEK_OF[W.ev_val]):
        week_start = AW_WEEKS[0] + pd.Timedelta(days=7) * int(week)
        history = (
            np.isfinite(prediction[:, 0])
            & W.meta["target_end"].le(week_start).to_numpy()
            & W.meta["origin"].ge(
                week_start - pd.Timedelta(days=28)
            ).to_numpy()
            & W.meta["origin"].ge(AW_WEEKS[0]).to_numpy()
        )
        if history.any():
            thresholds[AW_WEEK_OF == week] = choose(
                W.y[history], prediction[history], target_policy
            )
    return thresholds

AW_PRIMARY_PEAK_NAME = "gru_enhanced47_weekly_q075"
AW_PRIMARY_PEAK = AW_PEAK_PREDICTIONS[AW_PRIMARY_PEAK_NAME]
AW_THRESHOLD_R85 = aw_weekly_thresholds(AW_PRIMARY_PEAK, "R85")
AW_THRESHOLD_R95 = aw_weekly_thresholds(AW_PRIMARY_PEAK, "R95")

# 같은 최근 4주 결과만 사용한 피크 확률 보정.
AW_PROBABILITY = np.full((W.N, 96), np.nan, np.float32)
aw_labels = W.y >= AW_PEAK
for week in np.unique(AW_WEEK_OF[W.ev_val]):
    week_start = AW_WEEKS[0] + pd.Timedelta(days=7) * int(week)
    week_rows = AW_WEEK_OF == week
    history = (
        np.isfinite(AW_PRIMARY_PEAK[:, 0])
        & W.meta["target_end"].le(week_start).to_numpy()
        & W.meta["origin"].ge(
            week_start - pd.Timedelta(days=28)
        ).to_numpy()
        & W.meta["origin"].ge(AW_WEEKS[0]).to_numpy()
    )
    history_labels = aw_labels[history].reshape(-1)
    if history.any() and np.unique(history_labels).size == 2:
        calibration = LogisticRegression(C=1e4, max_iter=200)
        calibration.fit(
            (
                AW_PRIMARY_PEAK[history].reshape(-1, 1) - AW_PEAK
            ) / 20.0,
            history_labels,
        )
        AW_PROBABILITY[week_rows] = calibration.predict_proba(
            (
                AW_PRIMARY_PEAK[week_rows].reshape(-1, 1) - AW_PEAK
            ) / 20.0
        )[:, 1].reshape(-1, 96)

aw_peak_rows = [{
    "candidate": "gru_current_static_30_q090",
    "input_features": 30,
    "retraining": "once_before_july",
    "quantile": 0.90,
    "off_streak_weight": 1.0,
    "threshold_policy": "fixed_176",
    **aw_metrics(W.y[W.ev_val], aw_static_q090),
}]
for candidate, _, alpha, use_off_weight in AW_PEAK_SPECS:
    prediction = AW_PEAK_PREDICTIONS[candidate]
    aw_peak_rows.append({
        "candidate": candidate,
        "input_features": 30 if "official30" in candidate else 47,
        "retraining": "weekly",
        "quantile": alpha,
        "off_streak_weight": 0.1 if use_off_weight else 1.0,
        "threshold_policy": "fixed_176",
        **aw_metrics(W.y[W.ev_val], prediction[W.ev_val]),
    })
aw_peak_rows.append({
    "candidate": AW_PRIMARY_PEAK_NAME,
    "input_features": 47,
    "retraining": "weekly",
    "quantile": 0.75,
    "off_streak_weight": 0.1,
    "threshold_policy": "R85_recent_4_weeks",
    **aw_metrics(
        W.y[W.ev_val], AW_PRIMARY_PEAK[W.ev_val],
        AW_THRESHOLD_R85[W.ev_val],
    ),
})
aw_peak_results = pd.DataFrame(aw_peak_rows)
aw_peak_results.to_csv(
    AW_OUT / "aligned_peak_metrics_validation.csv",
    index=False, encoding="utf-8-sig",
)

AW_CURVE_PREDICTIONS = {
    "previous_week_baseline": W.base,
    "rule3_oof": AW_RULE,
    "gru_residual_all_cells": aw_curve_all,
    "gru_residual_operating_cells_rule_off": aw_curve_on_rule_off,
    "gru_residual_operating_cells_idle_profile": aw_curve_final,
}
aw_curve_rows = []
for candidate, prediction in AW_CURVE_PREDICTIONS.items():
    aw_curve_rows.append({
        "candidate": candidate,
        "training_schedule": (
            "weekly" if candidate.startswith("gru_") else "none"
        ),
        "training_cells": (
            "all" if candidate == "gru_residual_all_cells"
            else "operating_only" if candidate.startswith("gru_")
            else "not_applicable"
        ),
        "off_day_output": (
            "idle_profile" if candidate.endswith("idle_profile")
            else "rule3" if candidate.endswith("rule_off")
            else "model_or_baseline"
        ),
        **aw_metrics(W.y[W.ev_val], prediction[W.ev_val]),
    })
aw_curve_results = pd.DataFrame(aw_curve_rows)
aw_curve_results.to_csv(
    AW_OUT / "aligned_curve_metrics_validation.csv",
    index=False, encoding="utf-8-sig",
)
pd.DataFrame(AW_SCHEDULE).to_csv(
    AW_OUT / "training_schedule.csv",
    index=False, encoding="utf-8-sig",
)

aw_origins = W.meta.loc[W.ev_val, "origin"].to_numpy()
aw_steps = np.tile(np.arange(96), len(W.ev_val))
aw_origin_column = np.repeat(aw_origins, 96)
aw_target_time = (
    pd.to_datetime(aw_origin_column)
    + pd.to_timedelta(aw_steps // 4, unit="h")
    + pd.to_timedelta(15 * (aw_steps % 4 + 1), unit="m")
)
aw_prediction_table = pd.DataFrame({
    "기준시각": aw_origin_column,
    "예측대상시각": aw_target_time,
    "예측칸": aw_steps,
    "실제전력": W.y[W.ev_val].reshape(-1),
    "baseline_w1": W.base[W.ev_val].reshape(-1),
    "rule3_oof": AW_RULE[W.ev_val].reshape(-1),
    "gru_current_static_30_q090": aw_static_q090.reshape(-1),
    **{
        name: prediction[W.ev_val].reshape(-1)
        for name, prediction in AW_PEAK_PREDICTIONS.items()
    },
    **{
        name: prediction[W.ev_val].reshape(-1)
        for name, prediction in AW_CURVE_PREDICTIONS.items()
        if name.startswith("gru_")
    },
    "fixed_threshold": AW_PEAK,
    "r85_recent_4_weeks_threshold": np.repeat(
        AW_THRESHOLD_R85[W.ev_val], 96
    ),
    "r95_recent_4_weeks_threshold": np.repeat(
        AW_THRESHOLD_R95[W.ev_val], 96
    ),
    "peak_probability": AW_PROBABILITY[W.ev_val].reshape(-1),
})
aw_prediction_table.to_csv(
    AW_OUT / "aligned_predictions_validation.csv",
    index=False, encoding="utf-8-sig",
)

aw_manifest = {
    "shared_preprocessing": str(AW_PREP),
    "team_feature_source": str(AW_NOTEBOOK),
    "source_sha256": CFG["source_sha256"],
    "evaluation_period": "2021-07 primary_eligible",
    "evaluation_windows": int(len(W.ev_val)),
    "evaluation_cells": int(W.y[W.ev_val].size),
    "actual_peak_threshold": AW_PEAK,
    "weekly_retraining": [str(value.date()) for value in AW_WEEKS],
    "official_features": W.BASE,
    "enhanced_features": W.L13,
    "seed": AW_SEED,
    "epochs": AW_EPOCHS,
    "batch_size": AW_BATCH,
    "learning_rate": AW_LR,
    "weight_decay": AW_WD,
    "test_used_for_training_or_selection": False,
}
(AW_OUT / "experiment_manifest.json").write_text(
    json.dumps(aw_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("피크경보 동일 조건 GRU 실험")
display(aw_peak_results.round(4))
print("수치예측 동일 조건 GRU 실험")
display(
    aw_curve_results[["candidate", "mae", "rmse", "bias"]].round(4)
)
print("저장 완료:", AW_OUT.resolve())



In [ ]:
# ============================================================
# 11. 8~9월 동일 조건 개발 검증 및 최종 GRU 선정
# ============================================================

# 02와 동일하게 8~9월에도 매주 월요일 이전까지 정답이 확정된
# 데이터만 사용하여 다시 학습합니다.
AW_TEST_WEEKS = pd.date_range(
    "2021-08-02", "2021-09-13", freq="7D"
)

assert len(W.ev_test) == 1057
assert W.y[W.ev_test].size == 1057 * 96
assert W.meta.loc[W.ev_test, "origin"].dt.month.isin([8, 9]).all()


def aw_predict_additional_weeks(
    initial_prediction,
    candidate,
    inputs,
    target,
    objective,
    weeks,
    alpha=0.5,
    training_cells=None,
    use_off_weight=True,
):
    """
    기존 7월 예측을 유지하고 8~9월 주별 예측을 이어서 생성합니다.
    각 주 시작 전에 정답이 확정된 창만 학습에 사용합니다.
    """
    prediction = initial_prediction.copy()

    for week_start in weeks:
        train_rows = W.meta["target_end"].le(week_start).to_numpy()
        predict_rows = (
            W.meta["origin"].ge(week_start)
            & W.meta["origin"].lt(
                week_start + pd.Timedelta(days=7)
            )
        ).to_numpy()

        if not predict_rows.any():
            continue

        assert W.meta.loc[
            train_rows, "target_end"
        ].max() <= week_start

        weights = np.ones_like(
            target[train_rows], dtype=np.float32
        )

        known_off_windows = aw_known_off_windows(week_start)

        if use_off_weight:
            weights *= np.where(
                known_off_windows[train_rows, None, None],
                0.1,
                1.0,
            ).astype(np.float32)

        if training_cells is not None:
            weights *= training_cells[train_rows].astype(
                np.float32
            )

        started = time.time()

        week_prediction, history = aw_fit_predict(
            inputs[train_rows],
            target[train_rows],
            inputs[predict_rows],
            objective,
            alpha=alpha,
            cell_weight=weights,
        )

        prediction[predict_rows] = week_prediction

        AW_SCHEDULE.append({
            "candidate": candidate,
            "week_start": str(week_start.date()),
            "training_windows": int(train_rows.sum()),
            "prediction_windows": int(predict_rows.sum()),
            "downweighted_windows": int(
                (known_off_windows & train_rows).sum()
                if use_off_weight else 0
            ),
            "epochs": AW_EPOCHS,
            "last_train_loss": history[-1],
            "seconds": time.time() - started,
        })

        print(
            candidate,
            week_start.date(),
            f"학습 {int(train_rows.sum()):,}창",
            f"예측 {int(predict_rows.sum()):,}창",
        )

    return prediction


# ------------------------------------------------------------
# 1. 현재 Q1 대응 GRU: 1~7월 학습 후 8~9월 전체 예측
# ------------------------------------------------------------

aw_static_test_q090, aw_static_test_history = (
    aw_fit_predict(
        AW_X30[W.tr_test],
        W.y3[W.tr_test],
        AW_X30[W.ev_test],
        "quantile",
        alpha=0.90,
    )
)

aw_static_test_q090 = np.clip(
    aw_static_test_q090.reshape(-1, 96),
    0,
    None,
)

AW_SCHEDULE.append({
    "candidate": "gru_current_static_30_q090",
    "week_start": "2021-08-01",
    "training_windows": int(W.tr_test.sum()),
    "prediction_windows": int(len(W.ev_test)),
    "downweighted_windows": 0,
    "epochs": AW_EPOCHS,
    "last_train_loss": aw_static_test_history[-1],
    "seconds": np.nan,
})


# ------------------------------------------------------------
# 2. 개선 Q1 대응 GRU: 8~9월 주별 재학습
# ------------------------------------------------------------

AW_PEAK_PREDICTIONS_ALL = {}

for candidate, inputs, alpha, use_off_weight in AW_PEAK_SPECS:
    # 기존 배열에는 5월 31일~8월 1일 예측이 들어 있습니다.
    initial = AW_PEAK_PREDICTIONS[candidate].reshape(
        W.N, 24, 4
    )

    prediction = aw_predict_additional_weeks(
        initial_prediction=initial,
        candidate=candidate,
        inputs=inputs,
        target=W.y3,
        objective="quantile",
        weeks=AW_TEST_WEEKS,
        alpha=alpha,
        training_cells=None,
        use_off_weight=use_off_weight,
    )

    prediction = np.clip(
        prediction.reshape(W.N, 96),
        0,
        None,
    )

    assert np.isfinite(
        prediction[W.ev_val]
    ).all()

    assert np.isfinite(
        prediction[W.ev_test]
    ).all()

    AW_PEAK_PREDICTIONS_ALL[candidate] = prediction


# ------------------------------------------------------------
# 3. 수치예측 GRU: 8~9월 주별 잔차 재학습
# ------------------------------------------------------------

aw_curve_all_full = aw_predict_additional_weeks(
    initial_prediction=(
        aw_curve_all - AW_RULE
    ).reshape(W.N, 24, 4),
    candidate="gru_curve_residual_all_cells",
    inputs=AW_X47,
    target=AW_RESIDUAL_TARGET,
    objective="l1",
    weeks=AW_TEST_WEEKS,
    training_cells=None,
    use_off_weight=False,
)

aw_curve_all_full = np.clip(
    aw_curve_all_full.reshape(W.N, 96) + AW_RULE,
    0,
    None,
)

aw_curve_on_full = aw_predict_additional_weeks(
    initial_prediction=(
        aw_curve_on - AW_RULE
    ).reshape(W.N, 24, 4),
    candidate="gru_curve_residual_operating_cells",
    inputs=AW_X47,
    target=AW_RESIDUAL_TARGET,
    objective="l1",
    weeks=AW_TEST_WEEKS,
    training_cells=~AW_GATE,
    use_off_weight=False,
)

aw_curve_on_full = np.clip(
    aw_curve_on_full.reshape(W.N, 96) + AW_RULE,
    0,
    None,
)

aw_curve_on_rule_off_full = np.where(
    aw_gate_flat,
    AW_RULE,
    aw_curve_on_full,
).astype(np.float32)

aw_curve_final_full = np.where(
    aw_gate_flat & np.isfinite(AW_IDLE),
    AW_IDLE,
    aw_curve_on_rule_off_full,
).astype(np.float32)

assert np.isfinite(
    aw_curve_all_full[W.ev_test]
).all()

assert np.isfinite(
    aw_curve_final_full[W.ev_test]
).all()


# ------------------------------------------------------------
# 4. 7월·8~9월·전체 기간의 피크경보 성능 비교
# ------------------------------------------------------------

AW_STATIC_FULL = np.full(
    (W.N, 96),
    np.nan,
    dtype=np.float32,
)

AW_STATIC_FULL[W.ev_val] = aw_static_q090
AW_STATIC_FULL[W.ev_test] = aw_static_test_q090

aw_peak_candidates_all = {
    "gru_current_static_30_q090": AW_STATIC_FULL,
    **AW_PEAK_PREDICTIONS_ALL,
}

aw_periods = {
    "validation_7월": W.ev_val,
    "development_8_9월": W.ev_test,
    "pooled_7_9월": np.concatenate([
        W.ev_val,
        W.ev_test,
    ]),
}

aw_peak_period_rows = []

for period, rows in aw_periods.items():
    for candidate, prediction in (
        aw_peak_candidates_all.items()
    ):
        aw_peak_period_rows.append({
            "period": period,
            "candidate": candidate,
            **aw_metrics(
                W.y[rows],
                prediction[rows],
                AW_PEAK,
            ),
        })

aw_peak_period_results = pd.DataFrame(
    aw_peak_period_rows
)

display(
    aw_peak_period_results[[
        "period",
        "candidate",
        "precision",
        "recall",
        "f1",
        "f2",
        "fp",
        "fn",
        "mae",
    ]].round(4)
)


# 전체 7~9월 F1을 기준으로 최종 피크경보 GRU를 선택합니다.
aw_peak_pooled = (
    aw_peak_period_results.loc[
        aw_peak_period_results[
            "period"
        ].eq("pooled_7_9월")
    ]
    .copy()
)

# 전체 F1이 같으면 두 기간 중 낮은 F1이 더 높은 모델,
# 그다음 정밀도가 높은 모델을 선택합니다.
aw_period_f1 = (
    aw_peak_period_results.loc[
        aw_peak_period_results["period"].isin([
            "validation_7월",
            "development_8_9월",
        ])
    ]
    .pivot(
        index="candidate",
        columns="period",
        values="f1",
    )
)

aw_period_f1["minimum_period_f1"] = (
    aw_period_f1.min(axis=1)
)

aw_peak_pooled = aw_peak_pooled.merge(
    aw_period_f1[["minimum_period_f1"]],
    left_on="candidate",
    right_index=True,
    how="left",
)

aw_selected_peak_row = (
    aw_peak_pooled.sort_values(
        [
            "f1",
            "minimum_period_f1",
            "precision",
            "recall",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .iloc[0]
)

AW_SELECTED_PEAK_MODEL = (
    aw_selected_peak_row["candidate"]
)

AW_SELECTED_PEAK_PREDICTION = (
    aw_peak_candidates_all[
        AW_SELECTED_PEAK_MODEL
    ]
)

print(
    "최종 피크경보 GRU:",
    AW_SELECTED_PEAK_MODEL,
)

print(
    "7~9월 통합 F1:",
    round(
        float(aw_selected_peak_row["f1"]),
        4,
    ),
)


# ------------------------------------------------------------
# 5. 7월·8~9월·전체 기간의 수치예측 성능 비교
# ------------------------------------------------------------

aw_curve_candidates_all = {
    "previous_week_baseline": W.base,
    "rule3_oof": AW_RULE,
    "gru_residual_all_cells": (
        aw_curve_all_full
    ),
    "gru_residual_operating_cells_rule_off": (
        aw_curve_on_rule_off_full
    ),
    "gru_residual_operating_cells_idle_profile": (
        aw_curve_final_full
    ),
}

aw_curve_period_rows = []

for period, rows in aw_periods.items():
    for candidate, prediction in (
        aw_curve_candidates_all.items()
    ):
        score = aw_metrics(
            W.y[rows],
            prediction[rows],
            AW_PEAK,
        )

        aw_curve_period_rows.append({
            "period": period,
            "candidate": candidate,
            "mae": score["mae"],
            "rmse": score["rmse"],
            "bias": score["bias"],
        })

aw_curve_period_results = pd.DataFrame(
    aw_curve_period_rows
)

display(
    aw_curve_period_results.round(4)
)


# GRU 수치예측 후보 중 7~9월 통합 MAE가 가장 낮은 모델을 선택합니다.
aw_curve_pooled = (
    aw_curve_period_results.loc[
        aw_curve_period_results[
            "period"
        ].eq("pooled_7_9월")
        & aw_curve_period_results[
            "candidate"
        ].str.startswith("gru_")
    ]
    .copy()
)

aw_curve_period_mae = (
    aw_curve_period_results.loc[
        aw_curve_period_results["period"].isin([
            "validation_7월",
            "development_8_9월",
        ])
        & aw_curve_period_results[
            "candidate"
        ].str.startswith("gru_")
    ]
    .pivot(
        index="candidate",
        columns="period",
        values="mae",
    )
)

aw_curve_period_mae["maximum_period_mae"] = (
    aw_curve_period_mae.max(axis=1)
)

aw_curve_pooled = aw_curve_pooled.merge(
    aw_curve_period_mae[
        ["maximum_period_mae"]
    ],
    left_on="candidate",
    right_index=True,
    how="left",
)

aw_selected_curve_row = (
    aw_curve_pooled.sort_values(
        [
            "mae",
            "maximum_period_mae",
            "rmse",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="stable",
    )
    .iloc[0]
)

AW_SELECTED_CURVE_MODEL = (
    aw_selected_curve_row["candidate"]
)

AW_SELECTED_CURVE_PREDICTION = (
    aw_curve_candidates_all[
        AW_SELECTED_CURVE_MODEL
    ]
)

print(
    "최종 수치예측 GRU:",
    AW_SELECTED_CURVE_MODEL,
)

print(
    "7~9월 통합 MAE:",
    round(
        float(aw_selected_curve_row["mae"]),
        4,
    ),
)


# ------------------------------------------------------------
# 6. 결과 저장
# ------------------------------------------------------------

aw_peak_period_results.to_csv(
    AW_OUT / "aligned_peak_metrics_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_curve_period_results.to_csv(
    AW_OUT / "aligned_curve_metrics_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(AW_SCHEDULE).to_csv(
    AW_OUT / "training_schedule_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)


def aw_prediction_frame(rows, period):
    origins = W.meta.loc[
        rows,
        "origin",
    ].to_numpy()

    steps = np.tile(
        np.arange(96),
        len(rows),
    )

    origin_column = np.repeat(
        origins,
        96,
    )

    target_time = (
        pd.to_datetime(origin_column)
        + pd.to_timedelta(
            steps // 4,
            unit="h",
        )
        + pd.to_timedelta(
            15 * (steps % 4 + 1),
            unit="m",
        )
    )

    frame = pd.DataFrame({
        "period": period,
        "기준시각": origin_column,
        "예측대상시각": target_time,
        "예측칸": steps,
        "실제전력": W.y[rows].reshape(-1),
        "baseline_w1": W.base[
            rows
        ].reshape(-1),
        "rule3_oof": AW_RULE[
            rows
        ].reshape(-1),
    })

    for candidate, prediction in (
        aw_peak_candidates_all.items()
    ):
        frame[candidate] = prediction[
            rows
        ].reshape(-1)

    for candidate, prediction in (
        aw_curve_candidates_all.items()
    ):
        if candidate.startswith("gru_"):
            frame[candidate] = prediction[
                rows
            ].reshape(-1)

    frame["selected_peak_gru"] = (
        AW_SELECTED_PEAK_PREDICTION[
            rows
        ].reshape(-1)
    )

    frame["selected_curve_gru"] = (
        AW_SELECTED_CURVE_PREDICTION[
            rows
        ].reshape(-1)
    )

    frame["peak_threshold"] = AW_PEAK

    frame["actual_peak"] = (
        frame["실제전력"] >= AW_PEAK
    ).astype(np.int8)

    frame["selected_gru_alarm"] = (
        frame["selected_peak_gru"]
        >= AW_PEAK
    ).astype(np.int8)

    return frame


aw_predictions_7월 = aw_prediction_frame(
    W.ev_val,
    "validation_7월",
)

aw_predictions_8_9월 = aw_prediction_frame(
    W.ev_test,
    "development_8_9월",
)

aw_predictions_all = pd.concat(
    [
        aw_predictions_7월,
        aw_predictions_8_9월,
    ],
    ignore_index=True,
)

aw_predictions_7월.to_csv(
    AW_OUT / "aligned_predictions_validation_7월.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_predictions_8_9월.to_csv(
    AW_OUT / "aligned_predictions_development_8_9월.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_predictions_all.to_csv(
    AW_OUT / "aligned_predictions_all_7_9월.csv",
    index=False,
    encoding="utf-8-sig",
)


aw_final_selection = {
    "selection_period": (
        "2021-07 validation + "
        "2021-08~09 development validation"
    ),
    "independent_test_remaining": False,
    "actual_peak_threshold": AW_PEAK,
    "peak_selection_metric": (
        "pooled F1; tie: minimum period F1, "
        "precision, recall"
    ),
    "selected_peak_gru": (
        AW_SELECTED_PEAK_MODEL
    ),
    "selected_peak_metrics_pooled": {
        key: (
            int(aw_selected_peak_row[key])
            if key in [
                "observations",
                "tp",
                "fp",
                "fn",
                "tn",
            ]
            else float(
                aw_selected_peak_row[key]
            )
        )
        for key in [
            "observations",
            "mae",
            "rmse",
            "bias",
            "precision",
            "recall",
            "f1",
            "f2",
            "tp",
            "fp",
            "fn",
            "tn",
        ]
    },
    "curve_selection_metric": (
        "pooled MAE; tie: maximum period MAE, RMSE"
    ),
    "selected_curve_gru": (
        AW_SELECTED_CURVE_MODEL
    ),
    "selected_curve_metrics_pooled": {
        "mae": float(
            aw_selected_curve_row["mae"]
        ),
        "rmse": float(
            aw_selected_curve_row["rmse"]
        ),
        "bias": float(
            aw_selected_curve_row["bias"]
        ),
    },
    "weekly_retraining_through": (
        "2021-09-13"
    ),
    "test_used_for_selection": True,
    "test_label": "development_validation",
}

(
    AW_OUT / "final_gru_selection_7_9월.json"
).write_text(
    json.dumps(
        aw_final_selection,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "저장 완료:",
    AW_OUT.resolve(),
)

# 4. LightGBM GRU 앙상블 비교와 최종 경보 선택

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ROOT = PROJECT_ROOT
if not (ROOT / 'results/preprocessing_24h/config.json').exists():
    ROOT = PROJECT_ROOT

PREP = ROOT / 'results/preprocessing_24h'
LGBM_FILES = {
    'validation_7월': ROOT / 'results/team_submission_runtime/outputs/validation_predictions_7월.csv',
    'development_8_9월': ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv',
}
GRU_FILES = {
    'validation_7월': ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/aligned_predictions_validation_7월.csv',
    'development_8_9월': ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/aligned_predictions_development_8_9월.csv',
}
GRU_SELECTION_FILE = (
    ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/'
    'final_gru_selection_7_9월.json'
)
OUT = ROOT / 'results/ensemble_lgbm_gru_team_aligned'
OUT.mkdir(parents=True, exist_ok=True)

with (PREP / 'config.json').open(encoding='utf-8') as file:
    prep_config = json.load(file)
with GRU_SELECTION_FILE.open(encoding='utf-8') as file:
    gru_selection = json.load(file)

THRESHOLD = float(prep_config['peak_threshold'])
LGBM_COLUMN = 'q1_new'
GRU_COLUMN = 'gru_enhanced47_weekly_q080'
GRU_WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)
PERIODS = ['validation_7월', 'development_8_9월']
EXPECTED_ROWS = {'validation_7월': 60192, 'development_8_9월': 101472}

assert THRESHOLD == 176.0
assert gru_selection['selected_peak_gru'] == GRU_COLUMN
assert gru_selection['test_label'] == 'development_validation'
assert gru_selection['independent_test_remaining'] is False

print('LGBM 예측:', {k: str(v.resolve()) for k, v in LGBM_FILES.items()})
print('GRU 예측:', {k: str(v.resolve()) for k, v in GRU_FILES.items()})
print('GRU 최종 후보:', GRU_COLUMN)
print('피크 기준:', THRESHOLD)
print('GRU 비중 후보:', GRU_WEIGHTS.tolist())


In [ ]:
keys = ['기준시각', '예측대상시각', '예측칸']
aligned_parts = []

for period in PERIODS:
    lgbm_raw = pd.read_csv(
        LGBM_FILES[period],
        usecols=keys + ['실제전력', LGBM_COLUMN],
        parse_dates=['기준시각', '예측대상시각'],
    )
    gru_raw = pd.read_csv(
        GRU_FILES[period],
        usecols=keys + ['실제전력', GRU_COLUMN],
        parse_dates=['기준시각', '예측대상시각'],
    )
    assert not lgbm_raw.duplicated(keys).any()
    assert not gru_raw.duplicated(keys).any()

    part = lgbm_raw.merge(
        gru_raw,
        on=keys,
        how='inner',
        suffixes=('_lgbm', '_gru'),
        validate='one_to_one',
    ).sort_values(keys, kind='stable').reset_index(drop=True)

    assert len(part) == len(lgbm_raw) == len(gru_raw) == EXPECTED_ROWS[period]
    np.testing.assert_allclose(
        part['실제전력_lgbm'].to_numpy(float),
        part['실제전력_gru'].to_numpy(float),
        rtol=0,
        atol=1e-6,
    )
    part.insert(0, 'period', period)
    aligned_parts.append(part)

aligned = pd.concat(aligned_parts, ignore_index=True)
aligned['actual'] = aligned['실제전력_lgbm'].astype(np.float64)
aligned['lgbm_prediction'] = aligned[LGBM_COLUMN].astype(np.float64)
aligned['gru_prediction'] = aligned[GRU_COLUMN].astype(np.float64)

assert len(aligned) == sum(EXPECTED_ROWS.values())
assert np.isfinite(aligned[['actual', 'lgbm_prediction', 'gru_prediction']]).all().all()

for period in PERIODS:
    part = aligned.loc[aligned['period'].eq(period)]
    print(
        period, ':', len(part), '칸 / 실제 피크',
        int(part['actual'].ge(THRESHOLD).sum()), '칸 / 기준시각',
        part['기준시각'].min(), '~', part['기준시각'].max(),
    )
print('통합:', len(aligned), '칸')


In [ ]:
def evaluate_peak(y_true, prediction, threshold=THRESHOLD):
    actual_peak = y_true >= threshold
    predicted_peak = prediction >= threshold
    tp = int((actual_peak & predicted_peak).sum())
    fp = int((~actual_peak & predicted_peak).sum())
    fn = int((actual_peak & ~predicted_peak).sum())
    tn = int((~actual_peak & ~predicted_peak).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if precision + recall else 0.0
    error = prediction - y_true
    return {
        'observations': int(len(y_true)),
        'mae': float(np.abs(error).mean()),
        'rmse': float(np.sqrt(np.mean(error ** 2))),
        'bias': float(error.mean()),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
    }

weight_rows = []
for gru_weight in GRU_WEIGHTS:
    for period in PERIODS + ['pooled_7_9월']:
        part = aligned if period == 'pooled_7_9월' else aligned.loc[aligned['period'].eq(period)]
        prediction = (
            (1.0 - gru_weight) * part['lgbm_prediction'].to_numpy()
            + gru_weight * part['gru_prediction'].to_numpy()
        )
        weight_rows.append({
            'period': period,
            'gru_weight': float(gru_weight),
            'lgbm_weight': float(1.0 - gru_weight),
            **evaluate_peak(part['actual'].to_numpy(), prediction),
        })
weight_results = pd.DataFrame(weight_rows)

period_min_f1 = (
    weight_results.loc[weight_results['period'].isin(PERIODS)]
    .groupby('gru_weight', as_index=False)['f1']
    .min()
    .rename(columns={'f1': 'minimum_period_f1'})
)
selection_table = (
    weight_results.loc[weight_results['period'].eq('pooled_7_9월')]
    .merge(period_min_f1, on='gru_weight', how='left', validate='one_to_one')
)
selected = selection_table.sort_values(
    ['f1', 'minimum_period_f1', 'precision', 'recall', 'gru_weight'],
    ascending=[False, False, False, False, True],
    kind='stable',
).iloc[0]
selected_gru_weight = float(selected['gru_weight'])
selected_lgbm_weight = float(selected['lgbm_weight'])
aligned['ensemble_prediction'] = (
    selected_lgbm_weight * aligned['lgbm_prediction']
    + selected_gru_weight * aligned['gru_prediction']
)

display(selection_table[[
    'lgbm_weight', 'gru_weight', 'precision', 'recall',
    'f1', 'minimum_period_f1', 'f2', 'fp', 'fn',
]].round(4))
print(
    f'선택 비율: LGBM {selected_lgbm_weight:.0%} + '
    f'GRU {selected_gru_weight:.0%}'
)


In [ ]:
model_columns = {
    'LGBM_q075': 'lgbm_prediction',
    'GRU_q080': 'gru_prediction',
    'ensemble': 'ensemble_prediction',
}
comparison_rows = []
for period in PERIODS + ['pooled_7_9월']:
    part = aligned if period == 'pooled_7_9월' else aligned.loc[aligned['period'].eq(period)]
    for model, column in model_columns.items():
        comparison_rows.append({
            'period': period,
            'model': model,
            'lgbm_weight': (
                1.0 if model == 'LGBM_q075'
                else 0.0 if model == 'GRU_q080'
                else selected_lgbm_weight
            ),
            'gru_weight': (
                0.0 if model == 'LGBM_q075'
                else 1.0 if model == 'GRU_q080'
                else selected_gru_weight
            ),
            **evaluate_peak(
                part['actual'].to_numpy(),
                part[column].to_numpy(),
            ),
        })
comparison = pd.DataFrame(comparison_rows)
pooled_comparison = comparison.loc[comparison['period'].eq('pooled_7_9월')]
single_best_f1 = pooled_comparison.loc[
    pooled_comparison['model'].ne('ensemble'), 'f1'
].max()
ensemble_f1 = float(
    pooled_comparison.loc[pooled_comparison['model'].eq('ensemble'), 'f1'].iloc[0]
)
ensemble_improves_f1 = bool(ensemble_f1 > single_best_f1 + 1e-12)

display(comparison[[
    'period', 'model', 'lgbm_weight', 'gru_weight',
    'precision', 'recall', 'f1', 'f2', 'fp', 'fn',
]].round(4))
print('통합 단독 최고 F1:', round(float(single_best_f1), 4))
print('통합 앙상블 F1:', round(ensemble_f1, 4))
print('앙상블이 단독 최고 F1을 초과:', ensemble_improves_f1)


In [ ]:
weight_results.to_csv(
    OUT / 'weight_comparison_all_periods.csv',
    index=False,
    encoding='utf-8-sig',
)
comparison.to_csv(
    OUT / 'model_comparison_all_periods.csv',
    index=False,
    encoding='utf-8-sig',
)

prediction_output = aligned[['period'] + keys].copy()
prediction_output['actual'] = aligned['actual']
prediction_output['lgbm_q075'] = aligned['lgbm_prediction']
prediction_output['gru_q080'] = aligned['gru_prediction']
prediction_output['ensemble'] = aligned['ensemble_prediction']
prediction_output['threshold'] = THRESHOLD
prediction_output['actual_peak'] = aligned['actual'].ge(THRESHOLD).astype(np.int8)
prediction_output['lgbm_alarm'] = aligned['lgbm_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output['gru_alarm'] = aligned['gru_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output['ensemble_alarm'] = aligned['ensemble_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output.to_csv(
    OUT / 'predictions_all_7_9월.csv',
    index=False,
    encoding='utf-8-sig',
)

selection = {
    'evaluation_period': '2021-07 validation + 2021-08~09 development validation',
    'evaluation_cells': int(len(aligned)),
    'actual_peak_threshold': THRESHOLD,
    'selection_metric': 'pooled F1; tie: minimum period F1, precision, recall, lower GRU weight',
    'weight_candidates': GRU_WEIGHTS.tolist(),
    'lgbm_model': 'improved_Q1_q075',
    'lgbm_prediction_column': LGBM_COLUMN,
    'lgbm_sources': {key: str(value) for key, value in LGBM_FILES.items()},
    'gru_model': 'enhanced_47_features_q080',
    'gru_prediction_column': GRU_COLUMN,
    'gru_sources': {key: str(value) for key, value in GRU_FILES.items()},
    'selected_lgbm_weight': selected_lgbm_weight,
    'selected_gru_weight': selected_gru_weight,
    'selected_metrics_pooled': {
        key: (
            int(selected[key])
            if key in ['observations', 'tp', 'fp', 'fn', 'tn']
            else float(selected[key])
        )
        for key in [
            'observations', 'mae', 'rmse', 'bias',
            'precision', 'recall', 'f1', 'f2',
            'tp', 'fp', 'fn', 'tn',
        ]
    },
    'minimum_period_f1': float(selected['minimum_period_f1']),
    'single_best_f1_pooled': float(single_best_f1),
    'ensemble_improves_single_best_f1': ensemble_improves_f1,
    'test_used_for_selection': True,
    'test_label': 'development_validation',
    'independent_test_remaining': False,
}
(OUT / 'selection_all_7_9월.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

pooled_weights = weight_results.loc[weight_results['period'].eq('pooled_7_9월')]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for metric, label in [('f1', 'F1'), ('precision', '정밀도'), ('recall', '재현율')]:
    axes[0].plot(pooled_weights['gru_weight'], pooled_weights[metric], marker='o', label=label)
axes[0].axvline(selected_gru_weight, color='black', linestyle='--', label='선택 비율')
axes[0].set(xlabel='GRU 비중', ylabel='점수', title='7~9월 통합 피크경보 성능')
axes[0].legend()

axes[1].plot(pooled_weights['gru_weight'], pooled_weights['fp'], marker='o', label='오경보')
axes[1].plot(pooled_weights['gru_weight'], pooled_weights['fn'], marker='o', label='미탐지')
axes[1].axvline(selected_gru_weight, color='black', linestyle='--', label='선택 비율')
axes[1].set(xlabel='GRU 비중', ylabel='건수', title='7~9월 통합 오경보와 미탐지')
axes[1].legend()

fig.tight_layout()
fig.savefig(OUT / 'ensemble_weight_comparison_all_7_9월.png', dpi=160, bbox_inches='tight')
plt.show()

print('저장 완료:', OUT.resolve())


# 5. 영향요인과 오류 분석

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = PROJECT_ROOT

DATA_PATH = ROOT / 'dataset/okm_augumented_2021.csv'
PRED_PATHS = {
    '7월 검증': ROOT / 'results/team_submission_runtime/outputs/validation_predictions_7월.csv',
    '8~9월 개발 검증': ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv',
}
ENSEMBLE_PRED_PATH = (
    ROOT / 'results/ensemble_lgbm_gru_team_aligned/predictions_all_7_9월.csv'
)
OUT = ROOT / 'results/error_peak_condition_analysis_ensemble_fixed176'
FIG = OUT / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

PEAK_LIMIT = 176.0
WARNING_LIMIT = 164.5
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])

font_path = Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts/malgun.ttf'
if font_path.exists():
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False

if not DATA_PATH.exists():
    raise FileNotFoundError(f'원본 데이터가 없습니다: {DATA_PATH}')
for period, prediction_path in PRED_PATHS.items():
    if not prediction_path.exists():
        raise FileNotFoundError(f'{period} 예측 파일이 없습니다: {prediction_path}')
if not ENSEMBLE_PRED_PATH.exists():
    raise FileNotFoundError(f'앙상블 예측 파일이 없습니다: {ENSEMBLE_PRED_PATH}')

print('원본 데이터:', DATA_PATH.resolve())
print('최종 수치예측:', {k: str(v.resolve()) for k, v in PRED_PATHS.items()})
print('최종 피크경보:', ENSEMBLE_PRED_PATH.resolve())
print('결과 저장:', OUT.resolve())


In [ ]:
raw = pd.read_csv(DATA_PATH)
raw['source_date'] = pd.to_datetime(raw['날짜'].astype(str), format='%Y%m%d')

day_size = raw.groupby('source_date', sort=False).size()
if not day_size.eq(24).all():
    raise ValueError('모든 날짜가 24개 시간 행으로 구성되지 않았습니다.')

raw['row_hour'] = raw.groupby('source_date', sort=False).cumcount()
raw['original_hour'] = pd.to_numeric(raw['시간'], errors='coerce')
raw['time_repaired'] = raw['original_hour'].ne(raw['row_hour'])
raw['hour'] = raw['row_hour']
raw['hour_start'] = raw['source_date'] + pd.to_timedelta(raw['hour'], unit='h')
raw['production'] = pd.to_numeric(raw['생산량'], errors='coerce')

if raw[['production', '15분', '30분', '45분', '60분']].isna().any().any():
    raise ValueError('생산량 또는 전력값에 결측치가 있습니다.')

raw['previous_production'] = raw['production'].shift(1)
raw.loc[raw['source_date'].ne(raw['source_date'].shift(1)), 'previous_production'] = 0.0
raw['production_change'] = raw['production'] - raw['previous_production']

positive_change = raw.loc[raw['production_change'].abs().gt(0), 'production_change'].abs()
RAPID_CHANGE_LIMIT = float(positive_change.quantile(0.75))

is_start = raw['production'].gt(0) & raw['previous_production'].le(0)
is_stop = raw['production'].le(0) & raw['previous_production'].gt(0)
raw['production_transition'] = np.select(
    [is_start, is_stop,
     raw['production_change'].ge(RAPID_CHANGE_LIMIT),
     raw['production_change'].le(-RAPID_CHANGE_LIMIT)],
    ['생산 시작', '생산 종료', '생산량 급증', '생산량 급감'],
    default='큰 변화 없음'
)

positive_production = raw.loc[raw['production'].gt(0), 'production']
PROD_Q1, PROD_Q2 = positive_production.quantile([1/3, 2/3]).astype(float)
raw['production_level'] = np.select(
    [raw['production'].le(0), raw['production'].le(PROD_Q1), raw['production'].le(PROD_Q2)],
    ['비생산', '낮은 생산량', '중간 생산량'],
    default='높은 생산량'
)

long_parts = []
for minute, power_col in zip([15, 30, 45, 60], ['15분', '30분', '45분', '60분']):
    part = raw[[
        'source_date', 'hour_start', 'production', 'previous_production',
        'production_change', 'production_transition', 'production_level',
        'time_repaired', '전기요금(계절)', '인건비'
    ]].copy()
    part['target_time'] = part['hour_start'] + pd.to_timedelta(minute, unit='m')
    part['actual_power_raw'] = pd.to_numeric(raw[power_col], errors='coerce')
    part['quarter_minute'] = minute
    long_parts.append(part)

actual = pd.concat(long_parts, ignore_index=True).sort_values('target_time').reset_index(drop=True)
if actual['target_time'].duplicated().any():
    raise ValueError('15분 단위 실제 시각이 중복되었습니다.')

actual['calendar_date'] = actual['target_time'].dt.normalize()
actual['month'] = actual['target_time'].dt.strftime('%Y-%m')
actual['hour'] = actual['target_time'].dt.hour
actual['time_band'] = pd.cut(
    actual['hour'], bins=[-1, 5, 11, 17, 23],
    labels=['새벽(0~5시)', '오전(6~11시)', '오후(12~17시)', '저녁(18~23시)']
).astype(str)
actual['weekend'] = actual['calendar_date'].dt.dayofweek.ge(5)
actual['holiday'] = actual['calendar_date'].isin(HOLIDAYS)
actual['day_type'] = np.select(
    [actual['holiday'], actual['weekend']], ['공휴일', '주말'], default='평일'
)

state_by_time = actual.set_index('target_time')['production'].gt(0)
state_now = actual['production'].gt(0).to_numpy()
state_w1 = state_by_time.reindex(actual['target_time'] - pd.Timedelta(days=7)).to_numpy()
state_w2 = state_by_time.reindex(actual['target_time'] - pd.Timedelta(days=14)).to_numpy()
valid_w1 = pd.notna(state_w1)
valid_w2 = pd.notna(state_w2)
mismatch_w1 = valid_w1 & (state_now != state_w1)
mismatch_w2 = valid_w2 & (state_now != state_w2)
actual['weekly_state_match'] = np.select(
    [~valid_w1 | ~valid_w2,
     mismatch_w1 & mismatch_w2,
     mismatch_w1,
     mismatch_w2],
    ['과거 부족', '1·2주 모두 불일치', '1주만 불일치', '2주만 불일치'],
    default='1·2주 모두 일치'
)
actual['operation_state'] = np.where(actual['production'].gt(0), '가동 중', '비가동')
actual['power_level'] = pd.cut(
    actual['actual_power_raw'],
    bins=[-np.inf, 100, 150, PEAK_LIMIT, 200, np.inf],
    right=False,
    labels=['낮음(100 미만)', '보통(100~149)', '피크 근처(150~175)',
            '피크(176~199)', '큰 피크(200 이상)']
).astype(str)
actual['actual_peak'] = actual['actual_power_raw'].ge(PEAK_LIMIT)
actual['peak_excess'] = (actual['actual_power_raw'] - PEAK_LIMIT).clip(lower=0)
actual['suspect_date'] = actual['source_date'].isin(SUSPECT_DATES)

print(f'원본 시간 행: {len(raw):,} | 15분 관측: {len(actual):,}')
print('시간 복구 날짜:', raw.loc[raw['time_repaired'], 'source_date'].dt.strftime('%Y-%m-%d').unique().tolist())
print(f'생산량 구간 경계: {PROD_Q1:.3f}, {PROD_Q2:.3f}')
print(f'급격한 생산량 변화 기준: |변화량| >= {RAPID_CHANGE_LIMIT:.3f}')


In [ ]:
required_columns = [
    '기준시각', '예측대상시각', '예측칸', '실제전력', 'curve_final'
]
pred_parts = []
for period, prediction_path in PRED_PATHS.items():
    part = pd.read_csv(prediction_path, usecols=required_columns)
    part.insert(0, 'evaluation_period', period)
    pred_parts.append(part)
pred = pd.concat(pred_parts, ignore_index=True)

prediction_keys = ['기준시각', '예측대상시각', '예측칸']
peak_required_columns = prediction_keys + [
    'actual', 'lgbm_q075', 'gru_q080', 'ensemble', 'threshold'
]
peak_pred = pd.read_csv(ENSEMBLE_PRED_PATH, usecols=peak_required_columns)
if peak_pred.duplicated(prediction_keys).any():
    raise ValueError('앙상블 예측 파일의 예측 키가 중복되었습니다.')
pred = pred.merge(peak_pred, on=prediction_keys, how='left', validate='one_to_one')
if pred['ensemble'].isna().any():
    raise ValueError('일부 수치예측과 앙상블 예측을 연결하지 못했습니다.')

ensemble_difference = (
    pred['ensemble'] - (0.65 * pred['lgbm_q075'] + 0.35 * pred['gru_q080'])
).abs().max()
if ensemble_difference > 1e-6:
    raise ValueError(f'앙상블 비율이 65:35와 다릅니다. 최대 차이={ensemble_difference}')
if not pd.to_numeric(pred['threshold'], errors='coerce').eq(PEAK_LIMIT).all():
    raise ValueError('앙상블 예측 파일의 경보 기준이 176과 다릅니다.')

pred['origin'] = pd.to_datetime(pred['기준시각'])
pred['target_time'] = pd.to_datetime(pred['예측대상시각'])
daily_pred = pred.loc[
    pred['origin'].dt.hour.eq(0) & pred['origin'].dt.minute.eq(0)
].copy()

if daily_pred['target_time'].duplicated().any():
    duplicated = daily_pred.loc[
        daily_pred['target_time'].duplicated(False), 'target_time'
    ].head()
    raise ValueError(f'일별 예측에서 대상시각이 중복되었습니다: {duplicated.tolist()}')

analysis = daily_pred.merge(actual, on='target_time', how='left', validate='one_to_one')
if analysis['actual_power_raw'].isna().any():
    raise ValueError('일부 예측시각을 원본 15분 데이터와 연결하지 못했습니다.')

actual_difference = (
    pd.to_numeric(analysis['실제전력'], errors='coerce')
    - analysis['actual_power_raw']
).abs().max()
if actual_difference > 1e-6:
    raise ValueError(
        f'예측 파일과 원본의 실제 전력값이 다릅니다. 최대 차이={actual_difference}'
    )
ensemble_actual_difference = (
    pd.to_numeric(analysis['actual'], errors='coerce')
    - analysis['actual_power_raw']
).abs().max()
if ensemble_actual_difference > 1e-6:
    raise ValueError(
        f'앙상블 파일과 원본의 실제 전력값이 다릅니다. 최대 차이={ensemble_actual_difference}'
    )

analysis['actual_power'] = pd.to_numeric(analysis['실제전력'], errors='coerce')
analysis['point_prediction'] = pd.to_numeric(analysis['curve_final'], errors='coerce')
analysis['lgbm_peak_prediction'] = pd.to_numeric(analysis['lgbm_q075'], errors='coerce')
analysis['gru_peak_prediction'] = pd.to_numeric(analysis['gru_q080'], errors='coerce')
analysis['peak_prediction'] = pd.to_numeric(analysis['ensemble'], errors='coerce')
analysis['point_error'] = analysis['point_prediction'] - analysis['actual_power']
analysis['absolute_error'] = analysis['point_error'].abs()
analysis['squared_error'] = analysis['point_error'].pow(2)
analysis['lead_hour'] = (
    analysis['target_time'] - analysis['origin']
).dt.total_seconds() / 3600
analysis['horizon_band'] = pd.cut(
    analysis['lead_hour'], bins=[0, 6, 12, 18, 24.001], include_lowest=True,
    labels=['0~6시간', '6~12시간', '12~18시간', '18~24시간']
).astype(str)
analysis['predicted_alarm'] = analysis['peak_prediction'].ge(PEAK_LIMIT)
analysis['predicted_attention'] = analysis['peak_prediction'].ge(WARNING_LIMIT)
analysis['operation_level'] = np.select(
    [analysis['predicted_alarm'], analysis['predicted_attention']],
    ['경보', '주의'], default='정상'
)
analysis['actual_peak'] = analysis['actual_power'].ge(PEAK_LIMIT)
analysis['alarm_result'] = np.select(
    [
        analysis['actual_peak'] & analysis['predicted_alarm'],
        analysis['actual_peak'] & ~analysis['predicted_alarm'],
        ~analysis['actual_peak'] & analysis['predicted_alarm'],
    ],
    ['정확한 경보', '미탐지', '오경보'],
    default='정확한 정상 판단',
)

LARGE_ERROR_LIMIT = float(analysis['absolute_error'].quantile(0.90))
analysis['large_error'] = analysis['absolute_error'].ge(LARGE_ERROR_LIMIT)

print(f'전체 롤링 예측: {len(pred):,}칸')
print(f'일별 00시 기준 평가: {analysis["origin"].nunique():,}일, {len(analysis):,}칸')
print(f'평가기간: {analysis["target_time"].min()} ~ {analysis["target_time"].max()}')
print(f'전체 MAE: {analysis["absolute_error"].mean():.3f}')
print(f'큰 오차 기준(상위 10%): {LARGE_ERROR_LIMIT:.3f}')
print(f'실제 피크 비율: {analysis["actual_peak"].mean():.2%}')
display(analysis['alarm_result'].value_counts().rename_axis('판정').to_frame('건수'))


In [ ]:
def error_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        rows.append({
            '조건값': str(value),
            '표본수': len(group),
            'MAE': group['absolute_error'].mean(),
            'RMSE': np.sqrt(group['squared_error'].mean()),
            '평균오차_예측-실제': group['point_error'].mean(),
            '큰오차수': int(group['large_error'].sum()),
            '큰오차비율': group['large_error'].mean(),
            '실제평균전력': group['actual_power'].mean(),
        })
    return pd.DataFrame(rows)

error_conditions = {
    '평가기간': 'evaluation_period',
    '월': 'month',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '실제전력수준': 'power_level',
    '주간생산상태': 'weekly_state_match',
    '예측거리': 'horizon_band',
}

error_tables = {}
for name, column in error_conditions.items():
    table = error_summary(analysis, column)
    error_tables[name] = table
    table.to_csv(OUT / f'error_by_{column}.csv', index=False, encoding='utf-8-sig')

top_errors = analysis.nlargest(30, 'absolute_error')[[
    'evaluation_period', 'origin', 'target_time', 'actual_power',
    'point_prediction', 'point_error', 'absolute_error', 'production',
    'production_change', 'production_transition', 'production_level',
    'operation_state', 'power_level', 'day_type', 'weekly_state_match',
    'horizon_band'
]]
top_errors.to_csv(OUT / 'top_error_cases.csv', index=False, encoding='utf-8-sig')

for name, table in error_tables.items():
    print()
    print(f'[{name}별 수치예측 오차]')
    display(table.round(3))


In [ ]:
plot_names = ['평가기간', '시간대', '요일유형', '가동상태', '생산변화', '실제전력수준']
fig, axes = plt.subplots(3, 2, figsize=(15, 14))

for ax, name in zip(axes.flat, plot_names):
    table = error_tables[name]
    x = np.arange(len(table))
    bars = ax.bar(x, table['MAE'], color='#3B82F6')
    ax.axhline(
        analysis['absolute_error'].mean(), color='#DC2626', linestyle='--',
        label=f'전체 MAE {analysis["absolute_error"].mean():.2f}'
    )
    ax.set_title(f'{name}별 수치예측 MAE')
    ax.set_ylabel('MAE')
    ax.set_xticks(x, table['조건값'], rotation=25, ha='right')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(fontsize=8)
    for bar, count in zip(bars, table['표본수']):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(), f'n={count:,}',
            ha='center', va='bottom', fontsize=7
        )

fig.suptitle('어떤 조건에서 전력 수치예측 오차가 커지는가', fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(FIG / 'error_conditions.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
peak_data = actual.loc[~actual['suspect_date']].copy()

def peak_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        peak_group = group.loc[group['actual_peak']]
        rows.append({
            '조건값': str(value),
            '표본수': len(group),
            '피크발생수': int(group['actual_peak'].sum()),
            '피크발생률': group['actual_peak'].mean(),
            '평균전력': group['actual_power_raw'].mean(),
            '최대전력': group['actual_power_raw'].max(),
            '총피크초과량': group['peak_excess'].sum(),
            '피크시평균초과량': peak_group['peak_excess'].mean() if len(peak_group) else 0.0,
        })
    return pd.DataFrame(rows)

peak_conditions = {
    '월': 'month',
    '시간': 'hour',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '주간생산상태': 'weekly_state_match',
}

peak_tables = {}
for name, column in peak_conditions.items():
    table = peak_summary(peak_data, column)
    peak_tables[name] = table
    table.to_csv(OUT / f'peak_by_{column}.csv', index=False, encoding='utf-8-sig')

ordered_peak = peak_data.sort_values('target_time').copy()
previous_peak = ordered_peak['actual_peak'].shift(fill_value=False)
time_gap = ordered_peak['target_time'].diff().ne(pd.Timedelta(minutes=15))
event_start = ordered_peak['actual_peak'] & (~previous_peak | time_gap)
ordered_peak['peak_event_id'] = event_start.cumsum()
peak_events = (
    ordered_peak.loc[ordered_peak['actual_peak']]
    .groupby('peak_event_id')
    .agg(
        시작시각=('target_time', 'min'), 종료시각=('target_time', 'max'),
        지속칸=('target_time', 'size'), 최대전력=('actual_power_raw', 'max'),
        총초과량=('peak_excess', 'sum')
    ).reset_index(drop=True)
)
peak_events['지속시간_분'] = peak_events['지속칸'] * 15
peak_events.to_csv(OUT / 'peak_events.csv', index=False, encoding='utf-8-sig')

top_peaks = peak_data.nlargest(30, 'actual_power_raw')[[
    'target_time', 'actual_power_raw', 'peak_excess', 'production',
    'production_change', 'production_transition', 'production_level',
    'day_type', 'weekly_state_match'
]]
top_peaks.to_csv(OUT / 'top_peak_cases.csv', index=False, encoding='utf-8-sig')

print(f'분석 관측치: {len(peak_data):,}개')
print(f'피크 관측치: {peak_data["actual_peak"].sum():,}개 ({peak_data["actual_peak"].mean():.2%})')
print(f'연속 피크 사건: {len(peak_events):,}건')
display(peak_events['지속시간_분'].describe().to_frame('피크 지속시간(분)').round(2))

for name in ['월', '시간대', '요일유형', '가동상태', '생산량수준', '생산변화', '주간생산상태']:
    print()
    print(f'[{name}별 실제 피크]')
    display(peak_tables[name].round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

hour_table = peak_tables['시간'].sort_values('조건값', key=lambda s: pd.to_numeric(s))
axes[0, 0].plot(pd.to_numeric(hour_table['조건값']), hour_table['피크발생률'] * 100,
                marker='o', color='#DC2626')
axes[0, 0].set_title('시간별 실제 피크 발생률')
axes[0, 0].set_xlabel('시간')
axes[0, 0].set_ylabel('피크 발생률(%)')
axes[0, 0].set_xticks(range(0, 24, 2))

for ax, name in zip(axes.flat[1:], ['요일유형', '생산량수준', '생산변화']):
    table = peak_tables[name]
    x = np.arange(len(table))
    bars = ax.bar(x, table['피크발생률'] * 100, color='#F97316')
    ax.set_title(f'{name}별 실제 피크 발생률')
    ax.set_ylabel('피크 발생률(%)')
    ax.set_xticks(x, table['조건값'], rotation=25, ha='right')
    for bar, count in zip(bars, table['표본수']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'n={count:,}',
                ha='center', va='bottom', fontsize=7)

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('최대피크는 어떤 조건에서 발생하는가', fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(FIG / 'peak_conditions.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
def alarm_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        actual_positive = group['actual_peak'].to_numpy(bool)
        predicted_positive = group['predicted_alarm'].to_numpy(bool)
        tp = int(np.sum(actual_positive & predicted_positive))
        fp = int(np.sum(~actual_positive & predicted_positive))
        fn = int(np.sum(actual_positive & ~predicted_positive))
        tn = int(np.sum(~actual_positive & ~predicted_positive))
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
        rows.append({
            '조건값': str(value), '표본수': len(group),
            '실제피크수': tp + fn, '예측경보수': tp + fp,
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            '정밀도': precision, '재현율': recall, 'F1': f1,
            '미탐지율': fn / (tp + fn) if tp + fn else np.nan,
            '오경보율': fp / (fp + tn) if fp + tn else np.nan,
        })
    return pd.DataFrame(rows)

alarm_conditions = {
    '평가기간': 'evaluation_period',
    '월': 'month',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '실제전력수준': 'power_level',
    '주간생산상태': 'weekly_state_match',
    '예측거리': 'horizon_band',
}

alarm_tables = {}
for name, column in alarm_conditions.items():
    table = alarm_summary(analysis, column)
    alarm_tables[name] = table
    table.to_csv(OUT / f'alarm_by_{column}.csv', index=False, encoding='utf-8-sig')

case_columns = [
    'evaluation_period', 'origin', 'target_time', 'actual_power',
    'point_prediction', 'lgbm_peak_prediction', 'gru_peak_prediction',
    'peak_prediction', 'alarm_result', 'production',
    'production_change', 'production_transition', 'production_level',
    'operation_state', 'power_level', 'day_type', 'weekly_state_match',
    'horizon_band'
]
false_alarms = analysis.loc[analysis['alarm_result'].eq('오경보'), case_columns].copy()
missed_peaks = analysis.loc[analysis['alarm_result'].eq('미탐지'), case_columns].copy()
false_alarms['경보기준초과량'] = false_alarms['peak_prediction'] - PEAK_LIMIT
missed_peaks['실제피크초과량'] = missed_peaks['actual_power'] - PEAK_LIMIT
false_alarms.sort_values('경보기준초과량', ascending=False).to_csv(
    OUT / 'false_alarm_cases.csv', index=False, encoding='utf-8-sig'
)
missed_peaks.sort_values('실제피크초과량', ascending=False).to_csv(
    OUT / 'missed_peak_cases.csv', index=False, encoding='utf-8-sig'
)

for name, table in alarm_tables.items():
    print()
    print(f'[{name}별 피크경보 성능]')
    display(table.round(3))


In [ ]:
def binary_metrics(frame, prediction_column):
    actual_positive = frame['actual_peak'].to_numpy(bool)
    predicted_positive = frame[prediction_column].to_numpy(bool)
    tp = int(np.sum(actual_positive & predicted_positive))
    fp = int(np.sum(~actual_positive & predicted_positive))
    fn = int(np.sum(actual_positive & ~predicted_positive))
    tn = int(np.sum(~actual_positive & ~predicted_positive))
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    return {
        '표본수': len(frame), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        '정밀도': precision, '재현율': recall, 'F1': f1,
        '미탐지율': fn / (tp + fn) if tp + fn else np.nan,
    }


rolling_policy = pred.copy()
rolling_policy['actual_power'] = pd.to_numeric(rolling_policy['actual'], errors='coerce')
rolling_policy['peak_prediction'] = pd.to_numeric(rolling_policy['ensemble'], errors='coerce')
rolling_policy['actual_peak'] = rolling_policy['actual_power'].ge(PEAK_LIMIT)
rolling_policy['alarm_only'] = rolling_policy['peak_prediction'].ge(PEAK_LIMIT)
rolling_policy['alarm_or_warning'] = rolling_policy['peak_prediction'].ge(WARNING_LIMIT)
rolling_policy['operation_level'] = np.select(
    [rolling_policy['alarm_only'], rolling_policy['alarm_or_warning']],
    ['경보', '주의'], default='정상'
)

july = rolling_policy.loc[rolling_policy['evaluation_period'].eq('7월 검증')]
aug_sep = rolling_policy.loc[rolling_policy['evaluation_period'].eq('8~9월 개발 검증')]

july_warning_check = pd.DataFrame([{
    '주의기준': WARNING_LIMIT,
    **binary_metrics(july, 'alarm_or_warning'),
}])
july_warning_check.to_csv(
    OUT / 'warning_threshold_july_check.csv', index=False, encoding='utf-8-sig'
)

warning_policy_full = pd.DataFrame([
    {'운영방식': '경보만(176 이상)', **binary_metrics(aug_sep, 'alarm_only')},
    {'운영방식': '경보+주의(164.5 이상)', **binary_metrics(aug_sep, 'alarm_or_warning')},
])
warning_policy_full.to_csv(
    OUT / 'warning_policy_8_9_full_windows.csv', index=False, encoding='utf-8-sig'
)

daily_policy = pd.DataFrame([
    {'평가기간': '7~9월', '운영방식': '경보만(176 이상)', **binary_metrics(analysis, 'predicted_alarm')},
    {'평가기간': '7~9월', '운영방식': '경보+주의(164.5 이상)', **binary_metrics(analysis, 'predicted_attention')},
    {'평가기간': '8~9월', '운영방식': '경보만(176 이상)', **binary_metrics(
        analysis.loc[analysis['evaluation_period'].eq('8~9월 개발 검증')], 'predicted_alarm'
    )},
    {'평가기간': '8~9월', '운영방식': '경보+주의(164.5 이상)', **binary_metrics(
        analysis.loc[analysis['evaluation_period'].eq('8~9월 개발 검증')], 'predicted_attention'
    )},
])
daily_policy.to_csv(
    OUT / 'warning_policy_daily_00.csv', index=False, encoding='utf-8-sig'
)

return_week = aug_sep.loc[
    aug_sep['target_time'].between('2021-08-09', '2021-08-13 23:59:59')
]
return_week_policy = pd.DataFrame([
    {'운영방식': '경보만(176 이상)', **binary_metrics(return_week, 'alarm_only')},
    {'운영방식': '경보+주의(164.5 이상)', **binary_metrics(return_week, 'alarm_or_warning')},
])
return_week_policy.to_csv(
    OUT / 'warning_policy_return_week_full_windows.csv', index=False, encoding='utf-8-sig'
)

print('[7월 주의 기준 확인: 재현율 0.95 이상]')
display(july_warning_check.round(4))
print('[8~9월 모든 예측창: 경보만 / 경보+주의]')
display(warning_policy_full.round(4))
print('[매일 00시 예측: 경보만 / 경보+주의]')
display(daily_policy.round(4))
print('[8월 9~13일 복귀 주, 모든 예측창]')
display(return_week_policy.round(4))


In [ ]:
representative_rows = []

if len(analysis):
    row = analysis.loc[analysis['absolute_error'].idxmax()]
    representative_rows.append(('가장 큰 수치예측 오차', row))
if len(missed_peaks):
    row = analysis.loc[missed_peaks['actual_power'].idxmax()]
    representative_rows.append(('가장 큰 실제 피크 미탐지', row))
if len(false_alarms):
    row = analysis.loc[false_alarms['peak_prediction'].idxmax()]
    representative_rows.append(('가장 높은 예측값의 오경보', row))

representative_cases = pd.DataFrame([
    {
        '사례': label,
        '평가기간': row['evaluation_period'],
        '기준시각': row['origin'],
        '예측대상시각': row['target_time'],
        '실제전력': row['actual_power'],
        '수치예측': row['point_prediction'],
        'LightGBM경보예측': row['lgbm_peak_prediction'],
        'GRU경보예측': row['gru_peak_prediction'],
        '앙상블경보예측': row['peak_prediction'],
        '경보결과': row['alarm_result'],
        '가동상태': row['operation_state'],
        '생산변화': row['production_transition'],
    }
    for label, row in representative_rows
])
representative_cases.to_csv(
    OUT / 'representative_error_cases.csv', index=False, encoding='utf-8-sig'
)
display(representative_cases.round(3))

if representative_rows:
    fig, axes = plt.subplots(
        len(representative_rows), 1,
        figsize=(14, 4.2 * len(representative_rows)),
        squeeze=False,
    )
    for ax, (label, row) in zip(axes.flat, representative_rows):
        center = row['target_time']
        window = analysis.loc[
            analysis['target_time'].between(
                center - pd.Timedelta(hours=3),
                center + pd.Timedelta(hours=3),
            )
        ].sort_values('target_time')
        ax.plot(window['target_time'], window['actual_power'], label='실제 전력', linewidth=2)
        ax.plot(window['target_time'], window['point_prediction'], label='수치예측', linewidth=1.5)
        ax.plot(window['target_time'], window['peak_prediction'], label='앙상블 경보 예측', linewidth=1.2)
        ax.axhline(PEAK_LIMIT, color='#DC2626', linestyle='--', label='피크 기준 176')
        ax.axvline(center, color='black', linestyle=':', label='대표 오류 시점')
        ax.set_title(
            f'{label}: {center} | 실제 {row["actual_power"]:.1f}, '
            f'수치예측 {row["point_prediction"]:.1f}, '
            f'앙상블 경보 예측 {row["peak_prediction"]:.1f}'
        )
        ax.set_ylabel('전력')
        ax.grid(alpha=0.25)
        ax.legend(ncol=5, fontsize=8)
    axes.flat[-1].set_xlabel('예측 대상 시각')
    fig.tight_layout()
    fig.savefig(FIG / 'representative_error_cases.png', dpi=160, bbox_inches='tight')
    plt.show()


In [ ]:
finding_rows = []
for category, table in error_tables.items():
    eligible = table.loc[table['표본수'].ge(50)].copy()
    if len(eligible):
        best = eligible.loc[eligible['MAE'].idxmin()]
        worst = eligible.loc[eligible['MAE'].idxmax()]
        finding_rows.extend([
            {
                '분석': '수치예측', '분류': category, '구분': '오차가 작은 조건',
                '조건': best['조건값'], '표본수': int(best['표본수']),
                '지표': 'MAE', '값': float(best['MAE']),
            },
            {
                '분석': '수치예측', '분류': category, '구분': '오차가 큰 조건',
                '조건': worst['조건값'], '표본수': int(worst['표본수']),
                '지표': 'MAE', '값': float(worst['MAE']),
            },
        ])

for category, table in alarm_tables.items():
    eligible = table.loc[
        table['표본수'].ge(50) & table['실제피크수'].ge(10) & table['F1'].notna()
    ].copy()
    if len(eligible):
        best = eligible.loc[eligible['F1'].idxmax()]
        worst = eligible.loc[eligible['F1'].idxmin()]
        finding_rows.extend([
            {
                '분석': '피크경보', '분류': category, '구분': '경보가 잘 맞는 조건',
                '조건': best['조건값'], '표본수': int(best['표본수']),
                '지표': 'F1', '값': float(best['F1']),
            },
            {
                '분석': '피크경보', '분류': category, '구분': '경보가 잘 맞지 않는 조건',
                '조건': worst['조건값'], '표본수': int(worst['표본수']),
                '지표': 'F1', '값': float(worst['F1']),
            },
        ])

for category, table in peak_tables.items():
    eligible = table.loc[table['표본수'].ge(100)].copy()
    if len(eligible):
        highest = eligible.loc[eligible['피크발생률'].idxmax()]
        finding_rows.append({
            '분석': '실제피크', '분류': category, '구분': '피크가 자주 발생한 조건',
            '조건': highest['조건값'], '표본수': int(highest['표본수']),
            '지표': '피크발생률', '값': float(highest['피크발생률']),
        })

key_findings = pd.DataFrame(finding_rows)
key_findings.to_csv(OUT / 'chapter3_key_findings.csv', index=False, encoding='utf-8-sig')

overall_tp = int((analysis['actual_peak'] & analysis['predicted_alarm']).sum())
overall_fp = int((~analysis['actual_peak'] & analysis['predicted_alarm']).sum())
overall_fn = int((analysis['actual_peak'] & ~analysis['predicted_alarm']).sum())
overall_tn = int((~analysis['actual_peak'] & ~analysis['predicted_alarm']).sum())
overall_precision = overall_tp / (overall_tp + overall_fp) if overall_tp + overall_fp else np.nan
overall_recall = overall_tp / (overall_tp + overall_fn) if overall_tp + overall_fn else np.nan
overall_f1 = (
    2 * overall_precision * overall_recall / (overall_precision + overall_recall)
    if overall_precision + overall_recall else np.nan
)

summary = {
    'evaluation_rule': '7~9월 중 매일 00시 기준 향후 24시간 예측만 사용',
    'evaluation_start': str(analysis['target_time'].min()),
    'evaluation_end': str(analysis['target_time'].max()),
    'evaluation_days': int(analysis['origin'].nunique()),
    'evaluation_cells': int(len(analysis)),
    'point_model': 'curve_final',
    'peak_model': '0.65 * LightGBM improved_Q1_q075 + 0.35 * GRU enhanced47_q080',
    'peak_lgbm_weight': 0.65,
    'peak_gru_weight': 0.35,
    'peak_limit': float(PEAK_LIMIT),
    'warning_limit': float(WARNING_LIMIT),
    'warning_rule': '7월에서 재현율 0.95 이상을 만족하도록 정한 뒤 8~9월에 고정 적용',
    'point_mae': float(analysis['absolute_error'].mean()),
    'point_rmse': float(np.sqrt(analysis['squared_error'].mean())),
    'point_bias': float(analysis['point_error'].mean()),
    'large_error_p90': float(LARGE_ERROR_LIMIT),
    'alarm_precision': float(overall_precision),
    'alarm_recall': float(overall_recall),
    'alarm_f1': float(overall_f1),
    'true_positive': overall_tp,
    'false_alarm': overall_fp,
    'missed_peak': overall_fn,
    'true_negative': overall_tn,
    'raw_peak_cells_excluding_suspect_dates': int(peak_data['actual_peak'].sum()),
    'raw_peak_rate_excluding_suspect_dates': float(peak_data['actual_peak'].mean()),
    'peak_event_count': int(len(peak_events)),
    'production_low_boundary': float(PROD_Q1),
    'production_high_boundary': float(PROD_Q2),
    'rapid_change_limit': float(RAPID_CHANGE_LIMIT),
    'independent_test_remaining': False,
    'limitations': [
        '조건별 차이는 함께 나타난 관계이며 직접적인 원인을 증명하지 않는다.',
        '7~9월은 모델 선택에 사용했으므로 독립 테스트가 아닌 개발 검증 결과다.',
        '설비별 상태와 제품 종류가 없어 해당 조건은 분석하지 못한다.',
        '중복 제거를 위해 매일 00시 예측만 사용하므로 2장의 전체 예측칸 점수와 다를 수 있다.',
    ],
}
(OUT / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
pd.DataFrame([{
    '평가시작': summary['evaluation_start'],
    '평가종료': summary['evaluation_end'],
    '평가일수': summary['evaluation_days'],
    '평가칸수': summary['evaluation_cells'],
    '수치예측_MAE': summary['point_mae'],
    '수치예측_RMSE': summary['point_rmse'],
    '평균오차_예측-실제': summary['point_bias'],
    '정밀도': summary['alarm_precision'],
    '재현율': summary['alarm_recall'],
    'F1': summary['alarm_f1'],
    '오경보': summary['false_alarm'],
    '미탐지': summary['missed_peak'],
}]).to_csv(OUT / 'chapter3_overall_summary.csv', index=False, encoding='utf-8-sig')

print('[조건별 핵심 결과]')
display(key_findings.round(4))
print()
print('[3장 전체 평가 요약]')
display(pd.DataFrame([summary]).drop(columns='limitations').round(4))
print()
print('저장 완료:', OUT.resolve())


# 6. 7월과 8월 자원 최적화

생산계획을 바꿨을 때의 전력은 앞에서 학습한 LightGBM 수치 예측 모델로, 피크 경보는 최종 경보 모델인 앙상블(LightGBM 개선 Q1 65% + GRU q=0.80 35%, 기준 176)로 다시 계산합니다. 7월 13일과 15일은 시간 오류일이므로 7월 계산에서 제외합니다.

In [ ]:
# 반사실 예측 클래스: 별도 cf_predict.py 없이 실행
PEAK = 176.0
SAFE_W = 0.1
WEEKS = pd.date_range('2021-05-31', '2021-09-13', freq='7D')
ENS_W_LGBM, ENS_W_GRU = 0.65, 0.35   # 최종 경보 모델: LightGBM 개선 Q1 65% + GRU(q=0.80) 35%


def fit_gru_model(x_train, target_train, objective, alpha, cell_weight):
    """aw_fit_predict와 같은 학습(같은 시드·같은 순서)을 하되, 예측 대신 학습된 모델과 정규화 값을 돌려준다.
    바뀐 생산계획의 입력으로 GRU를 다시 계산하기 위한 것이다."""
    assert objective == 'quantile'
    cell_weight = cell_weight.astype(np.float32)
    active = cell_weight > 0
    x_mean = x_train.mean(axis=(0, 1), keepdims=True)
    x_std = x_train.std(axis=(0, 1), keepdims=True)
    x_std[x_std < 1e-6] = 1.0
    target_mean = float(target_train[active].mean())
    target_std = float(target_train[active].std())
    if target_std < 1e-6:
        target_std = 1.0
    x_train = ((x_train - x_mean) / x_std).astype(np.float32)
    target_train = ((target_train - target_mean) / target_std).astype(np.float32)
    np.random.seed(AW_SEED)
    torch.manual_seed(AW_SEED)
    generator = torch.Generator().manual_seed(AW_SEED)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(target_train), torch.from_numpy(cell_weight)),
        batch_size=AW_BATCH, shuffle=True, generator=generator,
    )
    model = AWGRU(x_train.shape[-1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=AW_LR, weight_decay=AW_WD)
    for _ in range(AW_EPOCHS):
        model.train()
        for batch_x, batch_target, batch_weight in loader:
            optimizer.zero_grad()
            error = batch_target - model(batch_x)
            element = torch.maximum(alpha * error, (alpha - 1) * error)      # 분위수 손실
            loss = (element * batch_weight).sum() / batch_weight.sum().clamp_min(1.0)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    model.eval()
    return model, (x_mean, x_std, target_mean, target_std)


class CounterfactualPredictor:
    def __init__(self, verbose=True):
        t0 = time.time(); self.verbose = verbose
        self.W = W = build_windows()
        self.idx = {o: i for i, o in enumerate(W.meta.origin)}
        self.ip, self.i1, self.i2 = (SEQ.index(n) for n in ("planned_production", "w1_production", "w2_production"))
        self.P_hist = W.hi.P.copy(); self.P_hist.setflags(write=False)           # 원본 시간별 생산량(읽기 전용: 과거는 바꾸지 않는다)
        # 3단 규칙의 시간순 프로파일(전 창의 정답 관측에서, 창 기준일 이전 값만 조회) — 변경하지 않는다
        sus = W.sus
        self.prof = OOFProfile(observation_table([(dict(sequence=W.seq, y=W.y, suspect_target=sus), W.meta)]))
        # 이례적 휴무 구간(연속 3평일 이상 생산 0) — 안전장치
        dp = pd.Series(W.hi.P).groupby(W.hi.date_d).sum(); off = (dp <= 0) & (pd.DatetimeIndex(dp.index).dayofweek < 5); self.streaks = []; run = []
        for d, o in off.items():
            if o: run.append(d)
            else:
                if len(run) >= 3: self.streaks.append((run[0], run[-1]))
                run = []
        if len(run) >= 3: self.streaks.append((run[0], run[-1]))
        # 휴무일 프로파일(곡선): 기준일 이전 휴무일 15분 값의 중앙값
        hr = pd.read_csv(PREP / "hourly_review.csv", parse_dates=["origin"]); hr["date"] = hr.origin.dt.floor("D"); hr["hr"] = hr.origin.dt.hour
        dsum = hr.groupby("date")["생산량"].sum(); self.off_days = [d for d in dsum[dsum == 0].index]
        self.byday = {d: g.sort_values("hr")[["15분", "30분", "45분", "60분"]].to_numpy() for d, g in hr.groupby("date") if len(g) == 24}
        self._idle_cache = {}; self._models = {}; self._gru = {}
        self.EVSET = set(W.EV.tolist()); self.ev_origins = W.meta.origin.iloc[W.EV]
        self._walk = None
        # 학습 범위 점검용 분포(전체 학습 가능 창의 계획 피처)
        self.key_feats = ["planned_production", "st_day_total", "st_win_plan_total", "st_win_on_hours", "w1_production_difference", "w2_production_difference", "st_win_vs_w1"]
        trn = W.F[self.key_feats].to_numpy(); self.rng = {k: (np.nanmin(trn[:, j]), np.nanpercentile(trn[:, j], 0.1), np.nanpercentile(trn[:, j], 99.9), np.nanmax(trn[:, j])) for j, k in enumerate(self.key_feats)}
        self.hourly_max = float(W.seq[:, :, self.ip].max()); self.hourly_p99 = float(np.percentile(W.seq[:, :, self.ip], 99.9))
        day_tot = pd.Series(W.hi.P).groupby(W.hi.date_d).sum(); self.day_max = float(day_tot.max()); self.day_p99 = float(day_tot.quantile(0.999))
        if verbose: print(f"[준비] 창 {W.N}개, 평가 창 {len(W.EV)}개 ({time.time() - t0:.0f}초)")

    # ------------------------------------------------------------ 주간 모델(변경 전과 똑같이 재학습, 캐시)
    def _week_start(self, origin):
        return WEEKS[0] + pd.Timedelta(days=7) * int((origin - WEEKS[0]) // pd.Timedelta(days=7))

    def _weights(self, S):
        W = self.W; known = np.zeros(W.N, bool)
        for a, b in self.streaks:
            if b + pd.Timedelta(days=1) <= S: known |= ((W.meta.origin < b + pd.Timedelta(days=1)) & (W.meta.target_end > a)).to_numpy()
        return np.repeat(np.where(known, SAFE_W, 1.0), 96)

    def _gate_cells(self):
        return (self.W.F["st_day_on"].to_numpy() == 0)                            # 휴무일 칸(원본 계획 기준; 학습 칸 선정용)

    def models(self, S):
        if S in self._models: return self._models[S]
        W = self.W; t0 = time.time(); tr_m = (W.meta.target_end <= S).to_numpy(); rows_tr = np.repeat(tr_m, 96); wts = self._weights(S)
        yl = W.y.reshape(-1).astype(np.float64)
        prm = dict(Q1_PARAMS); prm["alpha"] = 0.75; prm["random_state"] = 42; prm["objective"] = "quantile"
        m_q = lgb.LGBMRegressor(**prm).fit(W.F.loc[rows_tr, W.L13], yl[rows_tr], sample_weight=wts[rows_tr])
        rule = W.F["rule3_oof"].to_numpy(); on_rows = ~self._gate_cells()
        prm2 = dict(Q1_PARAMS); prm2["objective"] = "regression_l1"; r_tr = rows_tr & on_rows
        m_c = lgb.LGBMRegressor(**prm2).fit(W.F.loc[r_tr, W.L13], (yl - rule)[r_tr])
        self._models[S] = (m_q, m_c)
        if self.verbose: print(f"  [모델] {S.date()} 주 재학습 {time.time() - t0:.0f}초 (학습 창 {int(tr_m.sum())})")
        return self._models[S]

    # ------------------------------------------------------------ 최종 경보 모델의 GRU(q=0.80, 입력 47개): 주별 재학습, 캐시
    def gru_model(self, S):
        """주 S 시작 전에 정답이 확정된 창으로 학습한 GRU. 노트북 3절(02_aligned_gru)과 같은 방식이다."""
        if S in self._gru:
            return self._gru[S]
        W = self.W; t0 = time.time()
        train_rows = W.meta['target_end'].le(S).to_numpy()
        weights = np.ones_like(W.y3[train_rows], dtype=np.float32)
        weights *= np.where(aw_known_off_windows(S)[train_rows, None, None], 0.1, 1.0).astype(np.float32)
        self._gru[S] = fit_gru_model(AW_X47[train_rows], W.y3[train_rows], 'quantile', 0.80, weights)
        if self.verbose: print(f"  [GRU] {S.date()} 주 재학습 {time.time() - t0:.0f}초 (학습 창 {int(train_rows.sum())})")
        return self._gru[S]

    def gru_predict(self, S, F):
        """바뀐 계획의 피처표 F(96행)로 GRU 경보 예측 96칸을 계산한다."""
        model, (x_mean, x_std, target_mean, target_std) = self.gru_model(S)
        x = F[self.W.L13].to_numpy(np.float32).reshape(1, 24, 4 * len(self.W.L13))
        x = ((x - x_mean) / x_std).astype(np.float32)
        with torch.inference_mode():
            out = model(torch.from_numpy(x)).numpy()
        return np.clip(out * target_std + target_mean, 0, None).reshape(96).astype(np.float32)

    # ------------------------------------------------------------ 지난 주들의 예측(기준·확률 보정용, 변경 없음)
    def _walk_forward(self):
        if self._walk is None:
            W = self.W; pred = np.full((W.N, 96), np.nan, np.float32)
            for S in WEEKS:
                te_m = ((W.meta.origin >= S) & (W.meta.origin < S + pd.Timedelta(days=7))).to_numpy()
                if te_m.sum() == 0: continue
                m_q, _ = self.models(S); rows_te = np.repeat(te_m, 96)
                pred[te_m] = np.clip(m_q.predict(W.F.loc[rows_te, W.L13]).reshape(-1, 96), 0, None)
            self._walk = pred
        return self._walk

    def week_calibration(self, S):
        """그 주의 경보 기준, 주의 기준, Platt 보정(절편·기울기). 지난 4주 예측 결과로만 정한다."""
        if not hasattr(self, "_calib"): self._calib = {}
        if S not in self._calib: self._calib[S] = self._week_calibration(S)
        return self._calib[S]

    def _week_calibration(self, S):
        W = self.W; Q = self._walk_forward()
        hist = np.isfinite(Q[:, 0]) & (W.meta.target_end <= S).to_numpy() & (W.meta.origin >= WEEKS[0]).to_numpy() & (W.meta.origin >= S - pd.Timedelta(days=28)).to_numpy()
        thr_a = choose(W.y[hist], Q[hist], "R85"); thr_w = choose(W.y[hist], Q[hist], "R95")
        lr = LogisticRegression(C=1e4, max_iter=200).fit((Q[hist].reshape(-1, 1) - 176.0) / 20.0, (W.y[hist] >= PEAK).reshape(-1))
        return thr_a, thr_w, lr

    # ------------------------------------------------------------ 핵심: 한 창의 피처를 새 계획으로 다시 만든다
    def _features_for(self, i, plan):
        """창 i(기준시각)의 24시간 계획만 plan(길이 24)으로 바꾼 피처 표(96행). 과거 생산량(w1, w2)과 과거 실적은 그대로."""
        W = self.W; seq = W.seq[i:i + 1].astype(np.float32).copy(); rec = W.rec[i:i + 1]
        plan = np.asarray(plan, np.float32); w1 = seq[0, :, self.i1]; w2 = seq[0, :, self.i2]
        w1_before, w2_before = w1.copy(), w2.copy()
        seq[0, :, self.ip] = plan                                                        # 바꾸는 것은 이 한 줄뿐
        idx = {n: j for j, n in enumerate(SEQ)}
        seq[0, :, idx["w1_production_difference"]] = plan - w1; seq[0, :, idx["w2_production_difference"]] = plan - w2
        seq[0, :, idx["w1_production_state_mismatch"]] = ((plan > 0) != (w1 > 0)).astype(np.float32)
        seq[0, :, idx["w2_production_state_mismatch"]] = ((plan > 0) != (w2 > 0)).astype(np.float32)
        assert np.array_equal(seq[0, :, self.i1], w1_before) and np.array_equal(seq[0, :, self.i2], w2_before), "과거 1주·2주 전 생산량은 바꾸면 안 된다"
        F = long_features(dict(sequence=seq, recent=rec))
        r0 = int(W.r0[i]); P = self.P_hist.copy()
        before = P[:r0].copy(); P[r0:r0 + 24] = plan                                       # 목표 24시간만 교체(그 이전 실적은 그대로)
        assert np.array_equal(P[:r0], before) and np.array_equal(P[:r0], self.P_hist[:r0]), "기준시각 이전 생산 실적은 바꾸면 안 된다"
        st = state_features(W.hi, W.meta.origin.iloc[i:i + 1], seq, F, P=P)
        for c in ST_COLS: F[c] = st[c].to_numpy()
        meta1 = W.meta.iloc[i:i + 1].reset_index(drop=True)
        r, s = rule3_oof(F, meta1, self.prof)
        F["rule3_oof"] = r; F["rule_step"] = s; F["rule3_oof"] = F["rule3_oof"].fillna(F["w1_same"])
        return F

    def _idle_profile(self, i):
        W = self.W; D = pd.Timestamp(W.meta.origin.iloc[i]).floor("D")
        if D not in self._idle_cache:
            ds = [d for d in self.off_days if d < D and d in self.byday]
            self._idle_cache[D] = np.median(np.stack([self.byday[d] for d in ds]), axis=0) if len(ds) >= 3 else None
        h0 = pd.Timestamp(W.meta.origin.iloc[i]).hour
        return None if self._idle_cache[D] is None else self._idle_cache[D][(h0 + np.repeat(np.arange(24), 4)) % 24, np.tile(np.arange(4), 24)]

    def _outputs(self, i, F, S, calib):
        m_q, m_c = self.models(S); thr_a, thr_w, lr = calib
        q1 = np.clip(m_q.predict(F[self.W.L13]), 0, None).astype(np.float32)
        rule = F["rule3_oof"].to_numpy().astype(np.float32); gate = (F["st_day_on"].to_numpy() == 0)
        res = np.clip(m_c.predict(F[self.W.L13]) + rule, 0, None).astype(np.float32); idle = self._idle_profile(i)
        if idle is None: curve = np.where(gate, rule, res)
        else: curve = np.where(gate & np.isfinite(idle), idle, np.where(gate, rule, res))
        prob = lr.predict_proba((q1.reshape(-1, 1) - 176.0) / 20.0)[:, 1].astype(np.float32)
        stage = np.where(q1 >= thr_a, 2, np.where(q1 >= thr_w, 1, 0))
        return dict(curve_final=curve.astype(np.float32), q1_new=q1, prob_peak=prob, 경보단계=stage, rule3=rule, rule_step=F["rule_step"].to_numpy(), thr_alarm=thr_a, thr_watch=thr_w)

    # ------------------------------------------------------------ 적용 범위(학습 범위) 점검
    def check_applicability(self, plan, F_cf, base=None, F_base=None):
        msgs = []; level = "정상"
        plan = np.asarray(plan, float)
        if (plan < 0).any(): raise ValueError("생산계획에 음수가 있습니다.")
        def bump(lv):
            nonlocal level
            order = {"정상": 0, "경계": 1, "범위 밖": 2}
            if order[lv] > order[level]: level = lv
        if plan.max() > self.hourly_max: msgs.append(f"시간당 생산량 {plan.max():.0f}가 학습 자료 최대 {self.hourly_max:.0f}를 넘음"); bump("범위 밖")
        elif plan.max() > self.hourly_p99: msgs.append(f"시간당 생산량 {plan.max():.0f}가 학습 자료 상위 0.1%({self.hourly_p99:.0f})를 넘음"); bump("경계")
        tot = plan.sum()
        if tot > self.day_max: msgs.append(f"24시간 총생산량 {tot:.0f}가 학습 자료의 하루 최대 {self.day_max:.0f}를 넘음"); bump("범위 밖")
        elif tot > self.day_p99: msgs.append(f"24시간 총생산량 {tot:.0f}가 상위 0.1%({self.day_p99:.0f})를 넘음"); bump("경계")
        for k in self.key_feats:
            v = F_cf[k].to_numpy(); lo, p1, p99, hi = self.rng[k]
            if v.min() < lo or v.max() > hi: msgs.append(f"피처 {k}가 학습 범위 밖({v.min():.0f}~{v.max():.0f}, 학습 {lo:.0f}~{hi:.0f})"); bump("범위 밖")
            elif v.min() < p1 or v.max() > p99: msgs.append(f"피처 {k}가 학습 분포의 0.1~99.9% 밖"); bump("경계")
        if base is not None and F_base is not None:
            base = np.asarray(base, float)
            if (base.sum() > 0) != (plan.sum() > 0): msgs.append("하루 가동/휴무 상태가 바뀜(학습에서 드문 상황일 수 있음)"); bump("경계")
            ch = int(((base > 0) != (plan > 0)).sum())
            if ch >= 4: msgs.append(f"가동 시간 수가 {ch}시간 바뀜(가동↔정지 시간 변경)"); bump("경계")
            ds = int((F_base["rule_step"].to_numpy() != F_cf["rule_step"].to_numpy()).sum())
            if ds: msgs.append(f"3단 규칙 단계가 {ds}칸에서 바뀜(계획이 1·2주 전과 가동 상태가 달라짐)"); bump("경계")
            if abs(plan - base).max() > 0.5 * max(base.max(), 1): msgs.append(f"한 시간의 생산량이 기존 최대 시간 생산의 50% 넘게 바뀜(변화 {abs(plan - base).max():.0f})"); bump("경계")
        return dict(level=level, messages=msgs or ["학습 범위 안"])

    # ------------------------------------------------------------ 공개 함수
    def baseline_plan(self, origin_time):
        i = self._locate(origin_time); return self.W.seq[i, :, self.ip].astype(float)

    def _locate(self, origin_time):
        o = pd.Timestamp(origin_time)
        if o not in self.idx: raise ValueError(f"기준시각 {o}에 해당하는 창이 없습니다(정각, 학습 가능 창만).")
        i = self.idx[o]
        if i not in self.EVSET: raise ValueError("평가 창(7/1~9/14) 기준시각만 지원합니다. 그 이전 주는 경보 기준 이력이 없습니다.")
        return i

    def predict_counterfactual(self, origin_time, new_plan=None, plan_delta=None):
        """기준시각 origin_time 이후 24시간의 계획을 바꾸고 최종 출력을 다시 계산한다.
        new_plan  : 길이 24, 수정된 시간별 생산계획(기준시각부터 1시간 단위)
        plan_delta: 길이 24, 기존 계획 대비 변화량(new_plan = 기존 + delta). 둘 중 하나만 주거나, 둘 다 주면 서로 맞아야 한다.
        반환: dict(cells=96칸 표(변경 전 _base, 변경 후 _cf, 차이), summary=요약, applicability=적용 범위, plan=표)"""
        i = self._locate(origin_time); base = self.W.seq[i, :, self.ip].astype(np.float64)
        if new_plan is None and plan_delta is None: raise ValueError("new_plan 또는 plan_delta가 필요합니다.")
        if new_plan is not None:
            new_plan = np.asarray(new_plan, float)
            if plan_delta is not None and not np.allclose(new_plan - base, plan_delta): raise ValueError("new_plan과 plan_delta가 일치하지 않습니다.")
        else: new_plan = base + np.asarray(plan_delta, float)
        if new_plan.shape != (24,): raise ValueError("계획은 길이 24(기준시각 이후 24시간)여야 합니다.")
        S = self._week_start(pd.Timestamp(origin_time)); calib = self.week_calibration(S)
        F0 = self._features_for(i, base); F1 = self._features_for(i, new_plan)
        o0 = self._outputs(i, F0, S, calib); o1 = self._outputs(i, F1, S, calib)
        o = pd.Timestamp(origin_time); step = np.arange(96); tt = o + pd.to_timedelta(step // 4, "h") + pd.to_timedelta(15 * (step % 4 + 1), "m")
        cells = pd.DataFrame(dict(기준시각=o, 예측대상시각=tt, 예측칸=step, 실제전력=self.W.y[i]))
        for k in ("curve_final", "q1_new", "prob_peak", "경보단계", "rule3", "rule_step"):
            cells[k + "_base"] = o0[k]; cells[k + "_cf"] = o1[k]
        for k in ("curve_final", "q1_new", "prob_peak"): cells[k + "_diff"] = cells[k + "_cf"] - cells[k + "_base"]
        cells["경보단계_변화"] = cells["경보단계_cf"] - cells["경보단계_base"]
        ap = self.check_applicability(new_plan, F1, base, F0); ap_base = self.check_applicability(base, F0)
        ap["변경전_계획_범위"] = ap_base["level"]; ap["변경전_계획_메시지"] = ap_base["messages"]       # 변경 전 계획 자체가 이미 경계에 있으면 구분해서 본다
        if ap["level"] != "정상" and ap_base["level"] == ap["level"] and set(ap["messages"]) <= set(ap_base["messages"]) | {m for m in ap["messages"] if "3단 규칙 단계" in m or "바뀜" in m}:
            ap["메모"] = "경고의 일부는 변경 전 계획에서도 이미 나타남"
        a_b, a_c = (cells["경보단계_base"] == 2), (cells["경보단계_cf"] == 2); pk = cells["실제전력"] >= PEAK
        ex_b = np.clip(cells["curve_final_base"] - PEAK, 0, None).sum() / 4; ex_c = np.clip(cells["curve_final_cf"] - PEAK, 0, None).sum() / 4
        summ = dict(기준시각=str(o), 경보기준=o0["thr_alarm"], 주의기준=o0["thr_watch"], 총생산_변경전=float(base.sum()), 총생산_변경후=float(new_plan.sum()), 최대시간생산_변경전=float(base.max()), 최대시간생산_변경후=float(new_plan.max()),
                    이동량_합=float(np.abs(new_plan - base).sum() / 2), 곡선MAE_변경전=float(np.abs(cells.curve_final_base - cells.실제전력).mean()), 곡선MAE_변경후=float(np.abs(cells.curve_final_cf - cells.실제전력).mean()),
                    곡선_평균변화=float(cells.curve_final_diff.mean()), 곡선_최대변화=float(cells.curve_final_diff.abs().max()), q1_최대변화=float(cells.q1_new_diff.abs().max()),
                    경보칸_변경전=int(a_b.sum()), 경보칸_변경후=int(a_c.sum()), 주의이상칸_변경전=int((cells.경보단계_base >= 1).sum()), 주의이상칸_변경후=int((cells.경보단계_cf >= 1).sum()),
                    경보단계_바뀐칸=int((cells.경보단계_변화 != 0).sum()), 곡선_176초과량_변경전=float(ex_b), 곡선_176초과량_변경후=float(ex_c), 최대확률_변경전=float(cells.prob_peak_base.max()), 최대확률_변경후=float(cells.prob_peak_cf.max()),
                    적용범위=ap["level"], 적용범위_변경전=ap_base["level"])
        plan_tbl = pd.DataFrame(dict(시각=[o + pd.Timedelta(hours=h) for h in range(24)], 계획_변경전=base, 계획_변경후=new_plan, 변화량=new_plan - base))
        return dict(cells=cells, summary=summ, applicability=ap, plan=plan_tbl)


In [ ]:
"""Chapter 4: month-wide, model-based production and cost simulation.
피크 경보는 최종 경보 모델(앙상블 = 0.65*LightGBM 개선 Q1 + 0.35*GRU q=0.80, 기준 176)로 계산한다."""
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


PEAK_LIMIT = 176.0


def setup(root, settings):
    root = Path(root)
    cf = CounterfactualPredictor(verbose=True)
    raw = pd.read_csv(root / 'dataset/okm_augumented_2021.csv')
    raw['date'] = pd.to_datetime(raw['날짜'].astype(str), format='%Y%m%d')
    raw['hour'] = raw.groupby('date').cumcount()
    raw['origin'] = raw['date'] + pd.to_timedelta(raw['hour'], unit='h')

    month = pd.Period(settings['month'], freq='M')
    excluded = {pd.Timestamp(d) for d in settings.get('exclude_dates', [])}
    dates = [d for d in pd.date_range(month.start_time, month.end_time.normalize()) if d not in excluded]

    prediction_path = Path(settings['prediction_file'])
    if not prediction_path.is_absolute():
        prediction_path = root / prediction_path
    saved = pd.read_csv(prediction_path, parse_dates=['기준시각', '예측대상시각'])
    ensemble_path = Path(settings['ensemble_prediction_file'])
    if not ensemble_path.is_absolute():
        ensemble_path = root / ensemble_path
    saved_ensemble = pd.read_csv(ensemble_path, parse_dates=['기준시각', '예측대상시각'])

    days = {}
    for date in dates:
        day_rows = raw.loc[raw.date.eq(date)].sort_values('hour')
        reference = saved.loc[saved['기준시각'].eq(date)].sort_values('예측칸')
        reference_ensemble = saved_ensemble.loc[saved_ensemble['기준시각'].eq(date)].sort_values('예측칸')
        assert len(day_rows) == 24 and len(reference) == 96 and len(reference_ensemble) == 96, f'{date}: 완전한 일별 예측이 필요합니다.'
        past = raw.loc[raw.date.lt(date)]
        capacity = past.groupby('hour')['생산량'].quantile(.95).reindex(range(24)).to_numpy(float)
        plan = cf.baseline_plan(date)
        np.testing.assert_allclose(plan, day_rows['생산량'].to_numpy(float))
        capacity = np.maximum(capacity, plan)
        days[date] = {
            'plan': plan,
            'capacity': capacity,
            'labor': day_rows['인건비'].to_numpy(float),
            'tariff': day_rows['전기요금(계절)'].to_numpy(float),
            'reference': reference,
            'reference_ensemble': reference_ensemble,
        }
    return cf, days
def predict(cf, date, plan):
    """수치예측 곡선과 최종 피크 경보 모델(앙상블)을 계산한다.
    q: 앙상블 = 0.65 * LightGBM 개선 Q1 + 0.35 * GRU(q=0.80). 경보 칸 수와 초과량은 모두 이 값(기준 176)으로 센다."""
    i = cf._locate(date)
    features = cf._features_for(i, plan)
    week = cf._week_start(date)
    model_q, model_curve = cf.models(week)
    q_lgbm = np.maximum(model_q.predict(features[cf.W.L13]), 0).astype(np.float32)
    q_gru = cf.gru_predict(week, features)
    q = (ENS_W_LGBM * q_lgbm + ENS_W_GRU * q_gru).astype(np.float32)
    rule = features['rule3_oof'].to_numpy().astype(np.float32)
    residual_curve = np.maximum(
        model_curve.predict(features[cf.W.L13]) + rule, 0
    ).astype(np.float32)
    idle_gate = features['st_day_on'].to_numpy() == 0
    idle_profile = cf._idle_profile(i)
    curve = np.where(idle_gate, rule, residual_curve) if idle_profile is None else np.where(
        idle_gate & np.isfinite(idle_profile),
        idle_profile,
        np.where(idle_gate, rule, residual_curve),
    )
    assert np.isfinite(curve).all() and np.isfinite(q).all()
    return {'plan': np.asarray(plan, dtype=float).copy(), 'curve': curve, 'q': q, 'q_lgbm': q_lgbm, 'q_gru': q_gru}


def day_metrics(item, day, cfg, staff):
    curve = item['curve']
    plan = item['plan']
    energy_by_hour = curve.reshape(24, 4).sum(axis=1) * 0.25
    active = plan > cfg['active_epsilon']
    base_active = day['plan'] > cfg['active_epsilon']
    weighted_hours = float(day['labor'][active].sum())
    base_weighted_hours = float(day['labor'][base_active].sum())
    labor_won = staff * cfg['hourly_wage'] * weighted_hours
    added_labor_won = staff * cfg['hourly_wage'] * max(
        weighted_hours - base_weighted_hours, 0.0
    )
    q = item['q']
    return {
        'kwh': float(energy_by_hour.sum()),
        'peak_kw': float(curve.max()),
        'alarm_cells': int((q >= PEAK_LIMIT).sum()),
        'q_peak': float(q.max()),
        'q_excess': float(np.maximum(q - PEAK_LIMIT, 0).sum()),
        'usage_won': float(energy_by_hour @ day['tariff']),
        'labor_won': float(labor_won),
        'extra_labor_won': float(added_labor_won),
        'weighted_labor_hours': weighted_hours,
        'production': float(plan.sum()),
    }


def monthly(state, days, cfg, staff, basic_mode, baseline_peak):
    rows = [day_metrics(state[date], days[date], cfg, staff) for date in days]
    totals = {
        key: float(sum(row[key] for row in rows))
        for key in [
            'kwh', 'usage_won', 'labor_won', 'extra_labor_won',
            'weighted_labor_hours', 'production', 'alarm_cells', 'q_excess',
        ]
    }
    totals['peak_kw'] = max(row['peak_kw'] for row in rows)
    totals['q_peak'] = max(row['q_peak'] for row in rows)
    totals['billing_kw'] = baseline_peak if basic_mode == 'fixed' else totals['peak_kw']
    totals['basic_won'] = totals['billing_kw'] * cfg['basic_won_per_kw']
    totals['electricity_won'] = totals['basic_won'] + totals['usage_won']
    totals['total_won'] = totals['electricity_won'] + totals['labor_won']
    return totals


def objective_value(metrics, objective):
    if objective == 'electricity':
        return metrics['electricity_won']
    if objective == 'total':
        return metrics['total_won']
    raise ValueError(f'지원하지 않는 목적함수: {objective}')


def candidate_rank(metrics, objective):
    if objective == 'peak':
        return (
            round(metrics['peak_kw'], 6),
            round(metrics['electricity_won'], 2),
            round(metrics['total_won'], 2),
        )
    return (round(objective_value(metrics, objective), 2),)


def improves(trial, current, objective, cfg):
    if objective == 'peak':
        return trial['peak_kw'] < current['peak_kw'] - cfg['minimum_peak_improvement_kw']
    return objective_value(trial, objective) < (
        objective_value(current, objective) - cfg['minimum_improvement_won']
    )


def slot_pools(state, days, cfg, objective):
    sources = []
    destinations = []
    if objective == 'peak':
        source_pool_size = cfg['peak_source_pool_size']
        destination_pool_size = cfg['peak_destination_pool_size']
    elif objective == 'electricity':
        source_pool_size = cfg['electricity_source_pool_size']
        destination_pool_size = cfg['electricity_destination_pool_size']
    else:
        source_pool_size = cfg['source_pool_size']
        destination_pool_size = cfg['destination_pool_size']
    for date, item in state.items():
        plan = item['plan']
        risk = item['q'].reshape(24, 4).max(axis=1)
        power = item['curve'].reshape(24, 4).max(axis=1)
        for hour in range(24):
            amount = float(plan[hour])
            room = float(days[date]['capacity'][hour] - amount)
            labor = float(days[date]['labor'][hour])
            if amount > cfg['active_epsilon']:
                source_key = (
                    (-labor, amount, -risk[hour], -power[hour])
                    if objective == 'total'
                    # 기본요금은 월 최대전력으로 결정된다. 따라서 전기요금안은
                    # 경보값보다 현재 수치예측 전력이 큰 시간을 먼저 줄인다.
                    else (-power[hour], -risk[hour], amount)
                )
                sources.append((source_key, date, hour, amount))
            if room > cfg['active_epsilon']:
                already_active = amount > cfg['active_epsilon']
                if objective == 'total':
                    destination_key = (
                        0 if already_active else 1,
                        labor,
                        risk[hour],
                        power[hour],
                        -room,
                    )
                elif objective == 'peak':
                    destination_key = (
                        0 if already_active else 1,
                        power[hour],
                        risk[hour],
                        labor,
                        -room,
                    )
                else:
                    destination_key = (risk[hour], power[hour], -room, labor)
                destinations.append((destination_key, date, hour, room))

    sources.sort(key=lambda row: row[0])
    destinations.sort(key=lambda row: row[0])
    return (
        sources[:source_pool_size],
        destinations[:destination_pool_size],
    )


def candidate_moves(state, days, cfg, objective):
    sources, destinations = slot_pools(state, days, cfg, objective)
    for _, source_date, source_hour, source_amount in sources:
        for _, destination_date, destination_hour, destination_room in destinations:
            if source_date == destination_date and source_hour == destination_hour:
                continue
            seen_amounts = set()
            for fraction in cfg['move_fractions']:
                amount = min(source_amount * fraction, destination_room)
                amount_key = round(float(amount), 8)
                if amount <= cfg['active_epsilon'] or amount_key in seen_amounts:
                    continue
                seen_amounts.add(amount_key)
                yield (
                    source_date, source_hour,
                    destination_date, destination_hour,
                    float(amount),
                )


def peak_destination_orders(state, days, cfg, blocked_slots):
    rows = []
    for date, item in state.items():
        plan = item['plan']
        risk = item['q'].reshape(24, 4).max(axis=1)
        power = item['curve'].reshape(24, 4).max(axis=1)
        for hour in range(24):
            if (date, hour) in blocked_slots:
                continue
            room = float(days[date]['capacity'][hour] - plan[hour])
            if room <= cfg['active_epsilon']:
                continue
            rows.append({
                'date': date,
                'hour': hour,
                'room': room,
                'active': bool(plan[hour] > cfg['active_epsilon']),
                'power': float(power[hour]),
                'risk': float(risk[hour]),
                'labor': float(days[date]['labor'][hour]),
            })

    strategies = [
        lambda row: (not row['active'], row['power'], -row['room'], row['risk']),
        lambda row: (not row['active'], -row['room'], row['power'], row['risk']),
        lambda row: (row['power'], -row['room'], not row['active'], row['risk']),
        lambda row: (-row['room'], row['power'], not row['active'], row['risk']),
    ]
    scan_size = cfg['peak_destination_scan_size']
    return [sorted(rows, key=key)[:scan_size] for key in strategies]


def build_peak_batch(source_group, destination_order, fraction, cfg):
    blocked_slots = {(row[1], row[2]) for row in source_group}
    remaining_room = {
        (row['date'], row['hour']): row['room'] for row in destination_order
    }
    batch = []
    for _, source_date, source_hour, source_amount in source_group:
        remaining = source_amount * fraction
        leg_count = 0
        for destination in destination_order:
            destination_key = (destination['date'], destination['hour'])
            if destination_key in blocked_slots:
                continue
            room = remaining_room[destination_key]
            if room <= cfg['active_epsilon']:
                continue
            amount = min(remaining, room)
            if amount <= cfg['active_epsilon']:
                continue
            batch.append((
                source_date,
                source_hour,
                destination['date'],
                destination['hour'],
                float(amount),
            ))
            remaining_room[destination_key] -= amount
            remaining -= amount
            leg_count += 1
            if remaining <= cfg['active_epsilon']:
                break
            if leg_count >= cfg['peak_max_destination_legs']:
                break
    return batch


def peak_batch_candidates(state, days, cfg):
    sources, _ = slot_pools(state, days, cfg, 'peak')
    groups = [[source] for source in sources]
    max_group = min(cfg['peak_max_source_group'], len(sources))
    groups.extend([sources[:size] for size in range(2, max_group + 1)])
    seen = set()
    for source_group in groups:
        blocked_slots = {(row[1], row[2]) for row in source_group}
        orders = peak_destination_orders(state, days, cfg, blocked_slots)
        fractions = (
            cfg['peak_move_fractions']
            if len(source_group) == 1 else cfg['peak_group_fractions']
        )
        for destination_order in orders:
            for fraction in fractions:
                batch = build_peak_batch(
                    source_group, destination_order, fraction, cfg
                )
                if not batch:
                    continue
                signature = tuple(
                    (
                        str(source_date), source_hour,
                        str(destination_date), destination_hour,
                        round(amount, 6),
                    )
                    for source_date, source_hour, destination_date,
                    destination_hour, amount in batch
                )
                if signature in seen:
                    continue
                seen.add(signature)
                yield batch


def apply_moves(cf, state, moves):
    changed = dict(state)
    touched_dates = {
        date
        for move in moves
        for date in (move[0], move[2])
    }
    changed_plans = {
        date: state[date]['plan'].copy() for date in touched_dates
    }
    for source_date, source_hour, destination_date, destination_hour, amount in moves:
        changed_plans[source_date][source_hour] -= amount
        changed_plans[destination_date][destination_hour] += amount
    for date, plan in changed_plans.items():
        changed[date] = predict(cf, date, plan)
    return changed


def safety_ok(metrics, baseline_metrics, cfg):
    return (
        metrics['peak_kw'] <= baseline_metrics['peak_kw'] + cfg['safety_tolerance']
        and metrics['alarm_cells'] <= baseline_metrics['alarm_cells']
        and metrics['q_excess'] <= baseline_metrics['q_excess'] + cfg['safety_tolerance']
    )


def optimize(cf, baseline, days, cfg, objective, baseline_peak):
    state = dict(baseline)
    baseline_metrics = monthly(
        baseline, days, cfg, cfg['staff'], 'monthly_peak_assumption', baseline_peak
    )
    current_metrics = monthly(
        state, days, cfg, cfg['staff'], 'monthly_peak_assumption', baseline_peak
    )
    accepted_moves = []

    max_iterations = cfg['max_iterations_by_objective'][objective]
    for iteration in range(1, max_iterations + 1):
        best = None
        candidate_batches = (
            peak_batch_candidates(state, days, cfg)
            if objective == 'peak'
            else ([move] for move in candidate_moves(state, days, cfg, objective))
        )
        for moves in candidate_batches:
            trial_state = apply_moves(cf, state, moves)
            trial_metrics = monthly(
                trial_state, days, cfg, cfg['staff'],
                'monthly_peak_assumption', baseline_peak,
            )
            if not safety_ok(trial_metrics, baseline_metrics, cfg):
                continue
            if not improves(trial_metrics, current_metrics, objective, cfg):
                continue
            if best is None or candidate_rank(trial_metrics, objective) < candidate_rank(best[2], objective):
                best = (moves, trial_state, trial_metrics)

        if best is None:
            return state, current_metrics, accepted_moves, 'no_improvement_in_searched_candidates'

        moves, state, current_metrics = best
        for leg_number, move in enumerate(moves, start=1):
            source_date, source_hour, destination_date, destination_hour, amount = move
            accepted_moves.append({
                'objective': objective,
                'iteration': iteration,
                'batch_leg': leg_number,
                'batch_size': len(moves),
                'source_date': source_date,
                'source_hour': source_hour,
                'destination_date': destination_date,
                'destination_hour': destination_hour,
                'moved_production': amount,
                'electricity_won': current_metrics['electricity_won'],
                'labor_won': current_metrics['labor_won'],
                'total_won': current_metrics['total_won'],
                'peak_kw': current_metrics['peak_kw'],
            })
        moved_total = sum(move[4] for move in moves)
        print(
            objective, iteration,
            f'묶음 이동 {len(moves)}개',
            f'총 이동량={moved_total:.3f}',
            f'월 최대전력={current_metrics["peak_kw"]:.3f}',
            flush=True,
        )

    return state, current_metrics, accepted_moves, 'iteration_limit'


def run(root, cfg):
    root = Path(root)
    out = root / 'results' / cfg.get('output_name', 'resource_cost_optimization_chapter4')
    out.mkdir(parents=True, exist_ok=True)
    cf, days = setup(root, cfg)

    baseline = {}
    checks = []
    for date, day in days.items():
        item = predict(cf, date, day['plan'])
        for column, key, ref in [('curve_final', 'curve', day['reference']), ('q1_new', 'q_lgbm', day['reference']),
                                 ('ensemble', 'q', day['reference_ensemble'])]:
            error = float(np.abs(item[key] - ref[column].to_numpy()).max())
            checks.append({'date': date, 'column': column, 'maximum_difference': error})
            tolerance = 0.05 if column == 'ensemble' else 0.002      # 앙상블은 GRU를 다시 학습하므로 허용 오차를 조금 넓게 둔다
            assert error <= tolerance, f'{date} {column}: 02 최종 모델과 불일치 ({error})'
        baseline[date] = item
        print('원래 계획 확인:', date.date(), flush=True)
    pd.DataFrame(checks).to_csv(
        out / 'baseline_identity_checks.csv', index=False, encoding='utf-8-sig'
    )

    baseline_peak = max(float(item['curve'].max()) for item in baseline.values())
    chosen_states = {'baseline': baseline}
    stop_reasons = {'baseline': 'baseline'}
    all_moves = []
    for objective in ['peak', 'electricity', 'total']:
        state, _, moves, stop_reason = optimize(
            cf, baseline, days, cfg, objective, baseline_peak
        )
        chosen_states[objective] = state
        stop_reasons[objective] = stop_reason
        all_moves.extend(moves)

    baseline_production = sum(day['plan'].sum() for day in days.values())
    summaries = []
    schedules = []
    daily_rows = []
    prediction_rows = []
    sensitivity = []

    for objective, state in chosen_states.items():
        optimized_production = sum(item['plan'].sum() for item in state.values())
        assert np.isclose(optimized_production, baseline_production)
        for date, item in state.items():
            assert np.all(item['plan'] >= -cfg['safety_tolerance'])
            assert np.all(item['plan'] <= days[date]['capacity'] + cfg['safety_tolerance'])

        changed_days = sum(
            not np.allclose(item['plan'], days[date]['plan'])
            for date, item in state.items()
        )
        moved_production = 0.5 * sum(
            np.abs(item['plan'] - days[date]['plan']).sum()
            for date, item in state.items()
        )

        for basic_mode in ['fixed', 'monthly_peak_assumption']:
            current = monthly(state, days, cfg, cfg['staff'], basic_mode, baseline_peak)
            base = monthly(baseline, days, cfg, cfg['staff'], basic_mode, baseline_peak)
            row = {
                'basic_mode': basic_mode,
                'objective': objective,
                'optimization_basic_mode': 'monthly_peak_assumption',
                'staff': cfg['staff'],
                **current,
                'moved_production': moved_production,
                'changed_days': changed_days,
                'stop_reason': stop_reasons[objective],
            }
            for column in [
                'kwh', 'peak_kw', 'basic_won', 'usage_won', 'electricity_won',
                'labor_won', 'total_won', 'alarm_cells',
            ]:
                row[column + '_saving'] = base[column] - current[column]
            row['saving_percent'] = 100 * row['total_won_saving'] / base['total_won']
            summaries.append(row)

            for staff in cfg['staff_sensitivity']:
                for wage_factor in [1.0, 1.25, 1.5]:
                    sensitivity_cfg = {**cfg, 'hourly_wage': cfg['hourly_wage'] * wage_factor}
                    repriced = monthly(state, days, sensitivity_cfg, staff, basic_mode, baseline_peak)
                    repriced_base = monthly(baseline, days, sensitivity_cfg, staff, basic_mode, baseline_peak)
                    sensitivity.append({
                        'basic_mode': basic_mode,
                        'objective': objective,
                        'staff': staff,
                        'hourly_wage': sensitivity_cfg['hourly_wage'],
                        'saving_won': repriced_base['total_won'] - repriced['total_won'],
                    })

        for date, item in state.items():
            daily_rows.append({
                'objective': objective,
                'date': date,
                'baseline_production': float(days[date]['plan'].sum()),
                'optimized_production': float(item['plan'].sum()),
                'net_change': float(item['plan'].sum() - days[date]['plan'].sum()),
            })
            for hour in range(24):
                schedules.append({
                    'objective': objective,
                    'date': date,
                    'hour': hour,
                    'day_or_night': '주간' if 6 <= hour < 22 else '야간',
                    'labor_weight': days[date]['labor'][hour],
                    'capacity_reference': days[date]['capacity'][hour],
                    'baseline_production': days[date]['plan'][hour],
                    'optimized_production': item['plan'][hour],
                })
            for interval in range(96):
                prediction_rows.append({
                    'objective': objective,
                    'interval_start': date + pd.Timedelta(minutes=15 * interval),
                    'curve_kw': float(item['curve'][interval]),
                    'ensemble_kw': float(item['q'][interval]),
                    'lgbm_q1_kw': float(item['q_lgbm'][interval]),
                    'gru_q080_kw': float(item['q_gru'][interval]),
                })

    outputs = {
        'comparison_summary': summaries,
        'accepted_moves': all_moves,
        'hourly_plans': schedules,
        'daily_production_comparison': daily_rows,
        'predictions': prediction_rows,
        'staff_wage_sensitivity': sensitivity,
    }
    for name, rows in outputs.items():
        pd.DataFrame(rows).to_csv(out / f'{name}.csv', index=False, encoding='utf-8-sig')
    (out / 'run_settings.json').write_text(
        json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8'
    )

    result = pd.DataFrame(summaries)
    plot_data = result.loc[result['basic_mode'].eq('monthly_peak_assumption')]
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    axes[0].bar(plot_data['objective'], plot_data['total_won_saving'])
    axes[0].set_title('기존 계획 대비 총비용 절감액(원)')
    axes[1].bar(plot_data['objective'], plot_data['peak_kw_saving'])
    axes[1].set_title('기존 계획 대비 월 최대전력 감소(kW)')
    for axis in axes:
        axis.axhline(0, color='black', linewidth=0.5)
        axis.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    fig.savefig(out / 'cost_and_peak_comparison.png', dpi=160)
    plt.show()
    return result


In [ ]:
RESOURCE_COMMON = {
    'hourly_wage': 8720.0,
    'staff': 10,
    'staff_sensitivity': [5, 10, 20],
    'basic_won_per_kw': 8320.0,
    'alarm_model': 'ensemble: 0.65 * LightGBM improved Q1 (q=0.75) + 0.35 * GRU (q=0.80), fixed threshold 176',
    'ensemble_prediction_file': 'results/ensemble_lgbm_gru_team_aligned/predictions_all_7_9월.csv',
    'max_iterations_by_objective': {'peak': 100, 'electricity': 100, 'total': 100},
    'source_pool_size': 8,
    'destination_pool_size': 10,
    'electricity_source_pool_size': 12,
    'electricity_destination_pool_size': 12,
    'peak_source_pool_size': 16,
    'peak_destination_pool_size': 16,
    'peak_destination_scan_size': 96,
    'peak_max_destination_legs': 8,
    'peak_max_source_group': 3,
    'peak_move_fractions': [1.0, 0.75, 0.5, 0.35, 0.25, 0.15, 0.1],
    'peak_group_fractions': [0.5, 0.35, 0.25, 0.15],
    'move_fractions': [1.0, 0.5, 0.25],
    'minimum_improvement_won': 1.0,
    'minimum_peak_improvement_kw': 0.001,
    'active_epsilon': 1e-6,
    'safety_tolerance': 1e-5,
    'unit_interpretation': '15-minute average kW; kWh = kW * 0.25',
    'tariff_status': 'dataset seasonal rate assumed won/kWh; basic rate historical reference, not verified 2021 contract',
    'labor_status': 'assumed crew; 2021 minimum wage; no payroll or statutory overtime claim',
    'interpretation': 'retrospective monthly conditional model simulation; not observed causal savings',
}

RESOURCE_COLUMNS = [
    'basic_mode', 'objective', 'production', 'moved_production',
    'peak_kw', 'peak_kw_saving', 'kwh', 'kwh_saving',
    'basic_won', 'basic_won_saving', 'usage_won', 'usage_won_saving',
    'electricity_won', 'labor_won', 'labor_won_saving',
    'total_won', 'total_won_saving', 'saving_percent',
    'alarm_cells', 'changed_days', 'stop_reason',
]
print('자원 최적화 공통 설정:', RESOURCE_COMMON)

## 6.1 7월 분석 29일

In [ ]:
SETTINGS_07 = {
    **RESOURCE_COMMON,
    'month': '2021-07',
    'prediction_file': 'results/team_submission_runtime/outputs/validation_predictions_7월.csv',
    'exclude_dates': ['2021-07-13', '2021-07-15'],
    'output_name': 'resource_cost_optimization_chapter4_2021_07_available',
}
_t07 = time.perf_counter()
result_07 = run(ROOT, SETTINGS_07)
elapsed_07_minutes = (time.perf_counter() - _t07) / 60
print(f'7월 실행 시간: {elapsed_07_minutes:.1f}분')
display(result_07[RESOURCE_COLUMNS].round(2))

## 6.2 8월 전체 31일

In [ ]:
SETTINGS_08 = {
    **RESOURCE_COMMON,
    'month': '2021-08',
    'prediction_file': 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv',
    'exclude_dates': [],
    'output_name': 'resource_cost_optimization_chapter4_2021_08',
}
_t08 = time.perf_counter()
result_08 = run(ROOT, SETTINGS_08)
elapsed_08_minutes = (time.perf_counter() - _t08) / 60
print(f'8월 실행 시간: {elapsed_08_minutes:.1f}분')
display(result_08[RESOURCE_COLUMNS].round(2))

# 7. 전체 실행 완료

In [ ]:
RUN_ALL_MINUTES = (time.time() - RUN_ALL_STARTED) / 60
run_summary = {
    'project_root': str(ROOT),
    'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'run_all_minutes': RUN_ALL_MINUTES,
    'random_seed': 42,
    'resource_july_minutes': elapsed_07_minutes,
    'resource_august_minutes': elapsed_08_minutes,
    'main_result_folders': [
        'results/eda',
        'results/preprocessing_24h',
        'results/team_submission_runtime/outputs',
        'results/gru_24h_team_preprocessing',
        'results/ensemble_lgbm_gru_team_aligned',
        'results/error_peak_condition_analysis_ensemble_fixed176',
        'results/resource_cost_optimization_chapter4_2021_07_available',
        'results/resource_cost_optimization_chapter4_2021_08',
        'submission',
    ],
}
(ROOT / 'results/run_all_summary.json').write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pd.DataFrame([run_summary]))
print(f'전체 Run All 완료: {RUN_ALL_MINUTES:.1f}분')